# Spatial Metabolomics - Medulablastoma Project: DHB matrix

The below code is all code used to analyse the Visium + DHP MALDI matrix data

Author: Andrew Causer (uqacause@uq.edu.au)

## Load Packages

In [ ]:
#devtools::install_github("GenomicsMachineLearning/SpaMTP")

In [ ]:
library(SpaMTP) #This is a custom package that has been downloaded from: devtools::install_github("agc888/SpaMTP")

#General Libraries
library(Cardinal)
library(Seurat)

#For plotting + DE plots
library(ggplot2)
library(ggVennDiagram)
library(EnhancedVolcano)
library(viridis)

#For Correlation Analysis
# library(Hmisc)
# library(corrplot)
# library(cowplot)

#For Deconvolution Analysis
# library(CARD)
# library(MuSiC)

## Import MALDI and Visium Data

In [ ]:
### Specifiy data directory paths where data is stored

DATA_DIR <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/raw_data/MALDI/DHB/"
OUT_DIR  <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/DHB/"
UPDATED_OUT_DIR <- "/scratch/project/stseq/Andrew_C/MB/"


VIS_DATA_DIR <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/raw_data/Visium/"

### MALDI Spatial Metabolomic Data

**Import Data using SpaMTP** 

NOTE: Samples are input using a mass.range = (160,1500) and a resolution of 10ppm

In [ ]:
C1 <- LoadSM("vlp94a_dhb",path = DATA_DIR, mass.range = c(160,1500), resolution = 10, units = "ppm", bin_package = "Cardinal")
T1 <- LoadSM("vlp94c_dhb",path = DATA_DIR, mass.range = c(160,1500), resolution = 10, units = "ppm", bin_package = "Cardinal")
C2 <- LoadSM("vlp94b_dhb",path = DATA_DIR, mass.range = c(160,1500), resolution = 10, units = "ppm", bin_package = "Cardinal")
T2 <- LoadSM("vlp94d_dhb",path = DATA_DIR, mass.range = c(160,1500), resolution = 10, units = "ppm", bin_package = "Cardinal")

### Visium Spatial Transcriptomic Data

**Loads in visium data to overlay with seurat MALDI**

In [ ]:
visium_A <- Seurat::Load10X_Spatial(paste0(VIS_DATA_DIR,"VLP94_A/"))
visium_C <- Seurat::Load10X_Spatial(paste0(VIS_DATA_DIR,"VLP94_C/"))
visium_B <- Seurat::Load10X_Spatial(paste0(VIS_DATA_DIR,"VLP94_B/"))
visium_D <- Seurat::Load10X_Spatial(paste0(VIS_DATA_DIR,"VLP94_D/"))

Due to the way the data was imported some of the coordinate values for the image were imported incorrectly as strings rather then integers \
&nbsp;  &nbsp;  &nbsp; &nbsp;     - This will change them to their correct datatype

In [ ]:
# # Function to change the coordinate values of the Visium image to integer values -> required for plotting
# image_to_int <- function(seurat_obj){
#   seurat_obj@images[["slice1"]]@coordinates[["tissue"]] <- as.integer(seurat_obj@images[["slice1"]]@coordinates[["tissue"]])
#   seurat_obj@images[["slice1"]]@coordinates[["row"]] <- as.integer(seurat_obj@images[["slice1"]]@coordinates[["row"]])
#   seurat_obj@images[["slice1"]]@coordinates[["col"]] <- as.integer(seurat_obj@images[["slice1"]]@coordinates[["col"]])
#   seurat_obj@images[["slice1"]]@coordinates[["imagerow"]] <- as.integer(seurat_obj@images[["slice1"]]@coordinates[["imagerow"]])
#   seurat_obj@images[["slice1"]]@coordinates[["imagecol"]] <- as.integer(seurat_obj@images[["slice1"]]@coordinates[["imagecol"]])
  
#   return(seurat_obj)
  
# }
# visium_A <- image_to_int(visium_A)
# visium_B <- image_to_int(visium_B)
# visium_C <- image_to_int(visium_C)
# visium_D <- image_to_int(visium_D)

Imports CSV files which contain the corrected coordinates for the mapped Visium and MALDI data \
&nbsp;  &nbsp;  &nbsp; &nbsp;    - We import coordinates for each sample and then change the column names to match the required input

In [ ]:
## Import Coordinate File
mapped_A <- read.csv(paste0(VIS_DATA_DIR, "mappings/mapped_a_v1.csv"))
mapped_B <- read.csv(paste0(VIS_DATA_DIR, "mappings/mapped_b_v1.csv"))
mapped_C <- read.csv(paste0(VIS_DATA_DIR, "mappings/mapped_c_v1.csv"))
mapped_D <- read.csv(paste0(VIS_DATA_DIR, "mappings/mapped_d_v1.csv"))

In [ ]:
### Convert column names to match required names
mapped_A <- mapped_A[c("X_new","Y_new")]
mapped_B <- mapped_B[c("X_new","Y_new")]
mapped_C <- mapped_C[c("X_new","Y_new")]
mapped_D <- mapped_D[c("X_new","Y_new")]

In [ ]:
mapped_A$X_new1 <- mapped_A$Y_new
mapped_A$Y_new <- mapped_A$X_new
mapped_A$X_new <- mapped_A$X_new1
mapped_A$X_new1 <- NULL

mapped_B$X_new1 <- mapped_B$Y_new
mapped_B$Y_new <- mapped_B$X_new
mapped_B$X_new <- mapped_B$X_new1
mapped_B$X_new1 <- NULL

mapped_C$X_new1 <- mapped_C$Y_new
mapped_C$Y_new <- mapped_C$X_new
mapped_C$X_new <- mapped_C$X_new1
mapped_C$X_new1 <- NULL

mapped_D$X_new1 <- mapped_D$Y_new
mapped_D$Y_new <- mapped_D$X_new
mapped_D$X_new <- mapped_D$X_new1
mapped_D$X_new1 <- NULL


In [ ]:
Adjust_coords <- function(SpaMTP, coords){
    
    rownames(coords) <- rownames(SpaMTP@meta.data)
    cents <- SeuratObject::CreateCentroids(coords)
    
    
      segmentations.data <- list(
        "centroids" = cents
      )
    
      coords <- SeuratObject::CreateFOV(
        coords = segmentations.data,
        type = c("centroids"),
        molecules = NULL,
        assay = "Spatial"
      )
    
      SpaMTP[["fov"]] <- coords

      return(SpaMTP)
}

In [ ]:
C1 <- Adjust_coords(C1, mapped_A)
C2 <- Adjust_coords(C2, mapped_B)
T1 <- Adjust_coords(T1, mapped_C)
T2 <- Adjust_coords(T2, mapped_D)

In [ ]:
T1@meta.data$treatment <- "Treated"
C1@meta.data$treatment <- "Control"
T2@meta.data$treatment <- "Treated"
C2@meta.data$treatment <- "Control"

##### Merge Test

## Data QC

Next we will look that the quality of each object. 

First we will add sample infomation and merge the object together so they can be compared.

In [ ]:
all_data <- JoinLayers(merge(T1, y=list(C1,T2,C2)))

In [ ]:
MassIntensityPlot(all_data, split.by =  "sample", annotation.column = NULL)

#### Calculate shared m/z's between samples

In [ ]:
df <- statPlot(all_data, group.by = "sample")
non_zero <- df[df$x != 0,]


mz_names <- list("C1" = non_zero[non_zero$var == "vlp94a_dhb",]$mz,
                         "C2" = non_zero[non_zero$var == "vlp94b_dhb",]$mz,
                         "T1" = non_zero[non_zero$var == "vlp94c_dhb",]$mz,
                         "T2" = non_zero[non_zero$var == "vlp94d_dhb",]$mz)

p1 <- ggVennDiagram(mz_names, names(mz_names), label_percent_digit = 1, label = "percent", label_alpha = 0, label_size =  7, set_size = 8) + scale_fill_gradientn(colors = RColorBrewer::brewer.pal(9, "Blues")[1:8], limits = c(0, 6500), na.value = RColorBrewer::brewer.pal(9, "Blues")[9]) 
       options(repr.plot.width = 10, repr.plot.height = 10) 

p1
#ggsave(p1, filename = paste0(OUT_DIR, "plots/ven.pdf"), height = 10, width = 10)                  

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10) 

ridge_plot <- MZRidgePlot(seurat.obj = all_data, group.by = "sample", slot = "counts", bins = 200, verbose = F, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 

ridge_plot
#ggsave(ridge_plot,  filename = paste0(OUT_DIR, "plots/ridge_plot.pdf"), height = 10, width = 15)

#### Calculate Correlation values

In [ ]:
df <- statPlot(seurat.obj = all_data, group.by = "sample", slot = "counts", log.data = TRUE)

In [ ]:
pivoted_df <- tidyr::pivot_wider(df, names_from = var, values_from = x)

In [ ]:
mtx <- as.matrix(pivoted_df)
rownames(mtx) <- mtx[,"mz"]
mtx <- mtx[,c("vlp94d_dhb", "vlp94c_dhb", "vlp94b_dhb","vlp94a_dhb")]

correlation <- rcorr(mtx, type=c("pearson"))

correlation$P[is.na(correlation$P)] <- 0

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

#pdf(file = paste0(OUT_DIR,"plots/person_correlation_QC.pdf"), height = 10, width = 10)
corrplot(correlation$r, type = "upper", p.mat = correlation$P, sig.level = 0.05, addCoef.col = 'white', tl.pos = 'd', col = COL2('RdBu'), col.lim = c(0,1))
title(main = "Pearson's Correlation", cex.main = 1.5)
#dev.off()

lets plot some other QC plots to see how the data looks

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5) 

MZVlnPlot(seurat.obj = all_data, group.by = "sample", slot = "counts", show.points = FALSE, log.data = FALSE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5) 
FeatureScatter(all_data, feature1 = "nCount_Spatial", feature2 = "nFeature_Spatial", group.by = "sample")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5) 
RidgePlot(all_data, features = c("nCount_Spatial", "nFeature_Spatial"), group.by = "sample")

we can also remove the top and bottom 5% of values to look at the distribution between samples 

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5) 
MZRidgePlot(seurat.obj = all_data, group.by = "sample", slot = "counts", bins = 200, top.cutoff = 0.05,bottom.cutoff = 0.05, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))

## Normalise the Metabolomic Data

Next we will normalise the data using a median scaling normalisation between samples and *TIC* normalisation for each spot/pixel. For more about *TIC* normalisation use this reference (https://doi.org/10.1007/s00216-011-4929-z)

In [ ]:
all_data <- NormalizeSMData(all_data, normalisation.type = "TIC")

After normalising using *TIC* normalisation lets plot the data and see how its different to the raw data

In [ ]:
ridge_plot_raw <- MZRidgePlot(seurat.obj = all_data, group.by = "sample", slot = "counts", bins = 200, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 
ridge_plot_norm <- MZRidgePlot(seurat.obj = all_data, group.by = "sample", slot = "data", bins = 200, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 

box_plot_raw <- MZBoxPlot(seurat.obj = all_data, group.by = "sample", slot = "counts", show.points = FALSE, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 
box_plot_norm <- MZBoxPlot(seurat.obj = all_data, group.by = "sample", slot = "data", show.points = FALSE,  log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 16) 

(ridge_plot_raw | ridge_plot_norm)/
(box_plot_raw | box_plot_norm)

Now lets scale the data based on the median m/z intensity of one sample

In [ ]:
## This function scales all data to the same 
scalebymedian <- function(combined.obj, ident, refIdent = NULL, normalisation.type = "CPM", CPM.scale.factor = 1e6, assay = "Spatial") {

    data_list <- list()
    Idents(combined.obj) <- ident
    if (length(unique(Idents(combined.obj))) <= 1){
        stop("Specified ident has 0 or 1 catagory in seurat object. Length of Idents must be > 1 for TMM (between sample) normalisation factors to be calculated")
        
    }

    if (is.null(refIdent)){
        stop("A refIdent must be specified")
    }

    if (!(is.null(refIdent))) {
       if (!(refIdent %in% unique(Idents(combined.obj)))){
           stop("The refIdent supplied is not present in the ident column. Please specify a group that is found within the ident column specififed")
       }
    }

    if (!("data" %in% Layers(combined.obj[[assay]]))){
        stop("There is no 'data' layer in the assay provided. The seurat object must contain a data slot for the data to be scaled")
    }

    # if (normalisation.type == "CPM") {
    #     normalisation.type <- "RC"
    # }

    # if (normalisation.type == "TIC") {
    #     normalisation.type <- "RC"
    #     CPM.scale.factor <- length(rownames(combined.obj))
    # }

    

    for (name in unique(Idents(combined.obj))){
        suppressWarnings({
            suppressMessages({
                sub <- subset_SPM(combined.obj, idents = name)
            })
        })
            
        data_list[[name]] <- sub
    }
    
    df <- data.frame(mz = rownames(data_list[[1]]))
    rownames(df) <- df$mz
    
    for (dataset in names(data_list)){
        rowsum <- rowSums(data_list[[dataset]][[assay]]["data"])
        df[dataset] <- rowsum
    }
    mtx <- Matrix::as.matrix(df[names(data_list)])

    medians  <- apply(mtx, 2, median)
    factors <- lapply(medians, function(x) {x/medians[[refIdent]]})
    for (name in names(data_list)){

        data_list[[name]][[assay]]["scale.data"] <- data_list[[name]][[assay]]["data"]/factors[[name]]
    }

    merged.data <- JoinLayers(merge(data_list[[1]], y = data_list[2: length(names(data_list))]), merge.data = TRUE)
    return(merged.data)
}
    

In [ ]:
all_data <- scalebymedian(all_data, "sample", refIdent = "vlp94d_dhb", normalisation.type = "TIC", CPM.scale.factor = NULL, assay = "Spatial")

Now lets check the scaled.data results

In [ ]:
ridge_plot_scale <- MZRidgePlot(seurat.obj = all_data, group.by = "sample", slot = "scale.data", bins = 200, log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 

box_plot_scale <- MZBoxPlot(seurat.obj = all_data, group.by = "sample", slot = "scale.data", show.points = FALSE,  log.data = TRUE, cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))+ theme_minimal() +theme(panel.grid = element_blank()) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 8) 

(ridge_plot_raw | ridge_plot_norm|ridge_plot_scale)
(box_plot_raw | box_plot_norm|box_plot_scale)

In [ ]:
# Save merged object
saveRDS(all_data, paste0(UPDATED_OUT_DIR,"data_objects/all_data_scaled.RDS"))

### Load Saved Data

In [ ]:
all_data <- readRDS(paste0(OUT_DIR,"data_objects/all_data_scaled.RDS"))

Now we can see that the median of each sample is the same!

## Visualise the Data

Here we can use *SpaMTP* search and plotting functions to visualise the data

In [ ]:
#Finds the m/z value in the specified dataset closest to the one provided
FindNearestMZ(data = all_data, target_mz = 701.000001)

From this we know that *'mz-700.997145686424'* - is the closest entry we have to 701.000001

We can also plot m/z values spatially using *SpaMTP* **ImageMZPlot** function

Lets start with m/z = 448.245

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

ImageMZPlot(all_data, fov = "fov", mzs  = c(448.245), size = 2.5, axes = FALSE, border.color = "grey", dark.background = TRUE) & scale_fill_gradientn(colors = viridis(100)) 

We can also specify multiple m/z values to plot at once. Lets try m/z = 448.245 & 450.245

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
ImageMZPlot(all_data, fov = "fov", mzs  = c(448.245, 450.245), size = 2.5, axes = FALSE, border.color = "grey", dark.background = TRUE) & scale_fill_gradientn(colors = viridis(100)) 

We can also expand the range of m/z values to plot. This is done using the 'plusminus' option, which will include m/z intensity values within a plusminus range of what was given. 

Lets use plusminus 0.005

In [ ]:
ImageMZPlot(all_data, fov = "fov", mzs  = c(448.245, 450.245), plusminus = 0.005, size = 2.5, axes = FALSE, border.color = "grey", dark.background = TRUE) & scale_fill_gradientn(colors = viridis(100)) 

Because *SpaMTP* uses *Seurat* objects as a framework for it's objects we can use *ggplot2* functions to change and save the graph.

Lets agian check the drug expression in the <span style = "color: blue;">**Control**</span> samples are on the *left*, and the <span style = "color: red;">**Treated**</span> samples are on the *right*

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
plot <- (ImageMZPlot(all_data, fov = "fov.2", mzs  = c(448.245), plusminus = 0.005, size = 1.5, axes = FALSE, border.color = "grey", dark.background = TRUE)/
         ImageMZPlot(all_data, fov = "fov.4", mzs  = c(448.245), plusminus = 0.005, size = 1.5, axes = FALSE, border.color = "grey", dark.background = TRUE))& scale_fill_gradientn(colors = viridis(6)[2:6],limits = c(190,800), na.value = viridis(6)[1]) | 
        (ImageMZPlot(all_data, fov = "fov", mzs  = c(448.245), plusminus = 0.005, size = 1.5, axes = FALSE, border.color = "grey", dark.background = TRUE)/
         ImageMZPlot(all_data, fov = "fov.3", mzs  = c(448.245), plusminus = 0.005, size = 1.5, axes = FALSE, border.color = "grey", dark.background = TRUE))& scale_fill_gradientn(colors = viridis(6)[2:6],limits = c(190,800), na.value = viridis(6)[1]) 
plot

In [ ]:
ggsave(plot, file = paste0(OUT_DIR,"plots/drug_spatial_plots_square.png"), height = 15, width = 20)

## Annotating m/z values of *SpaMTP* Object

One important part about Spatial Metabolomic analysis is the ability to annotate m/z values to meaningful biological metabolite names.
*SpaMTP* has 4 inbuilt reference dataset which you can use simply by calling the dataset name. 

Here we will use the Human Metabolite Database (HMDB) to annotate each sample

In [ ]:
##shows the structure of the HMDB
head(HMDB_db)

This method has additional options to input into the function such as change which adducts to use for annotation. We will use the default for the DHP matrix which is = "M+H". 

Lets merge all the databases together into one

In [ ]:
all_data_annotated <- AnnotateSM(all_data, rbind(HMDB_db), adducts = c("M+H", "M+K", "M+NA"), ppm_error = 5 )

In [ ]:
library(dplyr)
db_3 <- all_data_annotated@tools$db_3

db_3 = db_3 %>%
    tidyr::separate_rows(Isomers_IDs, sep = "; ")

In [ ]:
db_3[db_3$Isomers_IDs == "hmdb:HMDB0248014",]

In [ ]:
1

The relative annotations will now be stored in the respective assay meta.data slot of each object.

This can be viewed by:

In [ ]:
head(all_data_annotated[["Spatial"]]@meta.data)

In [ ]:
SearchAnnotations(all_data_annotated, "Palbociclib")

We can see that the annotations have been stored in the "All_Isomer_Names" column

Additionally we can check if a metabolite has been assigned to an m/z value. Here we want to see if our drug called ***Palbociclib*** has been annotated in our treated object

##### SearchAnnotations(all_data_annotated, "Palbociclib")

We can also get exact matches for a metabolite or we can find all annotations which contain the metabolite term of interest. 

For example we have "Glucose" and "2-Deoxyglucose". We can change the 'search.exact' option to allow us to get either:

In [ ]:
SearchAnnotations(all_data_annotated, "Glucose",search.exact = TRUE)

In [ ]:
SearchAnnotations(all_data_annotated, "Glucose",search.exact = FALSE)[1:3,]

Another useful feature of *SpaMTP* using *Seurat* objects as a foundation is that we can easly add metadata to our objects. 
Here we will specifiy which samples are treated and which are control so we can merge all samples into one object.

we can also now plot based on the metabolite name annotated. You can use the *SearchAnnotations* function from *SpaMTP* to find your metabolite of interest and if it is in our current data object

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

ImageMZAnnotationPlot(all_data_annotated, fov = c("fov"),metabolite = "Palbociclib", plot.exact = TRUE, size = 2.5) & scale_fill_gradientn(colors = viridis(100), limits = c(0, 150)) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 4)

MassIntensityPlot(all_data_annotated, group.by = "sample", annotation.column = NULL, label.annotations = F, mz.labels = c(448.2422, 448.246), mass.range = c(448.2, 448.3),slot = "scale.data", nlabels.to.show = 2, y.lim = c(0,20))


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 4)

MassIntensityPlot(all_data_annotated, group.by = "sample", label.annotations = TRUE, mz.labels = c(448.246), mass.range = c(447.5, 449),slot = "scale.data", y.lim = c(0, 20))


### Save Data

It is important to be able to save the data. We have various options. 

1. we can use *SpaMTP* function to save the matrix.mtx, barcode.csv, counts.csv and meta.data.csv into a folder
2. we can use saveRDS to save the whole object.

Option 1 is best for export and use with python programs such as *scanpy*. Option 2 is easiest for easy loading into R, we will use option 2 for now.

In [ ]:
# Save merged object
saveRDS(all_data_annotated, paste0(UPDATED_OUT_DIR,"all_data_annotated.RDS"))

### Load Saved Data

In [ ]:
all_data_annotated <- readRDS(paste0(OUT_DIR,"data_objects/all_data_annotated.RDS"))

## SM ONLY Analysis

### Find Differentially Expressed m/z Peaks

*SpaMTP* has an inbuild function for calculating differentially expressed peaks (DEPs) which utilises the statistical framework of EdgeR (https://doi.org/10.1093/bioinformatics/btp616). 

One comparison which we are interested in is DEPs between <span style = "color: blue;">**Control**</span> and <span style = "color: red;">**Treated**</span> within the tumour region specifically. To do this we need to define and subset the merged object based on the tumour region. The ssc cluster which defines this region is = 3

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

ImageDimPlot(all_data_annotated, fov = c("fov", "fov.2", "fov.3", "fov.4"), group.by = "ssc", size = 2)

In [ ]:
suppressWarnings({
    Idents(all_data_annotated) <- "ssc"
    tumour <- subset(all_data_annotated, idents = "3")
})

Because the list of annotations and also the number of different metabolites that can be assigned to one m/z value is very large we can filter down the dataset by only including m/z values that were annotated with < 5 metabolites

In [ ]:
suppressWarnings({
    tumour_refined <- getRefinedAnnotations(tumour, n = 4)
})

We can then run DE of this group. We can save the output by seting the 'DE_output_dir' option to the filepath we want to store this is, but for now we will leave this blank

In [ ]:
tumour_DEPs <- FindAllDEPs(tumour_refined, "treatment", n = 3, logFC_threshold = 1.2, DE_output_dir =NULL,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames")

This *tumour_DEPs* object now contains the pooled count values generated by *EdgeR* along with a list of DEPs stored in the relative list shown below

In [ ]:
head(tumour_DEPs$DEPs)

We can now plot these on a heatmap

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

DEPsHeatmap(edgeR_output = tumour_DEPs)

We can see that our drug (m/z = 448.246) is one of the top UP-regulated DEPs in the treated tumour areas! 

Lets now add the annotations to this plot

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

DEPsHeatmap(edgeR_output = tumour_DEPs, plot_annotations_column =  "annotations", nlabels.to.show = 1)

***Palbociclib*** is in our list along with a number of other metabolites!!

We can also show the results in a volcano plot

In [ ]:


### Subsets the original dataset to only include cluster 5 and 6

### Sets up colouring for significant spots in volcano plot
keyvals <- ifelse(
    tumour_DEPs$DEPs$logFC < -1.2  & tumour_DEPs$DEPs$PValue < 10e-4, 'royalblue',
      ifelse(tumour_DEPs$DEPs$logFC > 1.2 & tumour_DEPs$DEPs$PValue < 10e-4, 'red',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == 'red'] <- 'Cluster 6'
  names(keyvals)[keyvals == 'black'] <- 'Non-Sig'
  names(keyvals)[keyvals == 'royalblue'] <- 'Cluster 5'


### Plots volcano plot with DEP results
volc_plot <- EnhancedVolcano::EnhancedVolcano( tumour_DEPs$DEPs,
                                  selectLab = c(""), 
                                  lab = tumour_DEPs$DEPs$gene,
                        
                                  colCustom = keyvals,
                                  #cutoffLineType = 'blank',
                                  pCutoff = 10e-4,
                                  FCcutoff = 1.2,
                                  pointSize = 2,
                                  labSize = 5,
                                  labCol = 'black',
                                  labFace = 'bold',
                                  colAlpha = 1,
                                  x = 'logFC',
                                  y = 'PValue', 
                                  gridlines.major = FALSE,
                                  gridlines.minor = FALSE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)
volc_plot

ggsave(volc_plot, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/treated_vs_control_DE.pdf", height = 15, width = 15)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

ImageMZPlot(all_data_annotated, mzs = 380.183, fov = c("fov", "fov.2", "fov.3", "fov.4"), size = 2) & scale_fill_gradientn(colors = viridis(100), limits= c(0,100))

We will now run the whole m/z list and save the output for deeper analysis

In [ ]:
all_tumour_DEPs <- FindAllDEPs(tumour, "treatment", n = 3, logFC_threshold = 1.2, slot = "data", #DE_output_dir = paste0(OUT_DIR, "DE/"),
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames")

In [ ]:
all_tumour_DEPs$Treated$DEPs

In [ ]:
keyvals <- ifelse(
    all_tumour_DEPs$Treated$DEPs$logFC < 0  & all_tumour_DEPs$Treated$DEPs$PValue < 10e-5, 'royalblue',
      ifelse(all_tumour_DEPs$Treated$DEPs$logFC > 0 & all_tumour_DEPs$Treated$DEPs$PValue < 10e-5, 'red',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == 'red'] <- 'Up'
  names(keyvals)[keyvals == 'black'] <- 'Normal'
  names(keyvals)[keyvals == 'royalblue'] <- 'Down'


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 10)

volcPlot <- EnhancedVolcano::EnhancedVolcano(all_tumour_DEPs$Treated$DEPs,
                                             selectLab = c("mz-448.246729024761", "mz-449.247434602517", "mz-1383.83486970307"), 
    #lab = rep("", length(rownames(all_tumour_DEPs$Treated$DEPs))),
                                             lab = rownames(all_tumour_DEPs$Treated$DEPs),
    FCcutoff = 0,
    colCustom = keyvals,
    cutoffLineType = 'blank',
    pCutoff = 10e-5,
    x = 'logFC',
    y = 'PValue', 
    gridlines.major = FALSE,
    gridlines.minor = FALSE)

In [ ]:
volcPlot
#ggsave(volcPlot, filename = paste0(OUT_DIR,"plots/volc_plot.pdf"), height = 10, width = 7)

In [ ]:
dim(all_tumour_DEPs$Treated$DEPs[all_tumour_DEPs$Treated$DEPs$regulate == "Down",])

In [ ]:
pdf(paste0(OUT_DIR,"plots/pie.pdf"), height = 10, width = 7)
pie(x = c(292,362))
dev.off()

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 10)

EnhancedVolcano::EnhancedVolcano(all_tumour_DEPs$Treated$DEPs,
    lab = rownames(all_tumour_DEPs$Treated$DEPs),
    x = 'logFC',
    y = 'PValue')

In [ ]:
DEPsHeatmap(edgeR_output = all_tumour_DEPs$Treated, plot_annotations = FALSE)

Lets now look at some of these metabolites spatially, lets take the next top up- and down-regulated metabolites from the volcano plot above

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

ImageMZPlot(all_data_annotated, mzs = 488.209, fov = c("fov", "fov.2", "fov.3", "fov.4"), size = 1.5) & scale_fill_gradientn(colors = viridis(100), limits = c(0, 100))
ImageMZPlot(all_data_annotated, mzs = 223.0762, fov = c("fov", "fov.2", "fov.3", "fov.4"), size = 1.5) & scale_fill_gradientn(colors = viridis(100), limits = c(0, 100))

### Analysis DEPs Lipids

Due to their structure lipids are very divise and one m/z mass can be representitive of numerous lipids. 

Therefore, we can use the SpaMTP function below to group lipids into their *Lipid.Maps* Category and Class!

In [ ]:
library(rgoslin)
RefineLipids <- function(data, database = "HMDB", add_infomation = c("Lipid.Maps.Category", "Lipid.Maps.Main.Class", "Species.Name")){
    

    rows_list <- list()


    total_rows = length(rownames(data))
      pb <- utils::txtProgressBar(min = 0,      # Minimum value of the progress bar
                       max = total_rows, # Maximum value of the progress bar
                       style = 3,    # Progress bar style (also available style = 1 and style = 2)
                       width = 50,   # Progress bar width. Defaults to getOption("width")
                       char = "=")
    
    for (idx in 1:total_rows){
        row_name <- rownames(data)[idx]
        row <- data[row_name,][["annotations"]]
        annotations <- strsplit(row, "; ")[[1]]

        suppressMessages({
            lipid.df <- parseLipidNames(annotations, grammar = database)
        })

        if (all(!add_infomation %in% colnames(lipid.df))){

            empty_df <- data.frame(matrix(NA, ncol = length(add_infomation), nrow = 1))
            colnames(empty_df) <- add_infomation
            
            rows_list[[row_name]] <- empty_df
        } else{
            
            lipid.df  <- lipid.df[add_infomation]
            new.row.data <- data.frame(lapply(lipid.df, function(col) paste(unique(col), collapse = "; ")))
            rows_list[[row_name]] <- new.row.data
        }
        utils::setTxtProgressBar(pb, idx)
    }

    close(pb)
                                              
    final_df <- dplyr::bind_rows(rows_list)    
    data[colnames(final_df)] <- final_df
                                              
      
    data <- data %>% 
      dplyr::mutate(across(all_of(add_infomation), ~ ., .names = "All.{.col}"))

    data <- data %>%
      dplyr::mutate_at(vars(add_infomation), ~ stringr::str_replace_all(., "\\bNA\\b", ""))
    
    for (col in colnames(data)) {
    
      data[[col]] <- gsub("(^|\\W)\\;\\s+|\\s+\\;($|\\W)", "\\1\\2", data[[col]])
        data[[col]] <- gsub("\\;\\s+$", "", data[[col]])
    }
    
    data <-data %>%
      dplyr::mutate_at(vars(add_infomation), ~ stringr::str_trim(.))

                                              
    return(data)                                        
}

In [ ]:
refined_Lipid_DEPs <- RefineLipids(all_tumour_DEPs$Treated$DEPs)

We can now split these results and look at the types of lipids that were up- and down-regulated in the <span style = "color: red;">**Treated**</span> compared to the <span style = "color: blue;">**Controls**</span> 

In [ ]:
UP <- refined_Lipid_DEPs[refined_Lipid_DEPs$regulate == "Up",]
Down <-refined_Lipid_DEPs[refined_Lipid_DEPs$regulate == "Down",]

In [ ]:
head(refined_Lipid_DEPs)

In [ ]:
table(refined_Lipid_DEPs$Lipid.Maps.Category)

In [ ]:
sort(table(refined_Lipid_DEPs$Lipid.Maps.Main.Class))

Great! Now lets plot the results

In [ ]:
# Assuming your dataframe is called df and contains a column named 'regulate' indicating 'Up' or 'Down' entries,
# and a column named 'Lipid.Maps.Category' containing the categories.

# Load required libraries
library(dplyr)
library(ggplot2)

options(repr.plot.width = 10, repr.plot.height = 10)

# Compute counts of Lipid Maps Categories for 'Up' and 'Down' entries
category_counts <- refined_Lipid_DEPs %>%
  group_by(regulate, Lipid.Maps.Category) %>%
  summarise(count = n()) %>%
  ungroup()

category_counts <- category_counts %>%
  filter(regulate != "Normal", Lipid.Maps.Category != "NA")

# Plot bar graph
ggplot(category_counts, aes(x = Lipid.Maps.Category, y = count, fill = regulate)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Relative Number of Lipid Maps Categories for Up and Down Entries",
       x = "Lipid Maps Category",
       y = "Count") +
  theme_classic() +
  #theme(axis.text.x = element_text(angle = 0, hjust = 1)) +
  scale_fill_manual(values = c("Up" = "red", "Down" = "blue"))  # Customizing colors for 'Up' and 'Down' entries


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
# Load required libraries
library(dplyr)
library(ggplot2)

# Compute counts of Lipid Maps Categories for 'Up' and 'Down' entries
category_counts <- refined_Lipid_DEPs %>%
  group_by(regulate, Lipid.Maps.Main.Class) %>%
  summarise(count = n()) %>%
  ungroup()

category_counts <- category_counts %>%
  filter(regulate != "Normal", Lipid.Maps.Main.Class != "NA")

# Plot bar graph
ggplot(category_counts, aes(x = Lipid.Maps.Main.Class, y = count, fill = regulate)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Relative Number of Lipid Maps Categories for Up and Down Entries",
       x = "Lipid Maps Category",
       y = "Count") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_manual(values = c("Up" = "red", "Down" = "blue"))  # Customizing colors for 'Up' and 'Down' entries


Awesome, we have found a bunch of lipids and metabolites that are differentially expressed in our <span style = "color: red;">**Treated**</span> tumours compared to the <span style = "color: blue;">**Control**</span> tumours.

Lets save the updated DEPs data.frame as a csv

In [ ]:
write.csv(refined_Lipid_DEPs, paste0(OUT_DIR, "DE/lipid_class_treated_DEPs.csv"))

In [ ]:
refined_Lipid_DEPs <- read.csv(paste0(OUT_DIR, "DE/lipid_class_treated_DEPs.csv"))

In [ ]:
TG <- subset(refined_Lipid_DEPs, Lipid.Maps.Main.Class == "TG")

In [ ]:
TG_up <- subset(TG, regulate == "Up")

In [ ]:
# this block of code is for combining the values of a metabolite into 1

bin_metabolites <- function(data, mzs, assay = "Spatial", slot = "data", bin_name = "Binned_Metabolites"){

    binned_counts <- colSums(data[[assay]][slot][mzs,])
    data[[bin_name]]<- binned_counts
    return(data)  
}


In [ ]:
bin <- bin_metabolites(all_data_annotated, TG_up$X)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

ImageFeaturePlot(bin, features = "Binned_Metabolites", fov = c("fov", "fov.2", "fov.3", "fov.4"), size = 1.5) & scale_fill_gradientn(colors = viridis(100))


## Merge Visium and MALDI data 

Next we will merge our spatial transcriptomics and spatial metabolomic data together so that each spot on our tissue section contains both transcriptional and metabolite data. This process takes a long time so we will save our objects after so that we can easily load them in and dont have to run this function every time

In [ ]:
# Function to calculate the coordinates of a square given its center and width
get_square_coordinates <- function(center_x, center_y, width, name) {
  # Calculate half width
  half_width <- width / 2
  
  # Calculate coordinates of the four corners
  x_coords <- c(center_x - half_width, center_x + half_width, center_x + half_width, center_x - half_width, center_x - half_width)
  y_coords <- c(center_y - half_width, center_y - half_width, center_y + half_width, center_y + half_width, center_y - half_width)
  
  # Return the coordinates as a matrix
  square_df <- data.frame(
    Selection = rep(name, n = 4),
    X = x_coords,
    Y = y_coords
    
  )
  return(square_df)
}

AlignData <- function(SM.data, ST.data,
                      SM.assay = "Spatial", ST.assay = "Spatial",
                      SM.fov = "fov", ST.image = "slice1",  
                      ST.scale.factor = "hires",
                      annotations = TRUE, 
                      overlap.threshold = 0.2, 
                       add.metadata = TRUE,  merge.unique.metadata = TRUE, map.data = FALSE, new_SPT.assay = "SPT", new_SPM.assay = "SPM", verbose = FALSE){

    sample_data <- ST.data
    
    ## make polygons from coordinates
    df <- GetTissueCoordinates(SM.data, image = SM.fov)

    message("median start...")
    diff_x <- diff(df$x)
    diff_y <- diff(df$y)
    
    # Combine the x and y differences into a data frame
    differences <- data.frame(diff_x, diff_y)
    
    # Filter out cases where the difference is zero
    non_zero_differences <- differences[differences$diff_x != 0 | differences$diff_y != 0,]
    
    # Choose the non-zero difference (maximum absolute value)
    new_widths <- apply(non_zero_differences, 1, function(row) max(abs(row)))
    
    #median(new_widths)
    message("median done...")
    
    polygon_list <- lapply(1:nrow(df), function(idx) {
      row <- df[idx,]
      get_square_coordinates(center_x = row$x, center_y = row$y, width = median(new_widths), name = row$cell)
    })

    # Combine the list of data frames into a single data frame
    polygons_df <- do.call(rbind, polygon_list)
    polygons <- CreateSegmentation(polygons_df)
    message("polygons done...")
    
    SM.data[[SM.fov]][["annotation"]] <- polygons
    pol <- GetTissueCoordinates(SM.data[[SM.fov]][["annotation"]])
    
    
        ###Make list of polygons from annotations
    active_polygons <- SpatialPolygons(lapply(unique(pol$cell), function(cell_name) {
        cell_data <- pol[pol$cell == cell_name, ]
        Polygons(list(Polygon(cbind(cell_data$x, cell_data$y))), cell_name)}),
                                       1:length(unique(pol$cell)))
    
    empty_poly_df = data.frame(cell = unique(pol$cell))
    rownames(empty_poly_df) <- empty_poly_df$cell
    
    
    df_cells_spdf <- SpatialPolygonsDataFrame(active_polygons, empty_poly_df)
    message("here...")
    polygon_df <- sf::st_as_sf(df_cells_spdf)
    message("here again...")
    ## Find Cells within each polygon
    st_coordinates <- GetTissueCoordinates(sample_data[[ST.image]][["centroids"]])
    st_coordinates$radius = sample_data[[ST.image]][["centroids"]]@radius * 0.5 #### 10X scalefactors_json.json file states @radius is actually = spot diameter

    if(!is.null(ST.scale.factor)){
        if(!ST.scale.factor %in% c("hires", "lowres")){
            stop("Invalid asignment of `ST.scale.factor`! values must be either 'hires' or 'lowres'. If fullres is required set `ST.scale.factor` = `NULL`.")
        } else {
            st_coordinates[c("x","y", "radius")] <- st_coordinates[c("x","y", "radius")] * sample_data[[ST.image]]@scale.factors[[ST.scale.factor]]
        }
    }
                        
    message("points stuff...")                    
    points_sf <- sf::st_as_sf(st_coordinates, coords = c("x", "y"))
    points_sf <- sf::st_buffer(points_sf, dist = st_coordinates$radius)
    
        # Find intersecting polygons
    buffer_areas <- sf::st_area(points_sf)

    message("intersection stuff...")                              
    # Find intersecting polygons
    intersections <- sf::st_intersects(points_sf, polygon_df, sparse = FALSE)

    # Create a data frame with results
    message("results stuff...") 
    result <- do.call(rbind, lapply(seq_len(nrow(points_sf)), function(i) {
      # Get the geometry of the buffered point
      point_geom <- points_sf$geometry[i]
      point_area <- buffer_areas[i]
      
      # Find polygons intersecting with the point
      intersecting_polygons <- polygon_df[intersections[i, ], ]
      
      # Calculate intersection areas
      intersection_areas <- sf::st_area(sf::st_intersection(point_geom, intersecting_polygons$geometry))
      
      # Calculate percentage of coverage
      coverage_percentage <- as.numeric(intersection_areas / point_area)
      
      # Filter polygons by coverage percentage
      valid_polygons <- intersecting_polygons$cell[coverage_percentage >= overlap.threshold]
      
      data.frame(
        point = st_coordinates$cell[i],
        polygons = paste(valid_polygons, collapse = ", ")
      )
    }))

    message("here2")                    
    SpaMTP.obj <- AddMetaData(ST.data, result$polygons,"SPM_pixels")

    # remove cells with no SM data
    cells_with_na <- rownames(SpaMTP.obj@meta.data)[is.na(SpaMTP.obj@meta.data$SPM_pixels)]

    # Subset the Seurat object based on cells with NA values
    SpaMTP.obj <- subset(SpaMTP.obj, cells = cells_with_na, invert = TRUE)



    mean_counts <- function(spm_pixels, count_matrix) {
      # Split SPM_pixels into individual pixel strings
      pixels <- strsplit(spm_pixels, ",\\s*")[[1]]
      
      if (length(pixels) > 1){
          row_means <- rowMeans(count_matrix[,pixels]) # Compute mean for each feature
      } else if (length(pixels) == 1){
          row_means <- count_matrix[,pixels]
      } else {
          row_means <- rep(0, length(rownames(count_matrix)))
      }
      return(row_means)
    }

    if (map.data) {
        if ("data" %in% Layers(SM.data, assay = SM.assay)){
            mean_count_list <- lapply(SpaMTP.obj$SPM_pixels, mean_counts, count_matrix = SM.data[[SM.assay]]["counts"])
            mean_counts_df <- do.call(rbind, mean_count_list)
            
            mean_data_list <- lapply(SpaMTP.obj$SPM_pixels, mean_counts, count_matrix = SM.data[[SM.assay]]["data"])
            mean_data_df <- do.call(rbind, mean_count_list)
            SpaMTP.obj[["SPM"]] <- CreateAssay5Object(counts = Matrix::t(mean_counts_df), data = Matrix::t(mean_data_df))
            
        } else {
            stop("data slot not present in SPM seurat object provided! Please run NormaliseSMData() or set map.data = FALSE")  
        }
    } else {
        # Apply the function to each row in the reference data frame
        mean_count_list <- lapply(SpaMTP.obj$SPM_pixels, mean_counts, count_matrix = SM.data[[SM.assay]]["counts"])
        mean_counts_df <- do.call(rbind, mean_count_list)
        colnames(mean_counts_df) <- rownames(SM.data)
        SpaMTP.obj[[new_SPM.assay]] <- CreateAssay5Object(counts = Matrix::t(mean_counts_df))
    }

    
    #add metadata from SM

    add_metadata <- function(spm_pixels, SM_metadata) {
      # Split SPM_pixels into individual pixel strings
      pixels <- strsplit(spm_pixels, ",\\s*")[[1]]
      
      if (length(pixels) == 0){
          
          col_names <- colnames(SM_metadata)
          combined_df <- as.data.frame(matrix(NA, nrow = 1, ncol = length(col_names)))
          colnames(combined_df) <- paste0(new_SPM.assay,"_",col_names)
      
      } else {
            sub.metadata <- SM_metadata[pixels,]
            # Apply the function to each column
            if (merge.unique.metadata){
                
                combined_row <- sapply(sub.metadata, function(x){
                    paste(unique(x), collapse = ", ")
                })
            } else {
                combined_row <- sapply(sub.metadata, function(x){
                    paste(unique(x), collapse = ", ")
                })
            }
            
            # Convert to data frame
            combined_df <- as.data.frame(t(combined_row), stringsAsFactors = FALSE)
            colnames(combined_df) <- paste0(new_SPM.assay,"_",colnames(sub.metadata))
          
      }
      return(combined_df)
    }
    
                        
    if (add.metadata) {
        metadata_list <- lapply(SpaMTP.obj$SPM_pixels, add_metadata, SM_metadata = SM.data@meta.data)
        metadata_df <- do.call(rbind, metadata_list)

        SpaMTP.obj@meta.data[colnames(metadata_df)] <- metadata_df[colnames(metadata_df)]
    }

    

    if (annotations) {
        ## adds m/z annotations to new object
        SpaMTP.obj[[new_SPM.assay]]@meta.data <- SM.data[[SM.assay]]@meta.data
    }

    SpaMTP.obj <- RenameAssays(SpaMTP.obj, assay.name = ST.assay, new.assay.name = new_SPT.assay)

    
    return(SpaMTP.obj)
    
}



In [ ]:
CheckAlignmentX <- function(SM.data, ST.data, image.res = NULL, names = c("SM", "ST"), cols = NULL, image.slice = "slice1", size = 0.5){

  if (is.null(image.res)){
    scale.factor <- 1
  } else {
    if (image.res %in% c("hires", "lowres")){
      scale.factor <- ST.data@images[[image.slice]]@scale.factors[[image.res]]
    } else {
      stop("invalid input for image.res! image.res must be either 'hires' or 'lowres'")
    }
  }

  df <- GetTissueCoordinates(ST.data)[c("x", "y")] * scale.factor
  df$sample <- names[2]

  df2 <- GetTissueCoordinates(SM.data)[c("x", "y")] #* scale.factor
  df2$sample <- names[1]

  df1 <- rbind(df,df2)

  if (is.null(cols)){
    cols <- c("#F8766D", "#00BFC4")
  } else {
    cols <- cols
  }

  p <- ggplot(df1, aes(x, y,color = sample)) +
    geom_point(size = size) + theme_void() +  scale_color_manual(values = cols)
  return(p)

}


In [ ]:
switch_coords <- function(SpaMTP, fov){
    coords <- GetTissueCoordinates(SpaMTP, image = fov)
    rownames(coords) <- coords$cells
    coords$x1 <- coords$x
    coords$x <- coords$y
    coords$y <- coords$x1
    coords$x1 <- NULL
    
    cents <- SeuratObject::CreateCentroids(coords)
    
    
      segmentations.data <- list(
        "centroids" = cents
      )
    
      coords <- SeuratObject::CreateFOV(
        coords = segmentations.data,
        type = c("centroids"),
        molecules = NULL,
        assay = "Spatial"
      )
    
      SpaMTP[[fov]] <- coords

      return(SpaMTP)
}

In [ ]:
#all_data_annotated <- switch_coords(all_data_annotated, fov = "fov")
#all_data_annotated <- switch_coords(all_data_annotated, fov = "fov.2")
#all_data_annotated <- switch_coords(all_data_annotated, fov = "fov.3")
#all_data_annotated <- switch_coords(all_data_annotated, fov = "fov.4")

In [ ]:
C1_MALDIVIS_annot <- AlignData(SM.data = subset_SPM(all_data_annotated, idents = "vlp94a_dhb") ,ST.data = visium_A, 
                        SM.assay = "Spatial", ST.assay = "Spatial",
                        ST.image = "slice1", SM.fov = "fov.2", 
                        annotations = TRUE, overlap.threshold = 0.2)

In [ ]:
T1_MALDIVIS_annot <- AlignData(SM.data =  subset_SPM(all_data_annotated,idents = "vlp94c_dhb"), ST.data = visium_C, 
                        SM.assay = "Spatial", ST.assay = "Spatial",
                        ST.image = "slice1", SM.fov = "fov", 
                        annotations = TRUE, overlap.threshold = 0.2)

In [ ]:
C2_MALDIVIS_annot <- AlignData(SM.data = subset_SPM(all_data_annotated, idents = "vlp94b_dhb"), ST.data = visium_B,
                            SM.assay = "Spatial", ST.assay = "Spatial",
                            ST.image = "slice1", SM.fov = "fov.4", 
                            annotations = TRUE, overlap.threshold = 0.2)           

In [ ]:
T2_MALDIVIS_annot <- AlignData(SM.data = subset_SPM(all_data_annotated, idents = "vlp94d_dhb"), ST.data = visium_D,
                            SM.assay = "Spatial", ST.assay = "Spatial",
                            ST.image = "slice1", SM.fov = "fov.3", 
                            annotations = TRUE, overlap.threshold = 0.2)       

In [ ]:
# save MALDI_VISIUM objects
saveRDS(C1_MALDIVIS_annot, paste0(UPDATED_OUT_DIR,"C1_MALDIVIS_annot_HMDB.RDS"))
saveRDS(T1_MALDIVIS_annot,paste0(UPDATED_OUT_DIR,"T1_MALDIVIS_annot_HMDB.RDS"))
saveRDS(T2_MALDIVIS_annot, paste0(UPDATED_OUT_DIR,"T2_MALDIVIS_annot_HMDB.RDS"))
saveRDS(C2_MALDIVIS_annot, paste0(UPDATED_OUT_DIR,"C2_MALDIVIS_annot_HMDB.RDS"))

### Load Saved Data

In [ ]:
#C1_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/C1_MALDIVIS_annot.RDS"))
T1_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/T1_MALDIVIS_annot.RDS"))
#T2_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/T2_MALDIVIS_annot.RDS"))
#C2_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/C2_MALDIVIS_annot.RDS"))

In [ ]:
T1_MALDIVIS_annot@meta.data$treatment <- "Treated"
C1_MALDIVIS_annot@meta.data$treatment <- "Control"
T2_MALDIVIS_annot@meta.data$treatment <- "Treated"
C2_MALDIVIS_annot@meta.data$treatment <- "Control"

T1_MALDIVIS_annot@meta.data$sample <- "T1"
C1_MALDIVIS_annot@meta.data$sample <- "C1"
T2_MALDIVIS_annot@meta.data$sample <- "T2"
C2_MALDIVIS_annot@meta.data$sample <- "C2"

In [ ]:
merged_MALDIVIS <- JoinLayers(merge(T1_MALDIVIS_annot, y = list(C1_MALDIVIS_annot, T2_MALDIVIS_annot,C2_MALDIVIS_annot), merge.data = TRUE))

In [ ]:
saveRDS(merged_MALDIVIS, paste0(UPDATED_OUT_DIR,"merged_MALDIVIS_HMDB.RDS"))

## Spatially Plot the Merged MALDI and Visium Data

After merging the data we now have each tissue sample with spots that contatin both transcriptomic and metabolomic infomation. We can plot metabolites like beore using SpaMTP functions

In [ ]:
FindNearestMZ(C1_MALDIVIS_annot, target_mz = 448.246, assay = "SPM")

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

DefaultAssay(T1_MALDIVIS_annot) <- "SPM"

SpatialFeaturePlot(T1_MALDIVIS_annot, features = FindNearestMZ(T1_MALDIVIS_annot,target_mz = 448.246), pt.size.factor = 2.2, slot = "counts",image.alpha = 0)

Again we can plot multiple m/z values, with a plusminus range and also using the metabolite annotation name

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 10)
SpatialMZPlot(T1_MALDIVIS_annot, mzs = c(448.24, 700),assay = "SPM",  plusminus = 0.05, slot = "counts") + SpatialMZAnnotationPlot(T1_MALDIVIS_annot, assay = "SPM",metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", images = NULL)

In [ ]:
C1_xy <- FetchData(C1_MALDIVIS_annot, vars = c('nCount_SPM','nCount_SPT'))
C2_xy <- FetchData(C2_MALDIVIS_annot, vars = c('nCount_SPM','nCount_SPT'))
T1_xy <- FetchData(T1_MALDIVIS_annot, vars = c('nCount_SPM','nCount_SPT'))
T2_xy <- FetchData(T2_MALDIVIS_annot, vars = c('nCount_SPM','nCount_SPT'))

In [ ]:
C1_results <- rcorr(as.matrix(C1_xy), type=c("pearson"))
T1_results <- rcorr(as.matrix(T1_xy), type=c("pearson"))
C2_results <- rcorr(as.matrix(C2_xy), type=c("pearson"))
T2_results <- rcorr(as.matrix(T2_xy), type=c("pearson"))

## Have to set all the NA values in the P-value matrix to 0 
C1_results$P[is.na(C1_results$P)] <- 0
T1_results$P[is.na(T1_results$P)] <- 0
T2_results$P[is.na(T2_results$P)] <- 0
C2_results$P[is.na(C2_results$P)] <- 0

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

par(mfrow = c(2, 2))
corrplot(C1_results$r, type = "upper", p.mat = C1_results$P, sig.level = 0.05)
title(main = "C1", cex.main = 1.5)

corrplot(T1_results$r, type = "upper", p.mat = T1_results$P, sig.level = 0.05)
title(main = "T1", cex.main = 1.5)

corrplot(C2_results$r, type = "upper", p.mat = C2_results$P, sig.level = 0.05)
title(main = "C2", cex.main = 1.5)

corrplot(T2_results$r, type = "upper", p.mat = T2_results$P, sig.level = 0.05)
title(main = "T2", cex.main = 1.5)

par(mfrow = c(1, 1))

## Cell State Enrichment Scores

Based on the transcriptome, we can run cell state enrichment scores for each spot using the transcriptomics data. These results were previously run and the results will be imported and added to these SpaMTP objects

In [ ]:
enrichment_scores <-  read.table("/QRISdata/Q1851/Andrew_C/Metabolomics/SHH_gsea_v2_scores.txt", header = TRUE, sep = "\t")

In [ ]:
## This code is for reformating the table 
row_names <- rownames(enrichment_scores)
parts <- strsplit(row_names, "_")
sample <- sapply(parts, function(x) x[1])
condition <- sapply(parts, function(x) x[2])
barcodes <- sapply(parts, function(x) x[length(x)])

# Add new columns to the data frame
enrichment_scores$sample <- sample
enrichment_scores$condition <- condition
enrichment_scores$barcodes <- barcodes

Lets add them to each object

In [ ]:
### This function is for adding the relative enrichment scores to each SpaMTP object
add_enrichment_scores <- function(vis_data, name) {
    
    df <- enrichment_scores[enrichment_scores$sample == name, ]
    rownames(df) <- df$barcode
    rows <- intersect(rownames(df), rownames(vis_data@meta.data))
    df <- df[rows,]
    sub <- subset(vis_data, cells = rows)
    sub@meta.data[c('GOBP_NEURON_DIFFERENTIATION.enrich.scores',
                     'GOBP_NEUROTRANSMITTER_TRANSPORT.enrich.scores',
                     'HALLMARK_E2F_TARGETS.enrich.scores',
                     'SHH.A.enrich.scores',
                     'SHH.B.enrich.scores',
                     'SHH.C.enrich.scores')] <- df[ c('GOBP_NEURON_DIFFERENTIATION.enrich.scores',
                     'GOBP_NEUROTRANSMITTER_TRANSPORT.enrich.scores',
                     'HALLMARK_E2F_TARGETS.enrich.scores',
                     'SHH.A.enrich.scores',
                     'SHH.B.enrich.scores',
                     'SHH.C.enrich.scores')]
    
    
  return(sub)
}

In [ ]:
C1_MALDIVIS_annot <- add_enrichment_scores(C1_MALDIVIS_annot, "C1") #these sample names are slightly different to the ones used above... C1 = C1, D1 = C2, A1 = T1 and B1 = T2
C2_MALDIVIS_annot <- add_enrichment_scores(C2_MALDIVIS_annot, "D1")
T1_MALDIVIS_annot <- add_enrichment_scores(T1_MALDIVIS_annot, "A1")
T2_MALDIVIS_annot <- add_enrichment_scores(T2_MALDIVIS_annot, "B1")

Lets now visualise the differences in cell state between the <span style = "color: blue;">**Control**</span> samples (*left*) and <span style = "color: red;">**Treated**</span> samples (*right*)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20)

## Scoring plots for SSH-A
(SpatialFeaturePlot(C1_MALDIVIS_annot, features = "SHH.A.enrich.scores") | 
SpatialFeaturePlot(C2_MALDIVIS_annot, features = "SHH.A.enrich.scores")|
SpatialFeaturePlot(T1_MALDIVIS_annot, features = "SHH.A.enrich.scores") |
SpatialFeaturePlot(T2_MALDIVIS_annot, features = "SHH.A.enrich.scores"))/

## Scoring plots for SSH-B
(SpatialFeaturePlot(C1_MALDIVIS_annot, features = "SHH.B.enrich.scores") | 
SpatialFeaturePlot(C2_MALDIVIS_annot, features = "SHH.B.enrich.scores")|
SpatialFeaturePlot(T1_MALDIVIS_annot, features = "SHH.B.enrich.scores") |
SpatialFeaturePlot(T2_MALDIVIS_annot, features = "SHH.B.enrich.scores"))/

## Scoring plots for SSH-C
(SpatialFeaturePlot(C1_MALDIVIS_annot, features = "SHH.C.enrich.scores") | 
SpatialFeaturePlot(C2_MALDIVIS_annot, features = "SHH.C.enrich.scores")|
SpatialFeaturePlot(T1_MALDIVIS_annot, features = "SHH.C.enrich.scores") |
SpatialFeaturePlot(T2_MALDIVIS_annot, features = "SHH.C.enrich.scores")) /

## Scoring plots for Palbociclib
(SpatialMZAnnotationPlot(C1_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", assay = "SPM", images = NULL) | 
SpatialMZAnnotationPlot(C2_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", assay = "SPM",images = NULL)|
SpatialMZAnnotationPlot(T1_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", assay = "SPM",images = NULL) |
SpatialMZAnnotationPlot(T2_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", assay = "SPM",images = NULL))

We can also combine these scores with the relative expression of ***Palbociclib***. 

In the plot below the colour refers to the expression intensity of ***Palbociclib*** and the relative spot size indicates the enrichment of SHH-B (where larger spots = more expression)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 8)

(SpatialMZAnnotationPlot(C1_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, assay = "SPM", slot = "counts", images = NULL, pt.size.factor = ifelse(C1_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']] > 0, C1_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']]/300 * 2 , 0.25))|
SpatialMZAnnotationPlot(C2_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, assay = "SPM", slot = "counts", images = NULL, pt.size.factor = ifelse(C2_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']] > 0, C2_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']]/300 * 2 , 0.25))|
SpatialMZAnnotationPlot(T1_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, assay = "SPM", slot = "counts", images = NULL, pt.size.factor = ifelse(T1_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']] > 0, T1_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']]/300 * 2 , 0.25))|
SpatialMZAnnotationPlot(T2_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, assay = "SPM", slot = "counts", images = NULL, pt.size.factor = ifelse(T2_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']] > 0, T2_MALDIVIS_annot@meta.data[['SHH.B.enrich.scores']]/300 * 2 , 0.25)) )& scale_fill_gradientn(colors = viridis(100), limits = c(0, 200)) 

### Correlation Analysis 

Here we are pulling the data from each object to use for correlation analysis

In [ ]:
C1_xy <- FetchData(C1_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
C2_xy <- FetchData(C2_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
T1_xy <- FetchData(T1_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
T2_xy <- FetchData(T2_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))

In [ ]:
C1_results <- rcorr(as.matrix(C1_xy), type=c("pearson"))
T1_results <- rcorr(as.matrix(T1_xy), type=c("pearson"))
C2_results <- rcorr(as.matrix(C2_xy), type=c("pearson"))
T2_results <- rcorr(as.matrix(T2_xy), type=c("pearson"))

In [ ]:
T2_results

In [ ]:
df <- rcorr(cbind(as.vector(C1_results$r), 
                                         as.vector(C2_results$r),
                                         as.vector(T1_results$r),
                                         as.vector(T2_results$r)))

In [ ]:
df

In [ ]:
rownames(df$r) <- colnames(df$r) <- c("C1", "C2", "T1", "T2")
rownames(df$P) <- colnames(df$P) <- c("C1", "C2", "T1", "T2")

In [ ]:
corrplot(df$r, type = "upper", p.mat = df$P, sig.level = 0.05)
title(main = "C1", cex.main = 1.5)

In [ ]:
## Have to set all the NA values in the P-value matrix to 0 
C1_results$P[is.na(C1_results$P)] <- 0
T1_results$P[is.na(T1_results$P)] <- 0
T2_results$P[is.na(T2_results$P)] <- 0
C2_results$P[is.na(C2_results$P)] <- 0

In [ ]:
df_results <- rcorr(as.matrix(df), type=c("pearson"))
## Have to set all the NA values in the P-value matrix to 0 
df_results$P[is.na(df_results$P)] <- 0



Next we can run the correlation analysis and then plot the results 

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

par(mfrow = c(2, 2))
corrplot(C1_results$r, type = "upper", p.mat = C1_results$P, sig.level = 0.05)
title(main = "C1", cex.main = 1.5)

corrplot(T1_results$r, type = "upper", p.mat = T1_results$P, sig.level = 0.05)
title(main = "T1", cex.main = 1.5)

corrplot(C2_results$r, type = "upper", p.mat = C2_results$P, sig.level = 0.05)
title(main = "C2", cex.main = 1.5)

corrplot(T2_results$r, type = "upper", p.mat = T2_results$P, sig.level = 0.05)
title(main = "T2", cex.main = 1.5)

par(mfrow = c(1, 1))

Based on these correlation scores we can see that there is no significant difference between ***Palbociclib*** expression and the SSH cell state enrichment scores in the <span style = "color: blue;">**Control**</span> samples (*left*). Comparitively, there is a significant correlation between ***Palbociclib*** expression and SSH-B and SSH-C enrichment scores in the <span style = "color: red;">**Treated**</span> samples (*right*)


## Process Visium Data

In [ ]:
run_qc <- function(data, min_genes, min_spots, plot = TRUE) {
    
    DefaultAssay(data) <- "SPT"
    # QC - removing Mitochondrial and Ribosomal Genes
    print("QC - removing Mitochondrial and Ribosomal Genes...")
    data <- PercentageFeatureSet(data, ".*MT-", col.name = "percent_mito", assay="SPT")
    data <- PercentageFeatureSet(data, ".*RP[SL]", col.name = "percent_ribo", assay="SPT")

    if (plot){
         ## check the feature and count distribution before qc
        feats <- c("nFeature_SPT", "nCount_SPT")
        print(VlnPlot(data, features = c(feats, "percent_mito")))
        
    }
   
        
    # filtering 
    selected_c <- WhichCells(data, expression = nFeature_SPT > min_genes)
    length(selected_c)
    
    selected_f <- rownames(data)[Matrix::rowSums(data) > min_spots]
    length(selected_f)
    
    data <- subset(data, features = selected_f, cells = selected_c)
    
    selected_c2=rownames(data@meta.data)[data$orig.ident!='']
    data <- subset(data,  cells = selected_c2)
    print("Keeping the filtered spots in Seurat object...")
    
    return(data)
}

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)

C1_MALDIVIS_annot <- run_qc(C1_MALDIVIS_annot, min_genes = 20, min_spots = 3)
T1_MALDIVIS_annot <- run_qc(T1_MALDIVIS_annot, min_genes = 20, min_spots = 3)
C2_MALDIVIS_annot <- run_qc(C2_MALDIVIS_annot, min_genes = 20, min_spots = 3)
T2_MALDIVIS_annot <- run_qc(T2_MALDIVIS_annot, min_genes = 20, min_spots = 3)

In [ ]:
process_data <- function(data, assay = "SPT") {
    DefaultAssay(data) <- assay
    data <- NormalizeData(data)
    data <- ScaleData(data)
    data <- FindVariableFeatures(data, verbose = FALSE, selection.method = "vst")
    data <- RunPCA(data, assay = assay, verbose = FALSE)
    data <- FindNeighbors(data, reduction = "pca", dims = 1:30, verbose = FALSE)
    data <- RunUMAP(data, reduction = "pca", dims = 1:30, verbose = FALSE)
    return(data)
}

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)

C1_MALDIVIS_annot <- process_data(C1_MALDIVIS_annot)
T1_MALDIVIS_annot <- process_data(T1_MALDIVIS_annot)
C2_MALDIVIS_annot <- process_data(C2_MALDIVIS_annot)
T2_MALDIVIS_annot <- process_data(T2_MALDIVIS_annot)

In [ ]:
get_organism_percent <- function(data, assay = "SPT", organism = "human"){

    if (organism == "human") {
        reg_ex <- "^hg38-"        
    } else if (organism == "mouse") {
         reg_ex <- "^mm10-"     
    } else {
        stop("must select either `human` or `mouse` as an organism")
    }
    
    data <- PercentageFeatureSet(data, assay = assay, pattern = reg_ex, col.name = paste0("percent.",organism))
    data_sub <- subset(data, cells = rownames(na.omit(data@meta.data)))

    return(data_sub)
}


In [ ]:
## Add Human gene percentage
C1_MALDIVIS_annot <- get_organism_percent(C1_MALDIVIS_annot, organism = "human")
T1_MALDIVIS_annot <- get_organism_percent(T1_MALDIVIS_annot, organism = "human")
C2_MALDIVIS_annot <- get_organism_percent(C2_MALDIVIS_annot, organism = "human")
T2_MALDIVIS_annot <- get_organism_percent(T2_MALDIVIS_annot, organism = "human")

## Add Mouse gene percentage
C1_MALDIVIS_annot <- get_organism_percent(C1_MALDIVIS_annot, organism = "mouse")
T1_MALDIVIS_annot <- get_organism_percent(T1_MALDIVIS_annot, organism = "mouse")
C2_MALDIVIS_annot <- get_organism_percent(C2_MALDIVIS_annot, organism = "mouse")
T2_MALDIVIS_annot <- get_organism_percent(T2_MALDIVIS_annot, organism = "mouse")

In [ ]:
merged_MALDIVIS <- JoinLayers(merge(T1_MALDIVIS_annot, y = list(C1_MALDIVIS_annot, T2_MALDIVIS_annot,C2_MALDIVIS_annot), merge.data = TRUE))

## Split Organism Spots

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

(SpatialFeaturePlot(C1_MALDIVIS_annot, features = "percent.human") / SpatialFeaturePlot(C1_MALDIVIS_annot, features = "percent.mouse"))|
(SpatialFeaturePlot(T1_MALDIVIS_annot, features = "percent.human") / SpatialFeaturePlot(T1_MALDIVIS_annot, features = "percent.mouse"))|
(SpatialFeaturePlot(C2_MALDIVIS_annot, features = "percent.human") / SpatialFeaturePlot(C2_MALDIVIS_annot, features = "percent.mouse"))|
(SpatialFeaturePlot(T2_MALDIVIS_annot, features = "percent.human") / SpatialFeaturePlot(T2_MALDIVIS_annot, features = "percent.mouse"))

In [ ]:
keep_organism_genes <- function(data, organism = "human",  assay = "SPT", slot = "counts") {

    DefaultAssay(data) <- assay
    
    if (organism == "human") {
        
        sub_genes <- rownames(data)[grep("^hg38-", rownames(data))]
        counts <- data[[assay]][slot][sub_genes,]
        rownames(counts) <-  gsub("^hg38-", "", rownames(counts))# changes names of human genes so that genes dont have "hg38-" at start
        data[[organism]] <- CreateAssayObject(counts = counts)
        
    } else if (organism == "mouse") {

        sub_genes <- rownames(data)[grep("^mm10-", rownames(data))]
        counts <- data[[assay]][slot][sub_genes,]
        rownames(counts) <-  gsub("^mm10-", "", rownames(counts)) # changes names of human genes so that genes dont have "hg38-" at start
        data[[organism]] <- CreateAssayObject(counts = counts)
        
    } else {
        stop("must select either `human` or `mouse` as an organism")
    }
    
    return(data)
}

In [ ]:
#This function removes all non-human genes from a seurat object 
subset_organism_genes <- function(data, organism = "human", percent.threshold = 10, assay = "SPT") {

    DefaultAssay(data) <- assay
    
    if (organism == "human") {
        
        data_sub <- subset(data, subset = percent.human >= percent.threshold) #subsets to only include cells that have > percent.threshold human genes
        #sub_genes <- rownames(data_sub)[grep("^hg38-", rownames(data_sub))]  #gets list of only human genes to subset later
        rownames(data_sub) <-  gsub("^hg38-", "", rownames(data_sub)) # changes names of human genes so that genes dont have "hg38-" at start
        
        
    } else if (organism == "mouse") {
        
        data_sub <- subset(data, subset = percent.mouse >= percent.threshold)
        #sub_genes <- rownames(data_sub)[grep("^mm10-", rownames(data_sub))]
        rownames(data_sub) <-  gsub("^mm10-", "", rownames(data_sub))
        
    } else {
        stop("must select either `human` or `mouse` as an organism")
    }

    DefaultAssay(data_sub) <- assay
    new_count_sum <- Matrix::colSums(data_sub)
    data_sub@meta.data$organism_counts <- new_count_sum 
    data_sub@meta.data$organism <- organism 
    
    return(data_sub)
}

In [ ]:
C1_human <- keep_organism_genes(C1_MALDIVIS_annot)
T1_human <- keep_organism_genes(T1_MALDIVIS_annot)
C2_human <- keep_organism_genes(C2_MALDIVIS_annot)
T2_human <- keep_organism_genes(T2_MALDIVIS_annot)

In [ ]:
C1_mouse <- keep_organism_genes(C1_MALDIVIS_annot, organism = "mouse")
T1_mouse<- keep_organism_genes(T1_MALDIVIS_annot, organism = "mouse")
C2_mouse <- keep_organism_genes(C2_MALDIVIS_annot, organism = "mouse")
T2_mouse <- keep_organism_genes(T2_MALDIVIS_annot, organism = "mouse")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

(SpatialFeaturePlot(C1_human, features = "nCount_human") / SpatialFeaturePlot(C1_human, features = "nFeature_human"))|
(SpatialFeaturePlot(T1_human, features = "nCount_human") / SpatialFeaturePlot(T1_human, features = "nFeature_human"))|
(SpatialFeaturePlot(C2_human, features = "nCount_human") / SpatialFeaturePlot(C2_human, features = "nFeature_human"))|
(SpatialFeaturePlot(T2_human, features = "nCount_human") / SpatialFeaturePlot(T2_human, features = "nFeature_human"))

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

(SpatialFeaturePlot(C1_mouse, features = "nCount_mouse") / SpatialFeaturePlot(C1_mouse, features = "nFeature_mouse"))|
(SpatialFeaturePlot(T1_mouse, features = "nCount_mouse") / SpatialFeaturePlot(T1_mouse, features = "nFeature_mouse"))|
(SpatialFeaturePlot(C2_mouse, features = "nCount_mouse") / SpatialFeaturePlot(C2_mouse, features = "nFeature_mouse"))|
(SpatialFeaturePlot(T2_mouse, features = "nCount_mouse") / SpatialFeaturePlot(T2_mouse, features = "nFeature_mouse"))

In [ ]:
C1_MALDIVIS_human <- subset_organism_genes(C1_MALDIVIS_annot, percent.threshold = 60)
T1_MALDIVIS_human <- subset_organism_genes(T1_MALDIVIS_annot, percent.threshold = 60)
C2_MALDIVIS_human <- subset_organism_genes(C2_MALDIVIS_annot, percent.threshold = 60)
T2_MALDIVIS_human <- subset_organism_genes(T2_MALDIVIS_annot, percent.threshold = 60)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

drug <- (SpatialFeaturePlot(C1_MALDIVIS_human, features = "mz-448.246729024761", slot = "data", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(C2_MALDIVIS_human, features = "mz-448.246729024761", slot = "data", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T1_MALDIVIS_human, features = "mz-448.246729024761", slot = "data", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T2_MALDIVIS_human, features = "mz-448.246729024761", slot = "data", crop = FALSE, pt.size.factor = 1.3)) &scale_fill_gradientn(colors = viridis(100), limits = c(0,100), na.value = viridis(100)[100])  

drug
#ggsave(drug, filename = paste0(OUT_DIR, "plots/drug_expression.pdf"), height = 7, width = 30)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

plot <- (SpatialFeaturePlot(C1_MALDIVIS_human, features = "mz-449.247434602517", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(C2_MALDIVIS_human, features = "mz-449.247434602517", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T1_MALDIVIS_human, features = "mz-449.247434602517", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T2_MALDIVIS_human, features = "mz-449.247434602517", slot = "counts", crop = FALSE, pt.size.factor = 1.3)) &scale_fill_gradientn(colors = viridis(100), limits = c(0, 80))  

plot
#ggsave(drug, filename = paste0(OUT_DIR, "plots/drug_expression.pdf"), height = 7, width = 30)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

plot <- (SpatialFeaturePlot(C1_MALDIVIS_human, features = "mz-1383.83486970307", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(C2_MALDIVIS_human, features = "mz-1383.83486970307", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T1_MALDIVIS_human, features = "mz-1383.83486970307", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T2_MALDIVIS_human, features = "mz-1383.83486970307", slot = "counts", crop = FALSE, pt.size.factor = 1.3)) &scale_fill_gradientn(colors = viridis(100), limits = c(0, 300))  

plot
#ggsave(drug, filename = paste0(OUT_DIR, "plots/drug_expression.pdf"), height = 7, width = 30)

In [ ]:
C1_MALDIVIS_human <- bin_metabolites(C1_MALDIVIS_human, TG_up$X[1:20], assay = "SPM", bin_name = "TG's")
T1_MALDIVIS_human <- bin_metabolites(T1_MALDIVIS_human, TG_up$X[1:20], assay = "SPM", bin_name = "TG's")
C2_MALDIVIS_human <- bin_metabolites(C2_MALDIVIS_human, TG_up$X[1:20], assay = "SPM", bin_name = "TG's")
T2_MALDIVIS_human <- bin_metabolites(T2_MALDIVIS_human, TG_up$X[1:20], assay = "SPM", bin_name = "TG's")


In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

plot <- (SpatialFeaturePlot(C1_MALDIVIS_human, features = "TG's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(C2_MALDIVIS_human, features = "TG's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T1_MALDIVIS_human, features = "TG's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T2_MALDIVIS_human, features = "TG's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)) &scale_fill_gradientn(colors = viridis(100), limits = c(0, 250))  

plot
#ggsave(drug, filename = paste0(OUT_DIR, "plots/drug_expression.pdf"), height = 7, width = 30)

In [ ]:
all_tumour_DEPs$Treated$DEPs[all_tumour_DEPs$Treated$DEPs$regulate == "Down",]

In [ ]:
GM2 <- SearchAnnotations(all_data_annotated, "Ganglioside GM2", search.exact = FALSE)

In [ ]:
C1_MALDIVIS_human <- bin_metabolites(C1_MALDIVIS_human, GM2$mz_names, assay = "SPM", bin_name = "Ganglioside GM2's")
T1_MALDIVIS_human <- bin_metabolites(T1_MALDIVIS_human, GM2$mz_names, assay = "SPM", bin_name = "Ganglioside GM2's")
C2_MALDIVIS_human <- bin_metabolites(C2_MALDIVIS_human, GM2$mz_names, assay = "SPM", bin_name = "Ganglioside GM2's")
T2_MALDIVIS_human <- bin_metabolites(T2_MALDIVIS_human, GM2$mz_names, assay = "SPM", bin_name = "Ganglioside GM2's")


In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

plot <- (SpatialFeaturePlot(C1_MALDIVIS_human, features = "Ganglioside GM2's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(C2_MALDIVIS_human, features = "Ganglioside GM2's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T1_MALDIVIS_human, features = "Ganglioside GM2's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)|
SpatialFeaturePlot(T2_MALDIVIS_human, features = "Ganglioside GM2's", slot = "counts", crop = FALSE, pt.size.factor = 1.3)) &scale_fill_gradientn(colors = viridis(100), limits = c(0,250))  

plot
#ggsave(drug, filename = paste0(OUT_DIR, "plots/drug_expression.pdf"), height = 7, width = 30)

## Clustering of Merged SpaMTP Object

We can then perform clustering on the sample to identify spots which contain similar transcriptional and metabolomic profiles. 

Clustering using ST vs SP can be quite different. Programs like *Cardinal* use a function called spatial shrunken centroids (ssc). Here we will compare that with *Seurats* clustering methods, and show the changes need to get similar quality results. In order to cluster the spaital metabolomics we must increase the resolution of the peak bin to avoid technical noise being introduced into the clustering results.

First we will use a peak bin size = 50 to generate the clusters:

### Cluster MALDI pixel level

In [ ]:
## Reimport data for clusterings with resolution = 50ppm
C1 <- Cardinal::readImzML("vlp94a_dhb",folder = DATA_DIR, mass.range = c(160,1500), resolution = 50)
T1 <- Cardinal::readImzML("vlp94c_dhb",folder = DATA_DIR, mass.range = c(160,1500), resolution = 50)
C2 <- Cardinal::readImzML("vlp94b_dhb",folder = DATA_DIR, mass.range = c(160,1500), resolution = 50)
T2 <- Cardinal::readImzML("vlp94d_dhb",folder = DATA_DIR, mass.range = c(160,1500), resolution = 50)

In [ ]:
## Merge Data into one Cardinal Object (used for ssc annotation)
data <- MergeCardinalData(list(C1, T1, C2, T2), shift.image = FALSE)

In [ ]:
#read ssc cardinal object
data_ssc <- readRDS(paste0(OUT_DIR,"ssc/dhb_ssc.RDS"))

#And add this data to the merged object
test <- add_ssc_annotation(data, data_ssc, resolution = 25)

In [ ]:
# Create seurat objects for clustering
C1_test <- CardinalToSeurat(test,"vlp94a_dhb", seurat.coord = mapped_A)
T1_test <- CardinalToSeurat(test,"vlp94c_dhb", seurat.coord = mapped_C)
C2_test <- CardinalToSeurat(test,"vlp94b_dhb", seurat.coord = mapped_B)
T2_test <- CardinalToSeurat(test,"vlp94d_dhb", seurat.coord = mapped_D)

Lets look at the orginal ssc clusterings from *Cardinal*

In [ ]:

options(repr.plot.width = 10, repr.plot.height = 10)

ImageDimPlot(T1_test, group.by = "ssc", size = 2)

We now need to perform the standard *Seurat* functioning to calculate clusters.

First we need to Normalise the data

In [ ]:
T1_test <- NormalizeSeuratData(T1_test)
C1_test <- NormalizeSeuratData(C1_test)
T2_test <- NormalizeSeuratData(T2_test)
C2_test <- NormalizeSeuratData(C2_test)

The next step is to then perform clusterings, we will use this function below for each sample.

In [ ]:
cluster_seurat_obj <- function (data, res = 0.35, pcas = 50, clust_name = "seurat_clusters" ){
    data <- FindVariableFeatures(data, verbose = FALSE)
    data <- ScaleData(data, verbose = FALSE)
    data <- RunPCA(data, npcs = pcas, verbose = FALSE, reduction.name = )
    data <- FindNeighbors(data, dims = 1:pcas, verbose = FALSE)
    data <- RunUMAP(data, dims = 1:pcas, verbose = FALSE)
    data <- FindClusters(data, resolution = res, verbose = FALSE, cluster.name = clust_name)
    return(data)
}

In [ ]:
T1_test <- cluster_seurat_obj(T1_test)
C1_test <- cluster_seurat_obj(C1_test)
T2_test <- cluster_seurat_obj(T2_test)
C2_test <- cluster_seurat_obj(C2_test)

In [ ]:
merged_data <- JoinLayers(merge(T1_test, y = list(C1_test, T2_test,C2_test), merge.data = TRUE))

In [ ]:
merged_data@meta.data$seurat_clusters_individual <- merged_data@meta.data$seurat_clusters 

In [ ]:
merged_data <- cluster_seurat_obj(merged_data)

In [ ]:
options(repr.plot.width = 40, repr.plot.height = 10)
DimPlot(merged_data, group.by = "seurat_clusters_individual") | 
ImageDimPlot(merged_data, fov = c("fov"), group.by = "seurat_clusters_individual", size = 2)| 
ImageDimPlot(merged_data, fov = c("fov.2"), group.by = "seurat_clusters_individual", size = 2) |
ImageDimPlot(merged_data, fov = c("fov.3"), group.by = "seurat_clusters_individual", size = 2) |
ImageDimPlot(merged_data, fov = c("fov.4"), group.by = "seurat_clusters_individual", size = 2) 

In [ ]:
options(repr.plot.width = 40, repr.plot.height = 10)
DimPlot(merged_data, group.by = "seurat_clusters") | 
ImageDimPlot(merged_data, fov = c("fov"), group.by = "seurat_clusters", size = 2)| 
ImageDimPlot(merged_data, fov = c("fov.2"), group.by = "seurat_clusters", size = 2) |
ImageDimPlot(merged_data, fov = c("fov.3"), group.by = "seurat_clusters", size = 2) |
ImageDimPlot(merged_data, fov = c("fov.4"), group.by = "seurat_clusters", size = 2) 

In [ ]:
saveRDS(merged_data, paste0(OUT_DIR,"data_objects/merged_clusters.RDS"))

In [ ]:
all_data_annotated@meta.data$seurat_clusters <- merged_data@meta.data$seurat_clusters

In [ ]:
head(all_data_annotated@meta.data)

In [ ]:
suppressWarnings({
    Idents(all_data_annotated) <- "seurat_clusters"
    tumour <- subset(all_data_annotated, subset = seurat_clusters %in% c("3","0"))
})


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6)
ImageDimPlot(tumour, fov = c("fov", "fov.2", "fov.3", "fov.4"), size = 2)

In [ ]:
tumour_DEPs <- FindAllDEPs(tumour, "seurat_clusters", n = 3, logFC_threshold = 1.2, DE_output_dir =NULL,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames")


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 10)

keyvals <- ifelse(
    tumour_DEPs[["3"]]$DEPs$logFC < 0  & tumour_DEPs[["3"]]$DEPs$PValue < 10e-5, 'royalblue',
      ifelse(tumour_DEPs[["3"]]$DEPs$logFC > 0 & tumour_DEPs[["3"]]$DEPs$PValue < 10e-5, 'red',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == 'red'] <- 'Up'
  names(keyvals)[keyvals == 'black'] <- 'Normal'
  names(keyvals)[keyvals == 'royalblue'] <- 'Down'


volcPlot <- EnhancedVolcano::EnhancedVolcano(tumour_DEPs[["3"]]$DEPs,
                                             #selectLab = c("mz-448.246729024761", "mz-449.247434602517", "mz-1383.83486970307"), 
    #lab = rep("", length(rownames(all_tumour_DEPs$Treated$DEPs))),
                                             lab = rownames(tumour_DEPs[["3"]]$DEPs),
    FCcutoff = 0,
    colCustom = keyvals,
    cutoffLineType = 'blank',
    pCutoff = 10e-5,
    x = 'logFC',
    y = 'PValue', 
    gridlines.major = FALSE,
    gridlines.minor = FALSE)

In [ ]:
volcPlot

In [ ]:
tumour_DEPs[["3"]]$DEPs

now we can visualise the clustering results, and compare them to the ssc clustering of *Cardinal*

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 15)

(DimPlot(T1_test)/ImageMZPlot(T1_test, fov = "fov", mzs  = c(448.25), size = 2)) |(ImageDimPlot(T1_test, size = 2)/ImageDimPlot(T1_test, group.by = "ssc",size = 2))

In [ ]:
(DimPlot(T2_test)/ImageMZPlot(T2_test, fov = "fov", mzs  = c(448.25), size = 2)) |(ImageDimPlot(T2_test, size = 2)/ImageDimPlot(T2_test, group.by = "ssc",size = 2))

In [ ]:
(ImageDimPlot(C1_test, size = 2)/ImageDimPlot(C1_test, group.by = "ssc",size = 2))|(ImageDimPlot(C2_test, size = 2)/ImageDimPlot(C2_test, group.by = "ssc",size = 2))

we can also merge the samples and cluster together for coorelated clustering results

In [ ]:
all_data_clustering <- JoinLayers(merge(T1_test, y=list(C1_test,T2_test,C2_test), merge.data = TRUE))

In [ ]:
all_data_clustering <- cluster_seurat_obj(all_data_clustering, res = 0.5)

lets check out the clustering

In [ ]:
(ImageDimPlot(all_data_clustering, fov = "fov.2", size = 2) | ImageDimPlot(all_data_clustering, fov = "fov.4", size = 2) | ImageDimPlot(all_data_clustering, fov = "fov", size = 2) | ImageDimPlot(all_data_clustering, fov = "fov.3", size = 2))/
(ImageDimPlot(all_data_clustering, group.by = "ssc",fov = "fov.2", size = 2) | ImageDimPlot(all_data_clustering, group.by = "ssc",fov = "fov.4", size = 2) | ImageDimPlot(all_data_clustering, group.by = "ssc",fov = "fov", size = 2) | ImageDimPlot(all_data_clustering, group.by = "ssc",fov = "fov.3", size = 2))

### Cluster Spot Level

In [ ]:
unique(merged_data$sample)

We can do the same with our merged ST and SP object, lets do that so we can compare directly with the ST clustering results

In [ ]:
T1_test_x <- AlignSpatialOmics(subset_SPM(merged_data, subset = sample == "vlp94c_dhb"),visium_C, img_res = "hires", res_increase = 4, assay = "Spatial", slice = "slice1", annotations = FALSE, slots = c("counts"), ST.layers = c("counts"))
gc()

In [ ]:
C1_test_x <- AlignSpatialOmics(subset_SPM(merged_data, subset = sample == "vlp94a_dhb"), visium_A, img_res = "hires", res_increase = 4, assay = "Spatial", slice = "slice1", annotations = FALSE, slots = c("counts"), ST.layers = c("counts"))
gc()


In [ ]:
T2_test_x <- AlignSpatialOmics(subset_SPM(merged_data, subset = sample == "vlp94d_dhb"), visium_D,img_res = "hires",  res_increase = 4, assay = "Spatial", slice = "slice1", annotations = FALSE, slots = c("counts"), ST.layers = c("counts"))
gc()


In [ ]:
C2_test_x <- AlignSpatialOmics(subset_SPM(merged_data, subset = sample == "vlp94b_dhb"), visium_B, img_res = "hires",  res_increase = 4, assay = "Spatial", slice = "slice1", annotations = FALSE, slots = c("counts"), ST.layers = c("counts"))
gc()

We will again save our data because the above steps take a long time

In [ ]:
# Save individual SpaMTP objects
saveRDS(T1_test_x, paste0(OUT_DIR,"data_objects/T1_Merged_clusters.RDS"))
saveRDS(C1_test_x, paste0(OUT_DIR,"data_objects/C1_Merged_clusters.RDS"))
saveRDS(T2_test_x, paste0(OUT_DIR,"data_objects/T2_Merged_clusters.RDS"))
saveRDS(C2_test_x, paste0(OUT_DIR,"data_objects/C2_Merged_clusters.RDS"))

### Load Saved Data

In [ ]:
T1_test_x <- readRDS( paste0(OUT_DIR,"data_objects/T1_Merged_clusters.RDS"))
C1_test_x<- readRDS(  paste0(OUT_DIR,"data_objects/C1_Merged_clusters.RDS"))
T2_test_x<- readRDS(  paste0(OUT_DIR,"data_objects/T2_Merged_clusters.RDS"))
C2_test_x<- readRDS(  paste0(OUT_DIR,"data_objects/C2_Merged_clusters.RDS"))

In [ ]:
#adds a data slot to each object (this is the same as the counts mtx)
T1_test_x <- NormalizeSeuratData(T1_test_x)
C1_test_x <- NormalizeSeuratData(C1_test_x)
T2_test_x <- NormalizeSeuratData(T2_test_x)
C2_test_x <- NormalizeSeuratData(C2_test_x)

In [ ]:
T1_test_x@meta.data$sample <- "T1"
C1_test_x@meta.data$sample <- "C1"
T2_test_x@meta.data$sample <- "T2"
C2_test_x@meta.data$sample <- "C2"

In [ ]:
merged_clustered <- JoinLayers(merge(T1_test_x, y = list(C1_test_x, T2_test_x,C2_test_x), merge.data = TRUE), assay = "SPM")

In [ ]:
merged_clustered <- JoinLayers(merged_clustered, assay = "SPT")

Next we will check the clustering at the spot level. We will try two clustering resolutions, res = 0.5 and res = 1 and compare the results for <span style = "color: red;">**Treated**</span> sample 1

In [ ]:
#T1_test_x <- cluster_seurat_obj(T1_test_x, res = 0.5, pcas = 50, clust_name = "seurat_cluster_0.5")
#T1_test_x <- cluster_seurat_obj(T1_test_x, res = 1, pcas = 50, clust_name = "seurat_cluster_1")

In [ ]:
# options(repr.plot.width = 30, repr.plot.height = 8)

# ImageDimPlot(T1_test, group.by = "ssc", size = 2)+coord_flip()+scale_x_reverse()| 
# SpatialDimPlot(T1_test_x, group.by = "seurat_cluster_0.5")| 
# SpatialDimPlot(T1_test_x, group.by = "seurat_cluster_1") | 
# SpatialMZAnnotationPlot(T1_MALDIVIS_annot, metabolites  = c("Palbociclib"), plusminus = 0.05, slot = "counts", images = NULL)

We can see that the clustering using the metabolomic data with *Seurat's* 'FindClusters' function is quite similar to *Cardinal's* ssc clusters. Also it seems with higher resolution these clutsers can detect regions where our drug is over expressed

lets run it for the rest of the samples ...

In [ ]:
DefaultAssay(merged_clustered) <- "SPT"
Idents(merged_clustered) <- "sample"

options(repr.plot.width = 10, repr.plot.height = 6)

merged_clustered_filtered <- run_qc(merged_clustered, min_genes = 20, min_spots = 3)

In [ ]:
DefaultAssay(merged_clustered_filtered) <- "SPM"

In [ ]:
merged_clustered_filtered <- NormalizeSeuratData(merged_clustered_filtered, assay = "SPM")
#merged_clustered <- cluster_seurat_obj(merged_clustered, res = 0.5, pcas = 50, clust_name = "seurat_cluster_0.5_SPM")
#merged_clustered <- cluster_seurat_obj(merged_clustered, res = 1, pcas = 50, clust_name = "seurat_cluster_1_SPM")

In [ ]:
merged_clustered_filtered <- FindVariableFeatures(merged_clustered_filtered, verbose = FALSE)
merged_clustered_filtered <- ScaleData(merged_clustered_filtered, verbose = FALSE)
merged_clustered_filtered <- RunPCA(merged_clustered_filtered, npcs = 50, verbose = FALSE, reduction.name = "spm.pca")
merged_clustered_filtered <- FindNeighbors(merged_clustered_filtered, dims = 1:50, verbose = FALSE, reduction ="spm.pca")
merged_clustered_filtered <- RunUMAP(merged_clustered_filtered, dims = 1:50, verbose = FALSE, reduction.name = "spm.umap", reduction ="spm.pca")
merged_clustered_filtered <- FindClusters(merged_clustered_filtered, resolution = 0.5, verbose = FALSE, cluster.name = "seurat_cluster_0.5_SPM")

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 8)

SpatialDimPlot(merged_clustered_filtered, group.by = "seurat_cluster_0.5_SPM") #/
#SpatialDimPlot(merged_clustered, group.by = "seurat_cluster_1_SPM") #| 
#SpatialMZAnnotationPlot(merged_MALDIVIS, metabolites  = c("Palbociclib"), plusminus = 0.05, assay = "SPM", slot = "counts", images = NULL)

In [ ]:
DefaultAssay(merged_clustered_filtered) <- "SPT"
merged_clustered_filtered <- NormalizeData(merged_clustered_filtered)
merged_clustered_filtered <- FindVariableFeatures(merged_clustered_filtered)
merged_clustered_filtered <- ScaleData(merged_clustered_filtered)
merged_clustered_filtered <- RunPCA(merged_clustered_filtered, verbose = FALSE, reduction.name = "spt.pca")
merged_clustered_filtered <- FindNeighbors(merged_clustered_filtered, dims = 1:50, reduction ="spt.pca")

In [ ]:
merged_clustered_filtered <- RunUMAP(merged_clustered_filtered, dims = 1:30, reduction.name = "spt.umap", reduction = "spt.pca", verbose = FALSE)
merged_clustered_filtered <- FindClusters(merged_clustered_filtered, resolution = 0.1, verbose = FALSE )

In [ ]:
saveRDS(merged_clustered_filtered, paste0(OUT_DIR, "data_objects/merged_clusters_bin.RDS"))

In [ ]:
merged_clustered_filtered <- readRDS(paste0(OUT_DIR, "data_objects/merged_clusters_bin.RDS"))

In [ ]:
colour_palate_SPM <- list( "0" = '#FB9A99',
                      "1" = '#6a4c8a',
                      "2" = '#A6CEE3',
                      "3" = '#CAB2D6',
                      "4" = '#B2DF8A',
                      "5" = '#E31A1C',
                      "6" = '#33A02C')


colour_palate_SPT <- list( "0" = '#FDBF6F',
                          "1" = '#1F78B4',
                          "2" = '#B2DF8A',
                          "3" = '#6a4c8a',
                          "4" = '#FF7F00',
                          "5" = '#33A02C')
                        

# colour_palate_SPT_6clust <- list( "0" = '#FDBF6F',
#                           "1" = '#B2DF8A',
#                           "2" = '#6a4c8a',
#                           "3" = '#1F78B4',
#                           "4" = '#CAB2D6',
#                           "5" = '#FF7F00',
#                           "6" = '#33A02C')

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

clustering_plots <- (SpatialDimPlot(merged_clustered_filtered, group.by = "seurat_cluster_0.5_SPM", cols = colour_palate_SPM)/
SpatialDimPlot(merged_clustered_filtered, group.by = "seurat_clusters",  cols = colour_palate_SPT))

clustering_plots

#ggsave(clustering_plots, filename = paste0(OUT_DIR, "plots/merged_clustering.pdf"), height = 10, width = 20)

In [ ]:
df <- merged_clustered_filtered@meta.data %>%
  group_by(sample, seurat_cluster_0.5_SPM) %>%
  summarise(count = n())

In [ ]:
df <- df %>%
  group_by(sample) %>%
  mutate(proportion = round((count / sum(count)) * 100,2))

In [ ]:
cluster_counts <- df[df$seurat_cluster_0.5_SPM %in% c("0","2"),]

In [ ]:
options(repr.plot.width = 3, repr.plot.height = 5)

bar_plot <- ggplot(cluster_counts, aes(x = sample, y = proportion, fill = seurat_cluster_0.5_SPM)) +
  geom_bar(stat = "identity", position = "stack") +
  labs(x = "Sample", y = "Proportion %", fill = "Clusters") +
  theme_classic() +
  theme(axis.text.x = element_text(hjust = 1)) +
  scale_fill_manual(values =  list("0" = '#FB9A99',"2" = '#A6CEE3'))

bar_plot

ggsave(bar_plot, filename = paste0(OUT_DIR, "plots/cluster_proportions.pdf"), height = 5, width = 3)

In [ ]:
combined.T1 <- subset(merged_clustered_filtered, subset = sample == "T1")

In [ ]:
library(mclust)
library(gplots)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)


labels_true <- combined.T1@meta.data$seurat_cluster_0.5_SPM

# Extract cluster assignments for method 2
labels_pred <- combined.T1@meta.data$seurat_clusters

# Calculate adjusted Rand index
adjusted_rand_index <- mclust::adjustedRandIndex(labels_true, labels_pred)

#adjusted_rand_index



cluster_table <- table(labels_true, labels_pred)

# Calculate proportions of cluster type 2 within each cluster of cluster type 1
proportions <- prop.table(cluster_table, margin = 1)

# Create a heatmap to visualize the proportions
pdf(paste0(OUT_DIR,"plots/correlation_clust_T1.pdf"), height = 10, width = 10)
heatmap.2(as.matrix(proportions),
                               Rowv = TRUE, Colv = FALSE,
                               dendrogram = "row",

                               scale = "none", # you can change this to "row", "column", or "none"
                               col = colorRampPalette(c("white", "blue"))(100),
                               key = TRUE, keysize = 1.5, key.title = NA, # Add a scale bar
                               trace = "none")
dev.off()



## Add clusters back to orginal Object

In [ ]:
merged_MALDIVIS

In [ ]:
merged_MALDIVIS@meta.data$SPM_Clusters <- merged_clustered_filtered@meta.data$seurat_cluster_0.5_SPM
merged_MALDIVIS@meta.data$SPT_Clusters <- merged_clustered_filtered@meta.data$seurat_clusters

In [ ]:
merged_MALDIVIS@reductions <- merged_clustered_filtered@reductions

In [ ]:
DefaultAssay(merged_MALDIVIS) <- "SPM"
merged_MALDIVIS <- JoinLayers(merged_MALDIVIS, assay = "SPM")

In [ ]:
saveRDS(merged_MALDIVIS, file = paste0(OUT_DIR, "data_objects/merged_MALDIVIS_with_clusters_HMDB.RDS"))

In [ ]:
OUT_DIR  <- "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/DHB/"

merged_MALDIVIS <- readRDS(file = paste0(OUT_DIR, "data_objects/merged_MALDIVIS_with_clusters_HMDB.RDS"))

In [ ]:
FindCorrelatedFeaturesX <- function(data, mz = NULL, gene = NULL, SM.assay = "SPM", ST.assay = "SPT", SM.slot = "counts", ST.slot = "counts", nfeatures = 10){
  met_counts <- data[[SM.assay]][SM.slot]
  tran_counts <- data[[ST.assay]][ST.slot]

  gene_mappings <- data.frame(gene = rownames(tran_counts))

  rownames(tran_counts) <- unlist(lapply(1:length(rownames(tran_counts)), function (x) {
    paste0("mz-",(round(as.numeric(gsub("mz-", "", rownames(met_counts)[length(rownames(met_counts))],))) + 100), x)
  }))

  gene_mappings$mz <- rownames(tran_counts)
  gene_mappings$raw_mz <- gsub("mz-", "", gene_mappings$mz)

  data[["tmp"]] <- SeuratObject::CreateAssay5Object(counts = rbind(met_counts, tran_counts))

  if (is.null(mz) & ! is.null(gene)){
    idx <- which(gene_mappings$gene == gene)
    mz <- as.numeric(gene_mappings$raw_mz[idx])

  } else if (!is.null(mz) & is.null(gene)){
    mz <- mz
  } else {
    stop("Invalid input for 'mz = ' and 'gene = '... Only one inupt can be provided. Either mz or gene, alternative must be set to NULL! Please check documentation ...")
  }
  
  data_cardinal <- ConvertSeuratToCardinalX(data = data, assay = "tmp", slot = "counts")
   
  result <- suppressWarnings(Cardinal::colocalized(data_cardinal, mz=mz, n = length(rownames(met_counts)) + length(rownames(tran_counts))))
  result <- result[order(result$mz), ]
  result$features <- c(rownames(met_counts),gene_mappings$gene)
  result$modality <- c(rep("metabolite", length(rownames(met_counts))), c(rep("gene", length(gene_mappings$gene))))
  result <- result[c("features", colnames(result)[!colnames(result) %in% c("mz", "features")])]
  result <- result[order(-abs(result$correlation)), ]

  if (!is.null(nfeatures)){
    return(head(result, n = nfeatures))
  } else{
    return(result)
  }
}

In [ ]:
ConvertSeuratToCardinalX <- function(data, assay = "Spatial", slot = "counts", run_col = NULL, feature.metadata = FALSE, verbose = TRUE){

  mzs <- unlist(lapply(rownames(data[[assay]]@features), function(x) as.numeric(stringr::str_split(x, "mz-")[[1]][2])))
  coord <- SeuratObject::GetTissueCoordinates(data)

  if (!(is.null(run_col))){
    run <- data@meta.data[[run_col]]
  } else {
    run <- factor("run0")
  }

  pdata <- Cardinal::PositionDataFrame(run = run,
                             coord=coord[c("imagerow","imagecol")],
                             Seurat_metadata = data@meta.data[c(colnames(data@meta.data))]) #adds all other columns

  if (feature.metadata){
    fdata <- Cardinal::MassDataFrame(mz=mzs, feature_metadata = data[[assay]]@meta.data[c(colnames(data[[assay]]@meta.data))])
  } else {
    fdata <- Cardinal::MassDataFrame(mz=mzs)
  }

  mat <- data[[assay]][slot]
  rownames(mat) <- NULL
  colnames(mat) <- NULL

  


  cardinal.obj <- Cardinal::MSImagingExperiment(spectraData= matter::sparse_mat(Matrix::as.matrix(mat)),
                                      featureData=fdata,
                                      pixelData=pdata)


  return(cardinal.obj)
}

In [ ]:
test <- subset(merged_MALDIVIS, subset = sample == "T1")
test_cardinal <- ConvertSeuratToCardinalX(data = test, assay = "SPM", slot = "counts")

In [ ]:
#human_tumour <- subset(test, subset = percent.human > 50)
human_tumour <- subset(merged_MALDIVIS, subset = percent.human > 50)

In [ ]:
T1_sub <- subset(merged_MALDIVIS, subset = sample == "T1")
T2_sub <- subset(merged_MALDIVIS, subset = sample == "T2")

In [ ]:
T1_human_tumour <- subset(T1_sub, subset = percent.human > 50)
T2_human_tumour <- subset(T2_sub, subset = percent.human > 50)

In [ ]:
human_tumour_features <- FindCorrelatedFeaturesX(T1_human_tumour, mz = 448.2467, gene = NULL, SM.assay = "SPM", ST.assay = "SPT", SM.slot = "counts", ST.slot = "counts", nfeatures = 20)

In [ ]:
y <- FindCorrelatedFeaturesX(test, mz = 448.2467, gene = NULL, SM.assay = "SPM", ST.assay = "SPT", SM.slot = "counts", ST.slot = "counts", nfeatures = NULL)

In [ ]:
y[y$modality == "gene",]

In [ ]:
write.csv(human_tumour_features, "/scratch/user/uqacause/files/project_data_objects/MB/drug_correlation.csv")

In [ ]:
human_tumour_features[human_tumour_features$modality == "gene",]

In [ ]:
df <- human_tumour_features[human_tumour_features$modality == "gene",][1:10,] # subsets to only include results from cluster 5
df$features <- gsub("hg38-","",df$features) # simplifies the m/z names for the plot
df$rank <- c(1:10)
## Plots the top 10 results
options(repr.plot.width = 5, repr.plot.height = 5)
p <- ggplot(df, aes(x = -rank, y = -correlation)) + geom_bar(stat = "identity") + geom_text(aes(label = features), vjust = -0.5, size = 3, color = "black") + coord_flip() + theme_classic()
p

In [ ]:
ggsave(p, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/correlation_plot.pdf", height = 5, width = 5)

In [ ]:
library(viridis)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)

p <- SpatialMZPlot(merged_MALDIVIS, mzs = c(448.2467), assay = "SPM", pt.size.factor = 2.2) #& scale_fill_gradientn(colors = inferno(100))

ggsave(p, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/Drug_expression.pdf", height = 5, width = 20)

p

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)

p <- SpatialFeaturePlot(merged_MALDIVIS, features = c("hg38-TUBA1B"), pt.size.factor = 2.2) #& scale_fill_gradientn(colors = inferno(100))

ggsave(p, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/TUBA1B_expression.pdf", height = 5, width = 20)

p

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)

p <- SpatialFeaturePlot(merged_MALDIVIS, features = c("hg38-HMGN2"), pt.size.factor = 2.2) #& scale_fill_gradientn(colors = inferno(100))

ggsave(p, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/HMGN2_expression.pdf", height = 5, width = 20)

p

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 5)

p <- SpatialFeaturePlot(merged_MALDIVIS, features = c("hg38-TUBA1B"), pt.size.factor = 2.2) #& scale_fill_gradientn(colors = verbose(100))

ggsave(p, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/TUBA1B_expression.pdf", height = 5, width = 20)

p

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

clustering_plots <- (SpatialDimPlot(merged_MALDIVIS, group.by = "SPM_Clusters", cols = colour_palate_SPM)/
SpatialDimPlot(merged_MALDIVIS, group.by = "SPT_Clusters",  cols = colour_palate_SPT))

clustering_plots

#ggsave(clustering_plots, filename = paste0(OUT_DIR, "plots/merged_clustering.pdf"), height = 10, width = 20)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

(DimPlot(merged_MALDIVIS, group.by = "SPM_Clusters", reduction = "spm.umap", cols = colour_palate_SPM)| 
 DimPlot(merged_MALDIVIS, reduction = "spt.umap", group.by = "SPT_Clusters", cols = colour_palate_SPT))/
(DimPlot(merged_MALDIVIS, group.by = "sample", reduction = "spm.umap",cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)]))| 
 DimPlot(merged_MALDIVIS, reduction = "spt.umap", group.by = "sample", cols = c(RColorBrewer::brewer.pal(name = "Paired", n = 7)[c(1:2, 5:6)])))

In [ ]:
## library(plotly)
library(tidyverse)
library(plotly)
suppressMessages({

logged_df <- merged_MALDIVIS@meta.data %>%
  mutate(
    nCount_SPT = log(nCount_SPT),
    nFeature_SPT = log(nFeature_SPT),
    nCount_SPM = log(nCount_SPM),
    nFeature_SPM = log(nFeature_SPM)
  )



df <- logged_df%>%
  select(sample, nCount_SPT, nFeature_SPT, nCount_SPM, nFeature_SPM) %>%
  gather(key = "label", value = "values", -sample)


fig <- df %>%
  plot_ly(type = 'violin', alpha = 0.3) 
fig <- fig %>%
  add_trace(
    x = ~sample[df$label == 'nCount_SPT'],
    y = ~values[df$label == 'nCount_SPT'],
    legendgroup = 'nCount_SPT',
    scalegroup = 'nCount_SPT',
    name = 'nCount_SPT',
    side = 'negative',
      width = 1,
    box = list(
      visible = T, 
          opacity = 1
    ),
    meanline = list(
      visible = F
    ),
       marker = list(
     opacity = 0),
    color = I('#B2DF8A')
  ) 
fig <- fig %>%
  add_trace(
    x = ~sample[df$label == 'nCount_SPM'],
    y = ~values[df$label == 'nCount_SPM'],
    legendgroup = 'nCount_SPM',
    scalegroup = 'nCount_SPM',
    name = 'nCount_SPM',
    side = 'positive',
    box = list(
      visible = T,
         opacity = 1
    ),
    meanline = list(
      visible = F
    ),
     marker = list(
     opacity = 0),
    color = I('#33A02C')
  ) 

fig <- fig %>%
  layout(
    xaxis = list(
      title = "",
        zeroline = T, 
        fixedrange=TRUE, 
        range = list(-1, 4)
    ),
    yaxis = list(
      title = "",
      zeroline = T, 
        showgrid = F, 
        fixedrange=TRUE,
        range = list(0, 15)
    ),
    violingap = 0,
    violingroupgap = 0,
    violinmode = 'overlay',
      width = 800
      
  )

fig

      })

In [ ]:
export(fig, file = paste0(OUT_DIR, "plots/violin_counts.pdf"))

In [ ]:
## library(plotly)
library(tidyverse)

suppressMessages({

logged_df <- merged_MALDIVIS@meta.data %>%
  mutate(
    nCount_SPT = log(nCount_SPT),
    nFeature_SPT = log(nFeature_SPT),
    nCount_SPM = log(nCount_SPM),
    nFeature_SPM = log(nFeature_SPM)
  )



df <- logged_df%>%
  select(sample, nCount_SPT, nFeature_SPT, nCount_SPM, nFeature_SPM) %>%
  gather(key = "label", value = "values", -sample)


fig <- df %>%
  plot_ly(type = 'violin', alpha = 0.3) 
fig <- fig %>%
  add_trace(
    x = ~sample[df$label == 'nFeature_SPT'],
    y = ~values[df$label == 'nFeature_SPT'],
    legendgroup = 'nFeature_SPT',
    scalegroup = 'nFeature_SPT',
    name = 'nFeature_SPT',
    side = 'negative',
      width = 1,
    box = list(
      visible = T, 
          opacity = 1
    ),
    meanline = list(
      visible = F
    ),
       marker = list(
     opacity = 0),
    color = I('#CAB2D6')
  ) 
fig <- fig %>%
  add_trace(
    x = ~sample[df$label == 'nFeature_SPM'],
    y = ~values[df$label == 'nFeature_SPM'],
    legendgroup = 'nFeature_SPM',
    scalegroup = 'nFeature_SPM',
    name = 'nFeature_SPM',
    side = 'positive',
    box = list(
      visible = T,
         opacity = 1
    ),
    meanline = list(
      visible = F
    ),
     marker = list(
     opacity = 0),
    color = I('#6A3D9A')
  ) 

fig <- fig %>%
  layout(
    xaxis = list(
      title = "",
        zeroline = T, 
        fixedrange=TRUE, 
        range = list(-1, 4)
    ),
    yaxis = list(
      title = "",
      zeroline = T, 
        showgrid = F, 
        fixedrange=TRUE, 
        range = list(0, 15)
    ),
    violingap = 0,
    violingroupgap = 0,
    violinmode = 'overlay',
      width = 800
      
  )

fig

      })

In [ ]:
export(fig, file = paste0(OUT_DIR, "plots/violin_features.pdf"))

#### DE for SM Clusters (with mouse region)

In [ ]:
Idents(merged_MALDIVIS) <- "SPM_Clusters"
suppressWarnings({
    Idents(merged_MALDIVIS) <- "SPM_Clusters"
    tumour <- subset(merged_MALDIVIS, subset = SPM_Clusters %in% c("2","0", "5"))
})
options(repr.plot.width = 20, repr.plot.height = 6)
SpatialDimPlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), group.by = "SPM_Clusters", cols = colour_palate_SPM, pt.size.factor = 2.1) & NoLegend()
SpatialDimPlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), group.by = "SPT_Clusters", cols = colour_palate_SPM, pt.size.factor = 2.1) & NoLegend()

In [ ]:
gene_markers_X <- FindAllDEGs(tumour, assay = "SPT", slot = "data", ident = "SPM_Clusters")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-MT2A"), pt.size.factor = 2.1)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-HMGA1"), pt.size.factor = 2.1)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 20)
x <- DEGsHeatmap(gene_markers_X, only.pos = F, n = 20)
x

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-MEG3"), pt.size.factor = 2.1)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-AQP5"), pt.size.factor = 2.1)

In [ ]:
gene_edgr <- gene_markers_X

In [ ]:
gene_edgr

In [ ]:
gene_edgr$DEGs <- gene_edgr$DEGs[grep("hg38-", gene_edgr$DEGs$gene),]
#gene_edgr$DEGs$gene <- gsub("hg38-", "", gene_markers_X$DEGs$gene)

options(repr.plot.width = 20, repr.plot.height = 20)
x <- DEGsHeatmap(gene_edgr, only.pos = F, n = 20)
x

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-PLCG2"), pt.size.factor = 2.1)
SpatialFeaturePlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), features = c("hg38-BST2"), pt.size.factor = 2.1)

In [ ]:
#library(Seurat)
#library(dplyr)
#library(data.table)
#library(tidyr)
#library(stringr)

#### SpaMTP m/z Annotation Functions #####################################################################################################################################################################################

#' Subset a Seurat Spatial Metabolomic object by list of m/z's
#'
#' @param data A Seurat Spatial Metabolomic Object for subsetting.
#' @param features A list of character strings defining the features/mz values to subset against.
#' @param assay A character string identifying the Seurat assay which contains the count data being subset.
#'
#' @returns A subset Seurat object containing only m/z values that were specified
#' @export
#'
#' @examples
#' # SubsetMZFeatures(SeuratObj, c("mz-160","mz-170","mz-180"))
SubsetMZFeatures <- function(data, features, assay = "Spatial"){
  feature.metadata <- data[[assay]]@meta.data
  sub.data <- subset(data, features = features)

  sub.data[[assay]]@meta.data <- feature.metadata[feature.metadata[["mz_names"]] %in% features,]
  return(sub.data)
}



#' Filters the annotation list to only include the first n number of annotations per m/z
#'
#' @param annotation_column Vector of the meta.data column containing the m/z annotations.
#' @param n Numeric value defining the number of annotations to keep (default = 3).
#'
#' @return Vector containing the first n number of annotations
#'
#' @examples
#' # labels_to_show(`SeuratObject[["Spatial"]]@meta.data$annotations`, n = 3)
labels_to_show <- function(annotation_column, n = 3) {
  new_column <- sapply(strsplit(annotation_column, "; "), function(x) {
    # Filter out NA values and select the first three entries
    non_na_values <- x[!is.na(x)]
    paste(non_na_values[1:min(n, length(non_na_values))], collapse = "; ")
  })
  return(new_column)
}


AnnotateSMX <- function(data, db, assay = "Spatial", raw.mz.column = "raw_mz", ppm_error = NULL, adducts = NULL, polarity = "positive", tof_resolution = 30000, filepath = NULL, return.only.annotated = TRUE, save.intermediate = TRUE, verbose = TRUE){

  if (is.null(data@assays[[assay]])) {
    stop(paste0("No assay '",assay,"'exists in SpaMTP object! Please check assay name input ..."))
  }

  ## Extracting m/z values from SpaMTP@assay@meta.data
  mz_df <- data[[assay]][[raw.mz.column]]
  mz_df$mz <- mz_df[[raw.mz.column]]
  mz_df$row_id <- seq(1, length(mz_df[[raw.mz.column]]))
  mz_df <- mz_df[c("row_id", "mz")]

  db_3 <- annotateTable(mz_df= mz_df, db = db, ppm_error = ppm_error, adducts = adducts, polarity = polarity,tof_resolution = tof_resolution,verbose = verbose)
  return(db_3)
}

#' ### HelperFunction
annotateTable <- function(mz_df, db, ppm_error = NULL, adducts = NULL, polarity = "positive", tof_resolution = 30000, verbose = TRUE){


  # Uses:
  # db: db that you want to search against

  # test_add_pos: which adducts you want to search for
  # Note; "M+NH4","M+Na","M+CH3OH+H","M+K" etc. Look at the formula filter func to get the rest of the possible adducts.

  # ppm_error: the ppm error/threshold for searching

  # Three main steps relates to the three main functions
  # Steps 1) & 2) are aimed at condensing the databases by applying 1) a filter to only consider the adducts that the user specifies. 2) Filtering the molecular formulas to contain only elements that the user specifies. # Step 3) This last function then does the database matching and searching.
  # 1) Filter DB by adduct.
  verbose_message(message_text = paste0("Filtering provided database by ", paste0(adducts, collapse = ", "), " adduct/s"), verbose = verbose)

  if (polarity == "positive") {
    test_add_pos = adduct_file$adduct_name[which(adduct_file$charge > 0)]
    test_add_pos <- gsub(" ", "", test_add_pos)
    test_add_pos = test_add_pos[which(test_add_pos %in% (adducts %||% test_add_pos))]
    # Using Chris' pipeline for annotation
    # 1) Filter DB by adduct.
    db_1 <- db_adduct_filter(db,
                            test_add_pos,
                            polarity = "pos",
                            verbose = verbose)
  } else if (polarity == "negative") {
    test_add_neg = adduct_file$adduct_name[which(adduct_file$charge < 0)]
    test_add_neg <- gsub(" ", "", test_add_neg)
    test_add_neg = test_add_neg[which(test_add_neg %in% (adducts %||% test_add_neg))]
    # Using Chris' pipeline for annotation
    # 1) Filter DB by adduct.
    db_1 <- db_adduct_filter(db,
                            test_add_neg,
                            polarity = "neg",
                            verbose = verbose)
  } else if (polarity == "neutral") {
    # Using Chris' pipeline for annotation
    # 1) Filter DB by adduct.
    db_1 <- db %>% mutate("M" = `M-H ` + 1.007276)
  } else{
    stop("Please enter correct polarity from: 'positive', 'negative', 'neutral'")
  }


  # 2) only select natural elements
  db_2 <- formula_filter(db_1)

  # 3) search db against mz df return results
  verbose_message(message_text = "Searching database against input m/z's to return annotaiton results", verbose = verbose)

  if (is.null(ppm_error) && is.null(tof_resolution)){
    stop("ppm_error and tof_resolution cannot both = NULL! Please set one of these variables to calculate the ppm threshold for annotations ... ")
  }

  ppm_error = ppm_error %||% (1e6 / tof_resolution / sqrt(2 * log(2)))

  db_3 <- proc_db(mz_df, db_2, ppm_error)

  ## Add in database labels
  db_3 = db_3 %>% dplyr::mutate(entry = stringr::str_split(Isomers, pattern = "; "))
  input_id = lapply(db_3$entry, function(x) {
    x = unlist(x)
    index_hmdb = which(grepl(x, pattern = "HMDB"))
    x[index_hmdb] = paste0("hmdb:", x[index_hmdb])
    index_chebi = which(grepl(x, pattern = "CHEBI"))
    x[index_chebi] = tolower(x[index_chebi])
    index_lipidm = which(grepl(x, pattern = "^LM"))
    x[index_lipidm] = paste0("LIPIDMAPS:", x[index_lipidm])
    return(x)
  })
  db_3$entry <- NULL

  db_3$Isomers_IDs <- input_id

  db_3 <- db_3 %>%
    dplyr::mutate(Isomers_IDs = sapply(Isomers_IDs, function(x) stringr::str_c(x, collapse = "; ")))

  return(db_3)
}



#' Used to subset dataset to only include annotations that have n number of entries
#'    - (i.e. some peaks can have multiple annotations. Peaks which have above n number of annotations assigned will be removed from Seurat Object)
#'
#' @param obj Seurat object needing annotation refinement. This object must have annotations present in 'obj`[[assay]]@meta.data`'
#' @param assay Character string defining the Seurat object assay where the annotation data is stored (default = "Spatial").
#' @param n Integer defining the number of entries an annotation can have assigned. Any higher counts will be removed (default = 1).
#'
#' @return Refined Seurat object that only contains annotated mz values that have n number of annotations assigned (per mz value)
#' @export
#'
#' @examples
#' # HMDB_db <- load("data/HMDB_1_names.rds")
#' # AnnotatedSeuratObj <- AnnotateSeuratMALDI(SeuratObj, HMDB_db)
#'
#' # getRefinedAnnotations(AnnotatedSeuratObj, n = 2)
getRefinedAnnotations <- function(obj, assay = "Spatial",n = 1){
  n <- n-1
  subset.metadata <- obj[[assay]]@meta.data %>% dplyr::filter(stringr::str_count(all_IsomerNames, ";") %in% c(0:n))
  subset.obj <- SubsetMZFeatures(data = obj, features = subset.metadata$mz_names,assay = assay)
  return(subset.obj)

}

########################################################################################################################################################################################################################


#### SpaMTP Find m/z Annotation Functions #####################################################################################################################################################################################

#' Adds in backslashes required to take into account special using grepl such as brackets and +
#'
#' @param input_string Character string requiring additional backslashes
#'
#' @return Character string containing double backslashes around special features
add_backslashes_to_specialfeatures <- function(input_string) {
  # Use gsub to replace ( with \\( and ) with \\)
  result_string <- gsub("\\(", "\\\\(", input_string)
  result_string <- gsub("\\)", "\\\\)", result_string)
  result_string <- gsub("\\[", "\\\\[", result_string)
  result_string <- gsub("\\]", "\\\\]", result_string)
  result_string <- gsub("\\+", "\\\\+", result_string)
  result_string <- gsub(" ", "[ ]", result_string)

  return(result_string)
}


#' Searches through annotated m/z values to return all which contain the metabolite search term provided
#'
#' @param data Seurat Spatial Metabolomic Object containing annotated m/z values.
#' @param metabolite Character string of metabolite search term.
#' @param assay Character string defining the Seurat assay that contains the annotated metadata corresponding to the m/z values (default = "Spatial").
#' @param search.exact Boolean value defining if to only return m/z values which contain the exact match to the metabolite search term (default = FALSE).
#' @param column.name Character string defining the column name where the annotations are stored in the slot meta.data (default = "all_IsomerNames").
#'
#' @return A Data.Frame containing the peak metadata corresponding to the metabolite search term provided
#' @export
#'
#' @examples
#' # SearchAnnotations(SeuratObj, "Glucose", search.exact = TRUE)
SearchAnnotations <- function (data, metabolite, assay = "Spatial",search.exact = FALSE, column.name = "all_IsomerNames"){

  ## Takes into account '( )' in the string name
  search_term <- add_backslashes_to_specialfeatures(metabolite)

  indexs <- which(grepl(search_term, data[[assay]]@meta.data[column.name][[1]], ignore.case = TRUE))

  if (search.exact){
    df <- data[[assay]]@meta.data[indexs,]
    indexs <- c()
    for (row in rownames(df)){
      annotation_list <- df[row,][[column.name]]
      split_list <- unlist(strsplit(annotation_list, "; "))

      if (metabolite %in% split_list){
        indexs <- c(indexs, row)
      }

    }
  }

  return(data[[assay]]@meta.data[indexs,])
}



#' Finds if any metabolite is duplicated across multiple m/z values.
#'
#' @param data Seurat Spatial Metabolomic Object containing annotated m/z values.
#' @param assay Character string defining the Seurat assay that contains the annotated metadata corresponding to the m/z values (default = "Spatial").
#'
#' @return Vector of character strings describing metabolites that are assigned to multiple m/z values
#' @export
#'
#' @examples
#' # FindDuplicateAnnotations(SeuratObj)
FindDuplicateAnnotations <- function (data, assay = "Spatial"){
  all_annotations <- data[[assay]]@meta.data$all_IsomerNames
  all_terms <- unlist(strsplit(all_annotations, ";"))
  all_terms <- trimws(all_terms)
  terms_counts <- table(all_terms)
  return(names(terms_counts[terms_counts > 1]))
}


########################################################################################################################################################################################################################

#!! ALL CODE BELOW WAS WRITEN BY Christopher Fitzgerald github https://github.com/ChemCharles !!#


#' Checks if the complete adduct is in the data base, else returns a truncated adduct
#'
#' @param adduct Character string defining the adduct to be checked.
#' @param db DataFrame of the current reference database.
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @return Character string of the complete or truncated adduct
#'
#' @examples
#' # HMDB_db <- load("data/HMDB_1_names.rds")
#' # check_and_truncate_adduct_vector(c("M+H"), HMDB_db)
check_and_truncate_adduct_vector <- function(adduct, db, verbose = TRUE) {
  element_exists <- adduct %in% colnames(db)
  missing_elements <- adduct[!element_exists]
  if (length(missing_elements) > 0) {
    for (missing_element in missing_elements) {
      verbose_message(message_text =  paste0("Adduct ",
                      missing_element,
                      " is not in the DB, it has been removed from the search."), verbose = verbose)

    }
    truncated_adduct <- adduct[element_exists]
    return(truncated_adduct)
  } else {
    return(adduct)
  }
}




#' Filters the provided metabolomic database by polarity and adducts
#'
#' @param adduct Character string defining the adduct to be checked.
#' @param db DataFrame of the current reference database.
#' @param polarity Character string defining the polarity of the adducts (default = "neg").
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @return A filtered reference metabolomic database DataFrame
#'
#' @examples
#' # HMDB_db <- load("data/HMDB_1_names.rds")
#' # db_adduct_filter(HMDB_db,c("M+H"), polarity = "pos")
db_adduct_filter <- function(db, adduct, polarity = "neg", verbose = TRUE) {
  # only include adducts from either neg or pos polarity
  if (polarity == "neg") {
    neg_adducts_1 <-
      c(
        "M-H2O-H",
        "M-H",
        "M+Na-2H",
        "M+Cl",
        "M+K-2H",
        "M+FA-H",
        "M+Hac-H",
        "M+Br",
        "M+TFA-H",
        "2M-H",
        "2M+FA-H",
        "2M+Hac-H",
        "3M-H"
      )
    pol <- neg_adducts_1
  }
  else if (polarity == "pos") {
    pos_adducts_1 <-
      c(
        "M+H",
        "M+NH4",
        "M+Na",
        "M+CH3OH+H",
        "M+K",
        "M+ACN+H",
        "M+2Na-H",
        "M+IsoProp+H",
        "M+ACN+Na",
        "M+2K+H",
        "M+DMSO+H",
        "M+2ACN+H",
        "M+IsoProp+Na+H",
        "2M+H",
        "2M+NH4",
        "2M+Na",
        "2M+K",
        "2M+ACN+H",
        "2M+ACN+Na"
      )
    pol <- pos_adducts_1
  } else{
    stop("Invalid polarity. Choose 'neg' or 'pos'.")
  }

  # get rid of spaces in the adduct names
  # in col names
  db <- db %>%
    dplyr::rename_all( ~ gsub(" ", "", .))

  # Filter the db by polarity
  db <- db %>%
    dplyr::select(formula, exactmass, isomers, isomers_inchikey, isomers_names, pol)

  # in adduct entry
  adduct <- gsub(" ", "", adduct)

  adduct <- check_and_truncate_adduct_vector(adduct, db, verbose = verbose)

  db_filtered <- db %>%
    dplyr::select(formula,
           exactmass,
           isomers,
           isomers_inchikey,
           isomers_names,
           tidyr::any_of(adduct)) %>%
    as.data.frame()
  return(db_filtered)
}




#' Checks if a formula contains only the allowed elements
#'
#' @param formula Character string defining t
#' @param allowed_elements Vector of character strings defining allowed elements
#'
#'
#' @examples
#' ### Helper function ###
is_formula_valid <- function(formula,allowed_elements) {
  elements <-
    stringr::str_extract_all(formula, "[A-Z][a-z]*")[[1]] # defines an element as a Uppercase immediately followed by none or more lowercase letters.

  all(elements %in% allowed_elements) # Then checks if they are in the vector of allowed elements.
}




#' Filters reference Database to only select natural elements
#'
#' @param df DataFrame of the reference database.
#' @param elements Vector of character strings of elements to include (default = c("H", "C", "N", "O", "S", "Cl", "Br", "F", "Na", "P", "I", "Si")).
#'
#' @return A refined DataFrame which only includes annotations containing the specified elements
#'
#' @examples
#' ## Get filtered DB by adduct
#' # db_1 <- db_adduct_filter(db, test_add_pos, polarity = "pos")
#'
#' ## Refine to only include natural elements
#' # db_2 <- formula_filter(db_1)
formula_filter <- function(df, elements = NULL) {
  if (is.null(elements)) {
    elements <- c("H", "C", "N", "O", "S", "Cl", "Br",
                  "F", "Na", "P", "I","Si")
  }

  # Elements to allow
  allowed_elements <- elements

  # Filter rows based on allowed elements
  filtered_df <- df %>%
    dplyr::filter(sapply(formula, allowed_elements = allowed_elements,is_formula_valid))

  return(filtered_df)

}




#' Calculates the mz range of the observed_df
#'
#' @param input_df DataFrame of the observed dataframe being annotated
#'
#' @return A list containing the lower and upper mz range for the provided sample
#'
#' @examples
#' # mz_df <- SeuratObject`[["Spatial"]][["mz"]]`
#' # mz_df$row_id <- seq(1, length(mz_df$mz))
#'
#' # mass_range <- calculate_bounds(mz_df)
#' # lower_bound <- mass_range$lower_bound
#' # upper_bound <- mass_range$upper_bound
calculate_bounds <- function(input_df) {

  lower_bound <- min(input_df$mz, na.rm = TRUE)
  upper_bound <- max(input_df$mz, na.rm = TRUE)

  bounds <-
    list(lower_bound = lower_bound, upper_bound = upper_bound)

  return(bounds)
}





#' Calculates the ppm error as a valve
#'
#' @param observed_mz Numeric value defining the observed mz value.
#' @param reference_mz Numeric value defining the reference mz value.
#' @param ppm Numeric value defining the maximum acceptable ppm_error/threshold for searching.
#'
#' @return Numeric value defining the ppm_error between the observed and reference mz value
#'
#' @examples
#' ### Helper Function ###
ppm_error <- function(observed_mz, reference_mz, ppm) {
  abs_diff_ppm <-
    abs(observed_mz - reference_mz) / abs(reference_mz) * 1e6
  if (abs_diff_ppm <= ppm) {
    return(abs_diff_ppm)
  }
  else{
    return("Out")
  }
}




#' Calculates the ppm range and check if mz values are within the range
#'    -  Returns TRUE if match is found and false if no match.
#'
#' @param observed_mz Numeric value defining the observed mz value.
#' @param reference_mz Numeric value defining the reference mz value.
#' @param ppm Numeric value defining the maximum acceptable ppm_error/threshold for searching.
#'
#' @return Boolean value indicating if a match is found (TRUE) or not (FALSE)
#'
#' @examples
#' ### Helper Function ###
ppm_range_match <- function(observed_mz, reference_mz, ppm) {
  abs_diff_ppm <-
    abs(observed_mz - reference_mz) / abs(reference_mz) * 1e6
  abs_diff_ppm <= ppm
}






#' Searches observed mz values against the data base list and returns matching annotations
#'
#' @param observed_df DataFrame containing the observed mz values
#' @param reference_df DataFrame contating the reference mz values and relative annotations.
#' @param ppm_threshold Numeric value defining the maximum acceptable ppm_error/threshold allowed between observed and reference mz values
#'
#' @return A DataFrame containing matched mz values between the observed and reference dataframes
#'
#' @examples
#' # HMDB_db <- load("data/HMDB_1_names.rds")
#' # mz_df <- SeuratObject[["Spatial"]][["mz"]]
#' # mz_df$row_id <- seq(1, length(mz_df$mz))
#'
#' ## 1) Filter DB by adduct.
#' # db_1 <- db_adduct_filter(HMDB_db, c("M+H"), polarity = "pos")
#'
#' ## 2) only select natural elements
#' # db_2 <- formula_filter(db_1)
#'
#' ## 3) search db against mz df return results
#' # db_3 <- proc_db(mz_df, db_2, ppm_threshold = 5)
proc_db <- function(observed_df,
                    reference_df,
                    ppm_threshold = 10) {
  # create an empty list to store matched results.
  result_list <- list()

  # Check if observed_df has only one row
  if (nrow(observed_df) == 1) {
    # Create a dummy entry with all values set to 0
    dummy_row <-
      data.frame(row_id = 0, mz = 0)  # Modify this line based on your column names

    # Combine the dummy entry with the original observed_df
    observed_df <- rbind(observed_df, dummy_row)
  } else if (nrow(observed_df) == 0) {
    # Handle the case when observed_df is empty
    verbose_message(message_text =  "Warning: No entries detected in input mz list.\n Please check format of input list,\n it must contain row_id and mz as the headers", verbose = verbose)

    return(NULL)
  }

  # extract out bounds
  lower_bound <- calculate_bounds(observed_df)$lower_bound
  upper_bound <- calculate_bounds(observed_df)$upper_bound

  #  For loop to go through each column of the reference_df that is provided.
  #  probably a good idea to filter reference_df to only adducts that you want before putting it into this function.
  # the -c(1:4) essentially makes it loop over the numeric portions of the DBs.

  for (col_name in names(reference_df)[-c(1:5)]) {
    result_col <- list()
    for (i in seq_len(nrow(observed_df))) {
      # Check if the reference mz is within the range
      within_range <-
        reference_df[[col_name]] >= lower_bound &
        reference_df[[col_name]] <= upper_bound

      # Condition 1: Only proceed if there are values within the range
      if (!any(within_range)) {
        next
      }

      # check for matches
      matches <- ppm_range_match(
        observed_mz = observed_df$mz[i],
        reference_mz = reference_df[[col_name]],
        ppm = ppm_threshold
      )

      # Condition 2: skip to the next iteration if no match
      if (!any(matches)) {
        next
      }

      # Condition 3: index the matches in the reference df
      # which keeps things that are TRUE.
      # This allows referencing to be quicker below.
      matching_indices <- which(matches)
      # print(matching_indices)

      # Calculate ppm error for each match.
      error <- mapply(
        ppm_error,
        observed_mz = observed_df$mz[i],
        reference_mz = reference_df[[col_name]][matching_indices],
        ppm = ppm_threshold
      )

      # Extract out relevant info for each match.
      filtered_matches <- data.frame(
        ID = observed_df$row_id[i],
        Match = matches[matching_indices],
        observed_mz = observed_df$mz[i],
        Reference_mz = reference_df[[col_name]][matching_indices],
        Error = error,
        Adduct = col_name,
        # DB_ID = reference_df$compound_id[matching_indices],
        Formula = reference_df$formula[matching_indices],
        Exactmass = reference_df$exactmass[matching_indices],
        Isomers = reference_df$isomers[matching_indices],
        InchiKeys = reference_df$isomers_inchikey[matching_indices],
        IsomerNames = reference_df$isomers_names[matching_indices]
      )
      # store each results df for each mz value in observed_df
      result_col[[i]] <- filtered_matches
    }
    # store results df in a list for each mz value for each adduct
    result_list[[col_name]] <- result_col
  }

  # Combine the individual matches per adduct df from the list into a dataframe
  combined_results <-
    do.call(rbind, lapply(result_list, function(result_col)
      do.call(rbind, result_col)))
}



########################################################################################################################################################################################################################

#' Adds custom metabolite annotations to respective m/z values (ideal for specific matrices such as FMP10)
#'
#' @param data SpaMTP Seurat object containing m/z intensity values.
#' @param annotations data.frame containing two columns named 'annotation' and 'mass'. These columns should contain the custom metabolite annotation and the relative m/z mass respectively.
#' @param assay Character string defining the Seurat object assay to store the respective annotations in the feature meta.data dataframe (default = "Spatial").
#' @param return.only.annotated Boolean defining whether to return a SpaMTP Seurat object containing only successfully annotated m/z values (default = FALSE).
#' @param mass.threshold Numeric value defining the acceptable threshold (plus-minus) between the custom annotations and the actual m/z values contained within the SpaMTP object (default = 0.05).
#' @param annotation.column Character string defining the feature meta.data column name that will contain the assigned annotations (default = "all_IsomerNames").
#'
#' @return SpaMTP Seurat object containing the custom annotations stored in the feature metadata dataframe.
#' @export
#'
#' @examples
#' # annotated_data <- AddCustomMZAnnotations(SpaMTP.obj, annotation.df)
AddCustomMZAnnotations <- function(data, annotations, assay = "Spatial", return.only.annotated = FALSE, mass.threshold = 0.05, annotation.column = "all_IsomerNames"){

  if (!all(c("annotation", "mass") %in% colnames(annotations))) {
    stop("Error: The annotation columns provided does not match the required format. Must bet 'annotation' and 'mass'")
  }

  true_mzs <- c()
  for (mass in annotations$mass){
    true_mz <- FindNearestMZ(data, mass)
    true_mzs <- c(true_mzs,true_mz)
  }

  annotations$true_mzs_name <- true_mzs
  annotations$true_mzs <- gsub("mz-", "", annotations$true_mzs_name)
  annotations$ppm_diff <- abs(annotations$mass - as.numeric(annotations$true_mzs))

  if (!is.null(mass.threshold)){
    annotations <- annotations %>% filter(ppm_diff < mass.threshold)
  }

  if (return.only.annotated) {

    data_sub <- subsetMZFeatures(data,features = annotations$true_mzs_name, assay = assay)
    data_sub[[assay]]@meta.data["all_IsomerNames"] <- annotations$annotation

  } else {
    metabolites <- lapply(rownames(data), function(x) {

      if (x %in% annotations$true_mzs_name) {
        annotations[annotations$true_mzs_name == x,][["annotation"]]
      } else {
        x
      }
    })

    data_sub <-  data
    data_sub[[assay]]@meta.data["all_IsomerNames"] <- unlist(metabolites)
  }

  return(data_sub)
}


In [ ]:
ROI_pathays <- AnnotateSMX(data = merged_MALDIVIS, db = rbind(Chebi_db,Lipidmaps_db,HMDB_db), ppm_error = 10, polarity = "positive", adducts = c("M+H"), assay = "SPM")

In [ ]:
merged_MALDIVIS@tools$db_3 <- ROI_pathays

In [ ]:
all_DE <- FindAllDEGs(merged_MALDIVIS, assay = "SPM", slot = "data", ident = "SPM_Clusters")

In [ ]:
all_DEGs <- FindAllDEGs(merged_MALDIVIS, assay = "SPT", slot = "data", ident = "SPM_Clusters")

In [ ]:
all_DEGs$DEGs <- all_DEGs$DEGs[grep("hg38-", all_DEGs$DEGs$gene),]
all_DEGs$DEGs$gene<- gsub("hg38-", "", all_DEGs$DEGs$gene)

In [ ]:
DE_pathways <- FindRegionalPathways(SpaMTP = merged_MALDIVIS,
                                ident = "SPM_Clusters",
                                DE.list = list(all_DE$DEGs, all_DEGs$DEGs),
                                analyte_types = c("metabolites", "genes"),
                                SM_assay = "SPM",
                                ST_assay = "SPT")

In [ ]:
PlotRegionalPathways(regpathway = DE_pathways, num_display = 4)

In [ ]:
DE_pathways$Cluster_id

In [ ]:
SpatialDimPlot(merged_MALDIVIS, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), group.by = "SPM_Clusters", cols = colour_palate_SPM, pt.size.factor = 2.1)

#### DE for SM Clusters

In [ ]:
Idents(merged_MALDIVIS) <- "SPM_Clusters"

In [ ]:
suppressWarnings({
    Idents(merged_MALDIVIS) <- "SPM_Clusters"
    tumour <- subset(merged_MALDIVIS, subset = SPM_Clusters %in% c("2","0"))
})


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 6)
SpatialDimPlot(tumour, images = c("slice1", "slice1.3", "slice1.2", "slice1.4"), group.by = "SPM_Clusters", cols = colour_palate_SPM, pt.size.factor = 2.1) & NoLegend()

In [ ]:
DefaultAssay(tumour) <- "SPT"
Idents(tumour) <- "SPM_Clusters"

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(stringr)
library(SingleCellExperiment)
library(scater)
library(edgeR)
library(pheatmap)
library(limma)
library(utils)
library(stats)
library(grDevices)

#### SpaMTP Differential Peaks Analysis Functions ########################################################################################################################################################################################

#' Helper function for suppressing function progress messages
#'
#' @param message_text Character string containing the message being shown
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the messsage will be suppressed (default = TRUE).
#'
verbose_message <- function(message_text, verbose) {
  if (verbose) {
    message(message_text)
  }
}

#' Runs pooling of a merged Seurat Dataset to generate pseudo-replicates for each sample
#'       - This function is used by run_edgeR_annotations()
#'
#' @param data.filt A Seurat Object containing count values for pooling.
#' @param idents A character string defining the idents column to pool the data against.
#' @param n An integer defining the amount of pseudo-replicates to generate for each sample (default = 3).
#' @param assay Character string defining the assay where the mz count data and annotations are stored (default = "Spatial").
#' @param slot Character string defining the assay storage slot to pull the relative mz intensity values from (default = "counts").
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @returns A SinglCellExpereiment object which contains pooled (n)-pseudo-replicate counts data based on the Seurat Object input
#' @export
#'
#' @examples
#' # run_pooling <- list(seuratObj, idents = "sample", n = 3, assay = "Spatial", slot = "counts")
run_pooling <- function(data.filt, idents, n, assay, slot, verbose = TRUE) {

  cell_metadata <- data.filt@meta.data
  samples <- unique(cell_metadata[[idents]])

  verbose_message(message_text = paste0("Pooling one sample into ", n ," replicates..."), verbose = verbose)

  nrg <- n
  for(i in c(1:length(samples))){
    set.seed(i)
    wo<-which(cell_metadata[[idents]]== samples[i])
    cell_metadata[wo,'orig.ident2']<-paste(samples[i],sample(c(1:n),length(wo)
                                                             ,replace=T,prob=rep(1/nrg,nrg)),sep='_')
  }
  gene_data <- row.names(data.filt)
  filtered.sce <- SingleCellExperiment::SingleCellExperiment(assays = list(counts = data.filt[[assay]][slot]),
                                       colData = cell_metadata)


  tempf=strsplit(filtered.sce@colData[["orig.ident2"]],'_')
  pid=NULL
  for(i in 1:length(tempf)){
    pidone=tempf[[i]]
    if(length(pidone)!=3){
      pidone=c(pidone[1],'yes',pidone[2])
    }
    pid=rbind(pid,pidone)
  }

  filtered.sce@colData$type=pid[,2]

  summed <- scater::aggregateAcrossCells(filtered.sce,
                                 id=SingleCellExperiment::colData(filtered.sce)[,'orig.ident2'])

  return(summed)
}




#' Runs EdgeR analysis for pooled data
#'       - This function is used by run_edgeR_annotations()
#'
#' @param pooled_data A SingleCellExperiment object which contains the pooled pseudo-replicate data.
#' @param seurat_data A Seurat object containing the merged Xenium data being analysed (this is subset).
#' @param ident A character string defining the ident column to perform differential expression analysis against.
#' @param output_dir A character string defining the ident column to perform differential expression analysis against.
#' @param run_name A character string defining the title of this DE analysis (will be used when saving DEGs to .csv file).
#' @param n An integer that defines the number of pseudo-replicates per sample (default = 3).
#' @param logFC_threshold A numeric value indicating the logFC threshold to use for defining significant genes (default = 1.2).
#' @param assay A character string defining the assay where the mz count data and annotations are stored (default = "Spatial").
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#' @param return.individual Boolean value defining whether to return a list of individual edgeR objects for each designated ident. If FALSE, one merged edgeR object will be returned (default = FALSE).
#'
#' @returns A modified edgeR object which contains the relative pseudo-bulking analysis outputs, including a DEGs data.frame with a list of differential expressed m/z peaks
#' @export
#'
#' @examples
#' # pooled_obj <- run_pooling(SeuratObj, "sample", n = 3)
#' # run_DE(pooled_obj, SeuratObj, "sample", "~/Documents/DE_output/", "run_1", n = 3, logFC_threshold = 1.2, annotation.column = "all_IsomerNames", assay = "Spatial")
run_DE <- function(pooled_data, seurat_data, ident, output_dir, run_name, n, logFC_threshold, assay, return.individual = FALSE, verbose = TRUE){

  verbose_message(message_text = paste("Running edgeR DE Analysis for ", run_name, " -> with samples [", paste(unique(unlist(seurat_data@meta.data[[ident]])), collapse = ", "), "]"), verbose = verbose)

  annotation_result <- list()
  for (condition in unique(seurat_data@meta.data[[ident]])){
    verbose_message(message_text = paste0("Starting condition: ",condition), verbose = verbose)

    groups <- SingleCellExperiment::colData(pooled_data)[[ident]]
    groups <- gsub(condition, "Comp_A", groups)
    groups <- ifelse(groups != "Comp_A", "Comp_B", groups)



    y <- edgeR::DGEList(SingleCellExperiment::counts(pooled_data), samples=SingleCellExperiment::colData(pooled_data)$orig.ident2, group = groups)


    y$samples$condition <- groups
    y$samples$ident <- sub("_(.*)", "", y$samples$samples)

    keep <- edgeR::filterByExpr(y, group = groups, min.count = 2, min.total.count = 10)
    y <- y[keep,]
    y <- edgeR::calcNormFactors(y)

    design <- stats::model.matrix(~groups)

    design[,2] <- 1-design[,2]

    y <- edgeR::estimateDisp(y, design, robust=TRUE)
    fit <- edgeR::glmQLFit(y, design, robust=TRUE)
    res <- edgeR::glmTreat(fit, coef=ncol(fit$design), lfc=log2(logFC_threshold))
    summary(limma::decideTests(res))
    res$table$regulate <- dplyr::recode(as.character(limma::decideTests(res)),"0"="Normal","1"="Up","-1"="Down")
    de_group_edgeR <- res$table[order(res$table$PValue),]
    table(limma::decideTests(res))
    res <- edgeR::topTags(res,n = nrow(y))
    res$table$regulate <- "Normal"
    res$table$regulate[res$table$logFC>0 & res$table$FDR<0.05] <- "Up"
    res$table$regulate[res$table$logFC<0 & res$table$FDR<0.05] <- "Down"
    de_group_edgeR <- res$table[order(res$table$FDR),]
    #table(de_group_edgeR$regulate)


    if (!(is.null(output_dir))){
      utils::write.csv(de_group_edgeR, paste0(output_dir,condition,"_",run_name, ".csv"))
    }

    de_group_edgeR$gene <- rownames(de_group_edgeR)

    y$DEGs <- de_group_edgeR
    annotation_result[[condition]] <- y

  }

  if (return.individual){
    annotation_result
    return(annotation_result)
  } else {



    edger <- edgeR::DGEList(
      counts = annotation_result[[1]]$counts,
      samples = annotation_result[[1]]$samples
    )
    edger$samples$group <- edger$samples$ident
    edger$samples$condition <- NULL

    degs <- lapply(names(annotation_result), function(x){
      annotation_result[[x]]$DEGs$cluster <- x
      rownames(annotation_result[[x]]$DEGs) <- NULL
      annotation_result[[x]]$DEGs
    })

    combined_degs <- do.call(rbind, degs)
    rownames(combined_degs) <- 1:length(combined_degs$cluster)

    edger$DEGs <- combined_degs
    return(edger)
  }

}


#' Finds all differentially expressed genes between comparison groups specified
#'       - This function uses run_pooling() and run_DE() to pool and run EdgeR analysis
#'
#' @param data A Seurat object
#' @param ident A character string defining the metadata column or groups to compare genes values between.
#' @param n An integer that defines the number of pseudo-replicates (pools) per sample (default = 3).
#' @param logFC_threshold A numeric value indicating the logFC threshold to use for defining significant genes (default = 1.2).
#' @param DE_output_dir A character string defining the directory path for all output files to be stored. This path must a new directory. Else, set to NULL as default.
#' @param run_name A character string defining the title of this DE analysis that will be used when saving DEDs to .csv file (default = 'FindAllDEGs').
#' @param assay A character string defining the assay where the count data and annotations are stored (default = "Spatial").
#' @param slot Character string defining the assay storage slot to pull the relative expression values from. Note: EdgeR requires raw counts, all values must be positive (default = "counts").
#' @param return.individual Boolean value defining whether to return a list of individual edgeR objects for each designated ident. If FALSE, one merged edgeR object will be returned (default = FALSE).
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @returns Returns an list() contains the EdgeR DE results. Pseudo-bulk counts are stored in $counts and DEGs are in $DEGs.
#' @export
#'
#' @examples
#' # FindAllDEGs(SeuratObj, "sample",DE_output_dir = "~/Documents/DE_output/")
FindAllDEGs <- function(data, ident, n = 3, logFC_threshold = 1.2, DE_output_dir = NULL, run_name = "FindAllDEGs", assay = "Spatial", slot = "counts", return.individual = FALSE, verbose = TRUE){

  if (!(is.null(DE_output_dir))){
    if (dir.exists(DE_output_dir)){
      warning("Please supply a directory path that doesn't already exist")
      stop("dir.exists(DE_output_dir) = TRUE")
    } else{
      dir.create(DE_output_dir)
    }
  }

  #Step 1: Run Pooling to split each unique ident into 'n' number of pseudo-replicate pools
  pooled_data <- run_pooling(data,ident, n = n, assay = assay, slot = slot, verbose = verbose)

  #Step 2: Run EdgeR to calculate differentially expressed m/z peaks
  DEG_results <- run_DE(pooled_data, data, ident = ident, output_dir = DE_output_dir, run_name = run_name, n=n, logFC_threshold=logFC_threshold, assay = assay, verbose = verbose, return.individual = return.individual)

  # Returns an EDGEr object which contains the pseudo-bulk counts in $counts and DEGs in $DEGs
  return(DEG_results)

}




#' Generates a Heatmap of DEGs generated from edgeR analysis run using FindAllDEGs().
#'       - this function uses pheatmap() to plot data
#'
#' @param edgeR_output A list containing outputs from edgeR analysis (from FindAllDEGs()). This includes pseudo-bulked counts and DEGs.
#' @param n A numeric integer that defines the number of UP and DOWN regulated peaks to plot (default = 25).
#' @param only.pos Boolean indicating if only positive markers should be returned (default = FALSE).
#' @param FDR.threshold Numeric value that defines the FDR threshold to use for defining most significant results (default = 0.05).
#' @param logfc.threshold Numeric value that defines the logFC threshold to use for filtering significant results (default = 0.5).
#' @param order.by Character string defining which parameter to order markers by, options are either 'FDR' or 'logFC' (default = "FDR").
#' @param scale A character string indicating if the values should be centered and scaled in either the row direction or the column direction, or none. Corresponding values are "row", "column" and "none"
#' @param color A vector of colors used in heatmap (default = grDevices::colorRampPalette(c("navy", "white", "red"))(50)).
#' @param cluster_cols Boolean value determining if columns should be clustered or hclust object (default = F).
#' @param cluster_rows Boolean value determining if rows should be clustered or hclust object (default = T).
#' @param fontsize_row A numeric value defining the fontsize of rownames (default = 15).
#' @param fontsize_col A numeric value defining the fontsize of colnames (default = 15).
#' @param cutree_cols A numeric value defining the number of clusters the columns are divided into, based on the hierarchical clustering(using cutree), if cols are not clustered, the argument is ignored (default = 9).
#' @param silent Boolean value indicating if the plot should not be draw (default = TRUE).
#' @param plot_annotations_column Character string indicating the column name that contains the metabolite annotations to plot. Annotations = TRUE must be used in FindAllDEGs() for edgeR output to include annotations. If plot_annotations_column = NULL, m/z vaues will be plotted (default = NULL).
#' @param save_to_path Character string defining the full filepath and name of the plot to be saved as.
#' @param plot.save.width Integer value representing the width of the saved pdf plot (default = 20).
#' @param plot.save.height Integer value representing the height of the saved pdf plot (default = 20).
#' @param nlabels.to.show Numeric value defining the number of annotations to show per m/z (default = NULL).
#'
#' @returns A heatmap plot of significantly differentially expressed peaks defined in the edgeR ouput object.
#' @export
#'
#' @import dplyr
#'
#' @examples
#' # DEGs <- FindAllDEGs(SeuratObj, "sample")
#'
#' # DEGsHeatmap(DEGs)
DEGsHeatmap <- function(edgeR_output,
                         n = 5,
                         only.pos = FALSE,
                         FDR.threshold = 0.05,
                         logfc.threshold = 0.5,
                         order.by = "FDR",
                         scale ="row",
                         color = grDevices::colorRampPalette(c("navy", "white", "red"))(50),
                         cluster_cols = F,
                         cluster_rows = T,
                         fontsize_row = 15,
                         fontsize_col = 15,
                         cutree_cols = 9,
                         silent = TRUE,
                         save_to_path = NULL,
                         plot.save.width = 20,
                         plot.save.height = 20){


  degs <- edgeR_output$DEGs
  degs <- subset(degs, FDR < FDR.threshold)

  if (order.by == "FDR"){

    grouped_pos<- degs %>%
      group_by(cluster) %>%
      filter( logFC > logfc.threshold) %>%
      arrange(desc(regulate)) %>%
      slice_head(n = n)


    if (only.pos) {
      grouped_neg <- NULL

    } else {
      grouped_neg <- degs %>%
        group_by(cluster) %>%
        filter(logFC < - logfc.threshold) %>%
        arrange(regulate) %>%
        slice_head(n = n)
    }
    df <- do.call(rbind, list(grouped_pos,grouped_neg))
    df <- df[order(df$cluster, dplyr::desc(df$regulate)), ]

  } else {
    if ( order.by != "logFC"){
      warning("order.by has invalid argument. Must be either 'FDR' or 'logFC'. Heatmap defaulting to order by logFC")
    }

    grouped_pos<- degs %>%
      group_by(cluster) %>%
      filter(logFC > logfc.threshold) %>%
      arrange(-logFC) %>%
      slice_head(n = n)


    if (only.pos) {
      grouped_neg <- NULL
    } else {
      grouped_neg <- degs %>%
        group_by(cluster) %>%
        filter(logFC < - logfc.threshold) %>%
        arrange(logFC) %>%
        slice_head(n = n)
    }
    df <- do.call(rbind, list(grouped_pos,grouped_neg))
    df <- df[order(df$cluster, -df$logFC), ]
  }



  col_annot <- data.frame(sample = edgeR_output$samples$ident)
  row.names(col_annot) <- colnames(as.data.frame(edgeR::cpm(edgeR_output,log=TRUE)))

  mtx <- as.matrix(as.data.frame(edgeR::cpm(edgeR_output,log=TRUE))[unique(df$gene),])

  p <- pheatmap::pheatmap(mtx,scale=scale,color=color,cluster_cols = cluster_cols, annotation_col=col_annot, cluster_rows = cluster_rows,
                          fontsize_row = fontsize_row, fontsize_col = fontsize_col, cutree_cols = cutree_cols, silent = silent)

   if (!(is.null(save_to_path))){
     save_pheatmap_as_pdf(pheatmap = p, filename = save_to_path, width = plot.save.width, height = plot.save.height)
   }

  return(p)
}


#' Saves the DEGs generated pheatmap as a PDF
#'
#' @param pheatmap A pheatmap plot object that is being saved.
#' @param filename Character string defining the full filepath and name of the plot to be saved as.
#' @param width Integer value representing the width of the saved pdf plot (default = 20).
#' @param height Integer value representing the height of the saved pdf plot (default = 20).
#'
#' @export
#'
#' @examples
#' # save_pheatmap_as_pdf(pheatmap, filename = "/Documents/plots/pheatmap1")
save_pheatmap_as_pdf <- function(pheatmap, filename, width=20, height=20){

  pdf(paste0(filename,".pdf"), width=width, height=height)
  grid::grid.newpage()
  grid::grid.draw(pheatmap$gtable)
  dev.off()
}

########################################################################################################################################################################################################################

In [ ]:
gene_markers_X <- FindAllDEGs(tumour, assay = "SPT", slot = "data", ident = "SPM_Clusters")

In [ ]:
head(gene_markers_X$DEGs)

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

DEGs <- gene_markers_X$DEGs[ gene_markers_X$DEGs$cluster == "2",]

keyvals <- ifelse(
    DEGs$logFC < 0  & DEGs$FDR < 10e-4, '#FB9A99',
      ifelse(DEGs$logFC > 0 & DEGs$FDR < 10e-4, '#A6CEE3',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == '#A6CEE3'] <- 'Up'
  names(keyvals)[keyvals == 'black'] <- 'Normal'
  names(keyvals)[keyvals == '#FB9A99'] <- 'Down'


volcPlot <- EnhancedVolcano::EnhancedVolcano(DEGs,
                                             selectLab = DEGs$gene[1:10], 
                                             lab = DEGs$gene,
                                            FCcutoff = 0,
                                            colCustom = keyvals,
                                            cutoffLineType = 'blank',
                                            pCutoff = 10e-5,
                                            x = 'logFC',
                                            y = 'FDR', 
                                            gridlines.major = FALSE,
                                            gridlines.minor = FALSE) 

volcPlot

In [ ]:
ggsave(volcPlot, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/DE_genes_2_0.pdf",width = 10, height = 10)

In [ ]:
gene_markers <- FindAllMarkers(tumour, assay = "SPT", slot = "data", logfc.threshold = 0)

In [ ]:
human_gene_markers <- gene_markers[grep("hg38-", gene_markers$gene),]
human_gene_markers$gene_name <- gsub("hg38-", "", human_gene_markers$gene)

In [ ]:
DEGs <- human_gene_markers[human_gene_markers$cluster == "2",]
rownames(DEGs) <- DEGs$gene_name

In [ ]:
head(DEGs)

In [ ]:
colour_palate_SPM

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 10)

keyvals <- ifelse(
    DEGs$avg_log2FC < 0  & DEGs$p_val_adj < 10e-4, '#FB9A99',
      ifelse(DEGs$avg_log2FC > 0 & DEGs$p_val_adj < 10e-4, '#A6CEE3',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == '#A6CEE3'] <- 'Up'
  names(keyvals)[keyvals == 'black'] <- 'Normal'
  names(keyvals)[keyvals == '#FB9A99'] <- 'Down'


volcPlot <- EnhancedVolcano::EnhancedVolcano(DEGs,
                                             selectLab = rownames(DEGs)[1:10], 
                                             lab = rownames(DEGs),
                                            FCcutoff = 0,
                                            colCustom = keyvals,
                                            cutoffLineType = 'blank',
                                            pCutoff = 10e-5,
                                            x = 'avg_log2FC',
                                            y = 'p_val_adj', 
                                            gridlines.major = FALSE,
                                            gridlines.minor = FALSE) 

volcPlot

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 30)

SpatialFeaturePlot(merged_MALDIVIS, features = c("hg38-HMGN2", "hg38-HMGB2", "hg38-VIM", "hg38-MAP1B", "hg38-NNAT", "hg38-PTMA"), pt.size.factor = 2)

In [ ]:
all_DE <- FindAllDEGs(merged_MALDIVIS, assay = "SPT", slot = "data", ident = "SPM_Clusters")

In [ ]:
write.csv(human_gene_markers, "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/Visium/de_genes.csv")

In [ ]:
x <- DEGsHeatmap(all_DE, only.pos = F, n = 5)

In [ ]:
save_pheatmap_as_pdf(pheatmap = x, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/DE_cluster.pdf", height = 10, width = 10)

In [ ]:
all_DEGs <- FindAllDEGs(merged_MALDIVIS, assay = "SPT", slot = "data", ident = "SPT_Clusters")

In [ ]:
x <- DEGsHeatmap(all_DEGs, only.pos = F)

In [ ]:
save_pheatmap_as_pdf(pheatmap = x, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/DE_ST_cluster.pdf", height = 10, width = 10)

In [ ]:
tumour_DEPs <- FindAllDEPs(tumour, "SPM_Clusters", n = 3, logFC_threshold = 1.2, DE_output_dir =NULL,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames", assay = "SPM", slot = "counts")


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 10)

keyvals <- ifelse(
    tumour_DEPs[["2"]]$DEPs$logFC < 0  & tumour_DEPs[["2"]]$DEPs$PValue < 10e-5, 'royalblue',
      ifelse(tumour_DEPs[["2"]]$DEPs$logFC > 0 & tumour_DEPs[["2"]]$DEPs$PValue < 10e-5, 'red',
        'black'))
  keyvals[is.na(keyvals)] <- 'black'
  names(keyvals)[keyvals == 'red'] <- 'Up'
  names(keyvals)[keyvals == 'black'] <- 'Normal'
  names(keyvals)[keyvals == 'royalblue'] <- 'Down'


volcPlot <- EnhancedVolcano::EnhancedVolcano(tumour_DEPs[["2"]]$DEPs,
                                             selectLab = c("mz-448.246729024761", "mz-1383.83486970307"), 
    #lab = rep("", length(rownames(tumour_DEPs[["2"]]$DEPs))),
                                             lab = rownames(tumour_DEPs[["2"]]$DEPs),
    FCcutoff = 0,
    colCustom = keyvals,
    cutoffLineType = 'blank',
    pCutoff = 10e-5,
    x = 'logFC',
    y = 'PValue', 
    gridlines.major = FALSE,
    gridlines.minor = FALSE)

In [ ]:
volcPlot

In [ ]:
#ggsave(volcPlot, filename = paste0(OUT_DIR, "plots/volcano.pdf"), height = 10, width = 7)

#### Spatially Plot Markers

In [ ]:
x <- ScaleData(merged_MALDIVIS)

In [ ]:
SearchAnnotations(merged_MALDIVIS, metabolite = "Malic acid", assay = "SPM",search.exact =F)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 8)

SpatialMZPlot(merged_MALDIVIS, mzs = 207.0135, #plusminus = 0.1,
              assay = "SPM", slot = "counts")

In [ ]:
colnames(x@meta.data)

In [ ]:
# this block of code is for combining the values of a metabolite into 1

bin_metabolites <- function(data, mzs, assay = "Spatial", slot = "data", bin_name = "Binned_Metabolites"){

    binned_counts <- colSums(data[[assay]][slot][mzs,])
    data[[bin_name]]<- binned_counts
    return(data)  
}


In [ ]:
df <- tumour_DEPs[["2"]]$DEPs[tumour_DEPs[["2"]]$DEPs$regulate == "Down",]

filtered_rows <- df[grepl("Ganglioside GM2", df$annotations), ]

# Get the row names of the filtered rows
filtered_row_names <- rownames(filtered_rows)

In [ ]:
merged_MALDIVIS_GM2 <- bin_metabolites(x, filtered_row_names, assay = "SPM", bin_name = "Ganglioside GM2's", slot = "scale.data")
human_spots <- rownames(merged_MALDIVIS_GM2@meta.data)[merged_MALDIVIS_GM2@meta.data$percent.human > 75]
merged_human <- subset(merged_MALDIVIS_GM2, cells = human_spots)



In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

plots <- SpatialFeaturePlot(merged_human, features = "Ganglioside GM2's", crop = FALSE, pt.size.factor = 1.2, combine = FALSE)

for (i in 1:length(plots)){
    plots[[i]] <- plots[[i]] & scale_fill_gradientn(colors = viridis(10), limits = c(0,10), na.value = viridis(10)[1])  & ggtitle("") 
}

plot <- do.call(ggarrange, c(plots, ncol = 2, nrow = 2, common.legend = TRUE, legend = "right"))

In [ ]:
plot 

ggsave(plot, filename = "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/Visium/plots/Ganglioside.pdf", height = 15, width = 15)

In [ ]:
vlndata <- VlnPlot(merged_human, features = "Ganglioside GM2's", group.by = "sample" )

In [ ]:
vlndata[[1]]$data$treated <- gsub("1", "", vlndata[[1]]$data$ident)
vlndata[[1]]$data$treated <- gsub("2", "", vlndata[[1]]$data$treated )

In [ ]:
ggviolin(vlndata[[1]]$data, x = "treated", y = "Ganglioside GM2's", fill = "treated",
         add = "boxplot", add.params = list(fill = "white"))+
  stat_compare_means(comparisons = list(c("T", "C")), label = "p.signif")+ # Add significance levels
  stat_compare_means(label.y = 50)                                      # Add global the p-value 

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 8)

plot <- ggviolin(vlndata[[1]]$data, x = "ident", y = "Ganglioside GM2's", fill = "treated",
         add = "boxplot", add.params = list(fill = "white"))

plot 

ggsave(plot, filename = "/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/Visium/plots/Ganglioside_vlnplot.pdf", height = 8, width = 8)

In [ ]:
refined_Lipid_DEPs <- RefineLipids(tumour_DEPs$DEPs)

In [ ]:
clust2 <- refined_Lipid_DEPs[refined_Lipid_DEPs$cluster == "2",]

In [ ]:
UP <- clust2[clust2$regulate == "Up",]

In [ ]:
GL <- UP[UP$Lipid.Maps.Category == "GL",]

In [ ]:
GLs <- na.omit(GL)$gene

In [ ]:
df <- UP[UP$gene %in% GLs,]


In [ ]:
merged_MALDIVIS_GM2 <- BinMetabolites(merged_MALDIVIS, df[c(colnames(df)[1:6],"gene")][1:33,]$gene, assay = "SPM", bin_name = "GL's", slot = "counts")
#human_spots <- rownames(merged_MALDIVIS_GM2@meta.data)[merged_MALDIVIS_GM2@meta.data$percent.human > 75]
merged_human <- subset(merged_MALDIVIS_GM2, subset = SPM_Clusters %in% c("0","2"))


options(repr.plot.width = 15, repr.plot.height = 15)

plots <- SpatialFeaturePlot(merged_human, features = "GL's", crop = FALSE, pt.size.factor = 2, combine = FALSE)

for (i in 1:length(plots)){
    plots[[i]] <- plots[[i]] & scale_fill_gradientn(colors = viridis::magma(10), limits = c(0,1500), na.value = magma(10)[10])  & ggtitle("") 
}

plot <- do.call(ggarrange, c(plots, ncol = 2, nrow = 2, common.legend = TRUE, legend = "right"))

In [ ]:
ggsave(plot, filename = "/scratch/user/uqacause/files/project_data_objects/MB/plots/GLs.pdf", height = 15, width = 15)

In [ ]:
T1_test <- subset(merged_MALDIVIS, subset = sample =="T1") 

In [ ]:
SeuratsRunMoransI <- function(data, pos, verbose = TRUE) {
  mysapply <- sapply
  if (verbose) {
    message("Computing Moran's I")
    mysapply <- pbsapply
  }
  Rfast2.installed <- PackageCheck("Rfast2", error = FALSE)
  if (Rfast2.installed) {
    MyMoran <- Rfast2::moranI
  } else if (!PackageCheck('ape', error = FALSE)) {
    stop(
      "'RunMoransI' requires either Rfast2 or ape to be installed",
      call. = FALSE
    )
  } else {
    MyMoran <- ape::Moran.I
    if (getOption('Seurat.Rfast2.msg', TRUE)) {
      message(
        "For a more efficient implementation of the Morans I calculation,",
        "\n(selection.method = 'moransi') please install the Rfast2 package",
        "\n--------------------------------------------",
        "\ninstall.packages('Rfast2')",
        "\n--------------------------------------------",
        "\nAfter installation of Rfast2, Seurat will automatically use the more ",
        "\nefficient implementation (no further action necessary).",
        "\nThis message will be shown once per session"
      )
      options(Seurat.Rfast2.msg = FALSE)
    }
  }
  pos.dist <- dist(x = pos)
  pos.dist.mat <- as.matrix(x = pos.dist)
  # weights as 1/dist^2
  weights <- 1/pos.dist.mat^2
  diag(x = weights) <- 0
  results <- mysapply(X = 1:nrow(x = data), FUN = function(x) {
    tryCatch(
      expr = MyMoran(data[x, ], weights),
      error = function(x) c(1,1,1,1)
    )
  })
  pcol <- ifelse(test = Rfast2.installed, yes = 2, no = 4)
  results <- data.frame(
    observed = unlist(x = results[1, ]),
    p.value = unlist(x = results[pcol, ])
  )
  rownames(x = results) <- rownames(x = data)
  return(results)
}

RowVar <- function(x) {
    .Call('_Seurat_RowVar', PACKAGE = 'Seurat', x)
}

FindSpatiallyVariableMetabolites <- function(object, assay = "SPM", image = "slice1", slot = "counts", verbose = TRUE){ 
                                               
    features <- rownames(x = object[[assay]])
    spatial.location <- GetTissueCoordinates(object = object[[image]])
    data <- GetAssayData(object = object, slot = slot)
    data <- as.matrix(x = data[features, ])
    data <- data[RowVar(x = data) > 0, ]
    svf.info <- RunMoransI(data = object, pos = spatial.location, verbose = verbose)
    colnames(x = svf.info) <- paste0("MoransI_", colnames(x = svf.info))
    var.name <- paste0("moransi", ".spatially.variable")
    var.name.rank <- paste0(var.name, ".rank")
    svf.infox[[var.name]] <- FALSE
    svf.infox[[var.name]][1:(min(nrow(x = svf.infox), 2000))] <- TRUE
    svf.infox[[var.name.rank]] <- 1:nrow(x = svf.infox)
    object[[names(x = svf.infox)]] <- svf.infox
    object[[assay]][[names(x = svf.infox)]] <- svf.infox
        
    return(object)
}

In [ ]:
object

In [ ]:
df <- object@assays[["SPM"]]@meta.data

df <- df[order(df$moransi.spatially.variable.rank), ]

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

SpatialFeaturePlot(merged_MALDIVIS, features = df$mz_names[1:10], ncol = 8)

In [ ]:
DEGs <- refined_Lipid_DEPs[refined_Lipid_DEPs$cluster == "2",]

In [ ]:
write.csv(DEGs, "/scratch/user/uqacause/files/project_data_objects/MB/plots/refined_annotations_DE.csv")

In [ ]:
gene_markers_X <- FindAllDEGs(tumour, assay = "SPT", slot = "data", ident = "SPM_Clusters")
gene_markers <- gene_markers_X$DEGs[gene_markers_X$DEGs$cluster == "2",]

In [ ]:
filtered_df <- gene_markers %>%
  filter(str_starts(gene, "hg38-"))


In [ ]:
filtered_df$gene <- gsub("hg38-", "", filtered_df$gene)

In [ ]:
write.csv(filtered_df, "/scratch/user/uqacause/files/project_data_objects/MB/gene_markers_DE.csv")

In [ ]:
SearchAnnotations(merged

In [ ]:
SpatialMZAnnotationPlot(merged_MALDIVIS, "", assay = "SPM")

In [ ]:

UP <- refined_Lipid_DEPs[refined_Lipid_DEPs$regulate == "Up",]
Down <-refined_Lipid_DEPs[refined_Lipid_DEPs$regulate == "Down",]# Assuming your dataframe is called df and contains a column named 'regulate' indicating 'Up' or 'Down' entries,
# and a column named 'Lipid.Maps.Category' containing the categories.

# Load required libraries
library(dplyr)
library(ggplot2)

options(repr.plot.width = 10, repr.plot.height = 10)

# Compute counts of Lipid Maps Categories for 'Up' and 'Down' entries
category_counts <- refined_Lipid_DEPs %>%
  group_by(regulate, Lipid.Maps.Category) %>%
  summarise(count = n()) %>%
  ungroup()

category_counts <- category_counts %>%
  filter(regulate != "Normal", Lipid.Maps.Category != "NA")

# Plot bar graph
ggplot(category_counts, aes(x = Lipid.Maps.Category, y = count, fill = regulate)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Relative Number of Lipid Maps Categories for Up and Down Entries",
       x = "Lipid Maps Category",
       y = "Count") +
  theme_classic() +
  #theme(axis.text.x = element_text(angle = 0, hjust = 1)) +
  scale_fill_manual(values = c("Up" = "red", "Down" = "blue"))  # Customizing colors for 'Up' and 'Down' entries


In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10)
# Load required libraries
library(dplyr)
library(ggplot2)

# Compute counts of Lipid Maps Categories for 'Up' and 'Down' entries
category_counts <- refined_Lipid_DEPs %>%
  group_by(regulate, Lipid.Maps.Main.Class) %>%
  summarise(count = n()) %>%
  ungroup()

category_counts <- category_counts %>%
  filter(regulate != "Normal", Lipid.Maps.Main.Class != "NA")

# Plot bar graph
ggplot(category_counts, aes(x = Lipid.Maps.Main.Class, y = count, fill = regulate)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Relative Number of Lipid Maps Categories for Up and Down Entries",
       x = "Lipid Maps Category",
       y = "Count") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_manual(values = c("Up" = "red", "Down" = "blue"))  # Customizing colors for 'Up' and 'Down' entries


In [ ]:
category_counts_GL <- subset(category_counts, subset = Lipid.Maps.Category == "GL")

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5)
# Load required libraries
library(dplyr)
library(ggplot2)

# Compute counts of Lipid Maps Categories for 'Up' and 'Down' entries
category_counts <- category_counts_GL %>%
  group_by(regulate, Lipid.Maps.Main.Class) %>%
  summarise(count = n()) %>%
  ungroup()

category_counts <- rbind(category_counts, c("Down", "MG", 0))
category_counts$count <- as.numeric(category_counts$count)

category_counts <- category_counts %>%
  filter(regulate != "Normal", Lipid.Maps.Main.Class != "NA")

# Plot bar graph
plot <- ggplot(category_counts, aes(x = Lipid.Maps.Main.Class, y = count, fill = regulate)) +
  geom_bar(stat = "identity", position = "dodge") +
  labs(title = "Relative Number of Lipid Maps Categories for Up and Down Entries",
       x = "Lipid Maps Category",
       y = "Count") +
  theme_classic() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  scale_fill_manual(values = c("Up" = "red", "Down" = "blue"))  # Customizing colors for 'Up' and 'Down' entries


In [ ]:
ggsave(plot, filename = paste0(OUT_DIR, "plots/lipids_0_2.pdf"), height = 5, width = 5)

In [ ]:
tumour_DEPs <- FindAllDEPs(merged_MALDIVIS, "SPM_Clusters", n = 3, logFC_threshold = 1.2, DE_output_dir =NULL,
                           run_name = "FindAllDEPs", 
                           annotation.column = "all_IsomerNames", assay = "SPM", slot = "counts")


In [ ]:
markers <- FindAllMarkers(merged_MALDIVIS, assay = "SPM", only.pos = TRUE)

In [ ]:
write.csv(markers, paste0(OUT_DIR, "DE/markers_HMDB.csv"))

In [ ]:
library(dplyr)

top_genes <- markers %>%
  group_by(cluster) %>%
  slice_head(n = 5) %>%
  ungroup() 

rownames(top_genes) <- top_genes$gene

In [ ]:
merged_MALDIVIS <- ScaleData(merged_MALDIVIS,assay = "SPM")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
DoHeatmap(merged_MALDIVIS, features = top_genes$gene, assay = "SPM")


In [ ]:
feature_meta.data <- merged_MALDIVIS@assays$SPM@meta.data
rownames(feature_meta.data) <- feature_meta.data$mz_names
annots <- c()
for (i in top_genes$gene){
    row <- feature_meta.data[i,]
    annotation <- row$all_IsomerNames
    annots <- c(annots, annotation)
}

top_genes$annotations <- annots

top_genes$first_3_annotations <- sapply(strsplit(top_genes$annotations, "; "), function(x) {
  # Filter out NA values and select the first three entries
  non_na_values <- x[!is.na(x)]
  paste(non_na_values[1:min(1, length(non_na_values))], collapse = "; ")
})

combined_mtx <- tumour_DEPs[["0"]]$counts[intersect(rownames(top_genes), rownames(tumour_DEPs[["0"]]$counts)),]

FDR_threshold = 1.2
degs <- top_genes
#degs$gene <- rownames(degs)
degs <- subset(degs, p_val_adj < FDR_threshold)
degs <- degs[order(degs$cluster),]
rownames(degs) <- degs$gene
col_annot <- data.frame(sample = tumour_DEPs[["0"]]$samples$ident)
row.names(col_annot) <- colnames(as.data.frame(edgeR::cpm(tumour_DEPs[["0"]],log=TRUE)))



In [ ]:
FDR_threshold = 0.05
scale ="row"
color = grDevices::colorRampPalette(c("navy", "white", "red"))(50)
cluster_cols = F
cluster_rows = F
fontsize_row = 15
fontsize_col = 15
cutree_cols = 9
silent = TRUE
plot_annotations_column = NULL
save_to_path = NULL
plot.save.width = 20
plot.save.height = 20

mtx <- as.matrix(as.data.frame(edgeR::cpm(tumour_DEPs[["0"]],log=TRUE))[rownames(degs),])


rownames(mtx) <- degs[["first_3_annotations"]]

p <- pheatmap::pheatmap(mtx,scale=scale,color=color,cluster_cols = cluster_cols, annotation_col=col_annot, cluster_rows = cluster_rows,
            fontsize_row = fontsize_row, fontsize_col = fontsize_col, cutree_cols = cutree_cols, silent = silent)

options(repr.plot.width =20, repr.plot.height = 10)
#pdf("/QRISdata/Q1851/Andrew_C/Metabolomics/Test_data/public_data/Results/deps.pdf", height = 10, width = 20)
p

In [ ]:
pdf(paste0(OUT_DIR, "plots/heatmap_HMDB.pdf"), height = 10, width = 23)
p
dev.off()

In [ ]:
merged_MALDIVIS

In [ ]:
## Scoring plots for SSH-A
SpatialFeaturePlot(merged_MALDIVIS, features = "SHH.A.enrich.scores")

In [ ]:
head(merged_MALDIVIS)

In [ ]:
Idents(merged_MALDIVIS) <- "SPM_Clusters"
sub_set <- subset(merged_MALDIVIS, subset = SPM_Clusters %in% c("0", "2", "5"))
df <- sub_set@meta.data


df$SHH.A.enrich.scores <- as.numeric(df$SHH.A.enrich.scores)
df$SHH.B.enrich.scores <- as.numeric(df$SHH.B.enrich.scores)
df$SHH.C.enrich.scores <- as.numeric(df$SHH.C.enrich.scores)

# Plot box plots
boxplot_data <- df[, c( "SHH.C.enrich.scores", "SHH.A.enrich.scores",
                       "SPM_Clusters")]

# Melt data for boxplot
boxplot_data_long <- reshape2::melt(boxplot_data, id.vars = "SPM_Clusters")

# Plot boxplots
box <- ggplot(boxplot_data_long, aes(x = SPM_Clusters, y = value, fill = variable)) +
  geom_boxplot() +
  labs(x = "SPM Clusters", y = "Enrichment Scores", fill = "Scores") +
  theme_classic()



In [ ]:
box

In [ ]:
options(repr.plot.width =6, repr.plot.height = 6)
ggsave(box, filename = paste0(OUT_DIR, "plots/SSH.scores.pdf"), height = 6, width = 6)

In [ ]:
boxplot_data_long

In [ ]:
anova_result <- aov(value ~ SPM_Clusters, data = boxplot_data_long)

# Summary of ANOVA
summary(anova_result)

# Post-hoc tests (Tukey's HSD)
tukey_result <- TukeyHSD(anova_result)

In [ ]:
tukey_result

In [ ]:
DefaultAssay(merged_MALDIVIS) <- "SPT"

In [ ]:
Idents(merged_MALDIVIS) <- "SPT_Clusters"

In [ ]:
gene_markers <- FindAllMarkers(merged_MALDIVIS)

In [ ]:
write.csv(gene_markers, paste0(OUT_DIR,"DE/gene_markers.csv"))

In [ ]:
FDR_threshold = 0.05
scale ="row"
color = grDevices::colorRampPalette(c("navy", "white", "red"))(50)
cluster_cols = F
cluster_rows = F
fontsize_row = 15
fontsize_col = 15
cutree_cols = 9
silent = TRUE
plot_annotations_column = NULL
save_to_path = NULL
plot.save.width = 20
plot.save.height = 20
top5 <- list()
#top5_mtx <- list()
for (cluster in names(tumour_DEPs)){
    df <- tumour_DEPs[[cluster]]$DEPs[tumour_DEPs[[cluster]]$DEPs$regulate == "Up",]
    if (dim(df)[1] > 1){
        
        df <- head(df, n = 5)
        
        #mtx <- tumour_DEPs[[cluster]]$counts[rownames(tumour_DEPs[[cluster]]$counts)[rownames(tumour_DEPs[[cluster]]$counts) %in% rownames(df)],]
        df$cluster <- cluster
        top5[[cluster]] <- df
    }
    #top5_mtx[[cluster]] <- mtx
}


In [ ]:
combined_df <- do.call(rbind, unname(top5))
combined_df$first_3_annotations <- sapply(strsplit(combined_df$annotations, "; "), function(x) {
  # Filter out NA values and select the first three entries
  non_na_values <- x[!is.na(x)]
  paste(non_na_values[1:min(2, length(non_na_values))], collapse = "; ")
})
combined_mtx <- tumour_DEPs[[cluster]]$counts[intersect(rownames(combined_df), rownames(tumour_DEPs[[cluster]]$counts)),]

degs <- combined_df
degs$gene <- rownames(degs)
degs <- subset(degs, FDR < FDR_threshold)
degs <- degs[order(degs$cluster),]
#top <- utils::head(degs, n=n)
#bot <- utils::tail(degs, n=n)
#degs <- merge(x = top, y = bot, all = TRUE)

col_annot <- data.frame(sample = tumour_DEPs[[cluster]]$samples$ident)
row.names(col_annot) <- colnames(as.data.frame(edgeR::cpm(tumour_DEPs[[cluster]],log=TRUE)))

mtx <- as.matrix(as.data.frame(edgeR::cpm(tumour_DEPs[[cluster]],log=TRUE))[rownames(degs),])


rownames(mtx) <- degs[["first_3_annotations"]]


p <- pheatmap::pheatmap(mtx,scale=scale,color=color,cluster_cols = cluster_cols, annotation_col=col_annot, cluster_rows = cluster_rows,
            fontsize_row = fontsize_row, fontsize_col = fontsize_col, cutree_cols = cutree_cols, silent = silent)

options(repr.plot.width = 50, repr.plot.height = 10)
#pdf("/QRISdata/Q1851/Andrew_C/Metabolomics/Test_data/public_data/Results/deps.pdf", height = 10, width = 20)
p
#dev.off()

In [ ]:
library(plotly)
#' Generates a 3D spatial feature plot from a SpaMTP object
#'
#' @param data A SpaMTP Seurat Object.
#' @param features A character vector specifying the features to be plotted.
#' @param assays A character vector specifying the Seurat Object assays to be used for plotting (default = c("SPT", "SPM")).
#' @param slots A character vector specifying the assay slots to be used for plotting (deafult = "counts").
#' @param between.layer.height A numeric value specifying the height between layers (default = 100).
#' @param names A character vector specifying custom names for the features. If NULL will use feature names (default = NULL).
#' @param size A numeric value specifying the size of markers in the plot (default = 3).
#' @param x.axis.label A character string specifying the label for the x-axis (default = "x").
#' @param y.axis.label A character string specifying the label for the y-axis (default = "y").
#' @param z.axis.label A character string specifying the label for the z-axis (default = "z").
#' @param show.x.ticks A logical value specifying whether to show ticks on the x-axis (default = FALSE).
#' @param show.y.ticks A logical value specifying whether to show ticks on the y-axis (default = FALSE).
#' @param show.z.ticks A logical value specifying whether to show ticks on the z-axis (default = FALSE).
#' @param show.image A logical value specifying whether to overlay an image on the plot (default = FALSE).
#'
#'
#' @return A 3D Plotly plot
#' @export
#'
#' @examples
#' # Plot3DFeature(data = my_data, features = c("gene1", "gene2"), assays = c("SPT", "SPM"))
Plot3DFeatureX <- function(data,
                          features,
                          assays = c("SPT", "SPM"),
                          slots = "counts",
                          between.layer.height = 100,
                          names= NULL,
                          size = 3,
                          x.axis.label = "x",
                          y.axis.label = "y",
                          z.axis.label = "z",
                          show.x.ticks = FALSE,
                          show.y.ticks = FALSE,
                          show.z.ticks = FALSE,
                          show.image = FALSE){

  ## handeling of inncorect input legnths
  if (length(features) < 1 | length(features) > 2){
    stop("Number of features supplied does not match required length. Either 1 or 2 features must be supplied")
  }
  if (length(assays) > 2){
    stop("Number of assays supplied does not match required length. 2 assays or less must be supplied")
  }
  if (length(slots) > 2){
    stop("Number of slots supplied does not match required length. 2 slots or less must be supplied")
  }

  ## Handling of only 1 submitted value
  if (length(features) ==  1 ){
    features <- c(features, features)
  }
  if (length(assays) ==  1 ){
    assays <- c(assays, assays)
  }
  if (length(assays) ==  1 ){
    assays <- c(assays, assays)
  }


  feature_data <- list()

  i <- 1
  default_names <- c()
  for (feature in features) {

    if (feature %in% colnames(data@meta.data)){
      default_names <- c(default_names, feature)
      feature_data[[i]] <- data@meta.data[[feature]]
    } else {
      if (length(assays) != 0){

        feature_data[[i]] <- tryCatch({data[[assays[i]]][slots[i]][feature,]},
                                      error = function(err){
                                        stop("The feature provided does not exist in the ", assays[i],": ",slots[i], " object. Please provide a value feature")})
      } else {
        stop("No assay supplied. The feature ", feature," either does not exist in @meta.data slot, or no matching assay was provided for the gene/feature. Please check SpaMTP object")
      }
      default_names <- c(default_names, paste0(feature,"_", assays[i]))
    }
    i <- i + 1
  }
  if (!(is.null(names))){
    if (length(names) == 2) {
      default_names <- names
    } else {
      warning("length of names must be == 2. Default names will be used instead ... ")
    }
  }


  # Create scatter3d traces for each layer
  trace1 <-
    plot_ly(GetTissueCoordinates(data),
            x = ~imagerow,
            y = ~imagecol,
            z = rep(0 + between.layer.height, times = dim(GetTissueCoordinates(data))[1]),
            type = "scatter3d",
            mode = "markers",
            width = 2000,
            height = 1000,
            name = default_names[1],
            marker = list(color = feature_data[[1]],
                          coloraxis = 'coloraxis', size = size)) %>%
    layout(scene = list(
      aspectmode = "data",
      xaxis = list(title = x.axis.label, showticklabels = show.x.ticks),
      yaxis = list(title = y.axis.label, showticklabels = show.y.ticks),
      zaxis = list(title = z.axis.label, showticklabels = show.z.ticks)))

  plot <-  trace1 %>% add_trace(GetTissueCoordinates(data),
                                x = ~imagerow,
                                y = ~imagecol,
                                z = rep(0, times = dim(GetTissueCoordinates(data))[1]),
                                type = "scatter3d",
                                mode = "markers",
                                name = default_names[2],
                                marker = list(color = feature_data[[2]],
                                              coloraxis = 'coloraxis2')) %>%
    layout(autosize = F,
      scene = list(
        aspectmode = "data",
        xaxis = list(title = x.axis.label, showticklabels = show.x.ticks),
        yaxis = list(title = y.axis.label, showticklabels = show.y.ticks),
        zaxis = list(title = z.axis.label, showticklabels = show.z.ticks)),
      coloraxis = list(colorbar = list(orientation = "v",
                                       xanchor ="right",
                                       
                                       len = 0.5,
                                       x = 0,
                                       title = list( side = "top",
                                                     text = default_names[1]
                                       )), 
                      colorscale = "Hot",
                      cmin = 0,     # Minimum value for the color scale
                      cmax = 5    # Maximum value for the color scale
                      ),
      coloraxis2 = list(colorbar = list(orientation = "v",
                                        xanchor ="left",
                                        x = 0,
                                        len = 0.5,
                                        title = list( side = "top",
                                                      text = default_names[2]
                                        )), 
                       colorscale = "Electric",
                        cmin = 0,     # Minimum value for the color scale
                      cmax = 250    # Maximum value for the color scale
                       )

    )

  if (show.image){

    color_matrix <- GetImage(data)$raster
    row_indices <- rep(seq_len(nrow(color_matrix)), each = ncol(color_matrix))
    col_indices <- rep(seq_len(ncol(color_matrix)), times = nrow(color_matrix))

    # Flatten the color matrix into a vector
    colors <- as.vector(color_matrix)

    # Create a data frame
    df <- data.frame(row = row_indices,
                     col = col_indices,
                     color = colors)


    plot <- plot %>% add_trace(df,
                               x = df$row,
                               y = df$col,
                               z = rep(0 - between.layer.height, times = dim(df)[1]),
                               type = "scatter3d",
                               mode = "markers",
                               name = "H&E",
                               marker = list(color = df$color))

  }
  return(plot)

}


In [ ]:
human <- subset(merged_MALDIVIS, subset = percent.human > 75)

In [ ]:
T1 <- subset(human, subset = sample == "T1")

In [ ]:
C1 <- subset(human, subset = sample == "C1")

In [ ]:
p1 <- Plot3DFeatureX(T1, features = c("hg38-E2F1","mz-448.246729024761"), assays = c("SPT", "SPM"), slots = c("counts", "counts"), names = c("E2F", "Palbociclib"), between.layer.height = 150, show.image = T, size = 3)

p1

In [ ]:
p2 <- Plot3DFeatureX(C1, features = c("hg38-E2F1","mz-448.246729024761"), assays = c("SPT", "SPM"), slots = c("counts", "counts"), names = c("E2F", "Palbociclib"), between.layer.height = 150, show.image = T, size = 3)

p2

In [ ]:
library(htmlwidgets)

# Save the Plotly plot as an HTML file
saveWidget(p2, "/scratch/user/uqacause/files/project_data_objects/MB/plots/C1_plotly.html")

In [ ]:
saveWidget(p1, "/scratch/user/uqacause/files/project_data_objects/MB/plots/T1_plotly.html")


In [ ]:
p1

In [ ]:
suppressWarnings ({
    ## Sample C1
    C1_test_x <- cluster_seurat_obj(C1_test_x, res = 0.5, pcas = 50, clust_name = "seurat_cluster_0.5")
    C1_test_x <- cluster_seurat_obj(C1_test_x, res = 1, pcas = 50, clust_name = "seurat_cluster_1")
    
    ## Sample C2
    C2_test_x <- cluster_seurat_obj(C2_test_x, res = 0.5, pcas = 50, clust_name = "seurat_cluster_0.5")
    C2_test_x <- cluster_seurat_obj(C2_test_x, res = 1, pcas = 50, clust_name = "seurat_cluster_1")
    
    ## Sample C1
    T2_test_x <- cluster_seurat_obj(T2_test_x, res = 0.5, pcas = 50, clust_name = "seurat_cluster_0.5")
    T2_test_x <- cluster_seurat_obj(T2_test_x, res = 1, pcas = 50, clust_name = "seurat_cluster_1")
})

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 10)

(SpatialDimPlot(C1_test_x, group.by = "seurat_cluster_0.5")/ SpatialDimPlot(C1_test_x, group.by = "seurat_cluster_1"))|
(SpatialDimPlot(C2_test_x, group.by = "seurat_cluster_0.5")/ SpatialDimPlot(C2_test_x, group.by = "seurat_cluster_1"))|
(SpatialDimPlot(T1_test_x, group.by = "seurat_cluster_0.5")/ SpatialDimPlot(T1_test_x, group.by = "seurat_cluster_1"))|
(SpatialDimPlot(T2_test_x, group.by = "seurat_cluster_0.5")/ SpatialDimPlot(T2_test_x, group.by = "seurat_cluster_1"))

From looking at the clustering we can see that there is also regions that are being clustered which directly link to the expression of the drug

In [ ]:
vis_merge <- JoinLayers(merge(visium_C, y = list(visium_A, visium_D,visium_B), merge.data = TRUE))

## Cell State Spot Typing using Label Transfer

Similar to the enrichment scores above we can use label transfer and a reference single-cell dataset to peform label transfer to calculate cell-states of each spot. The reference dataset we will be using is from https://doi.org/10.1093/neuonc/noab135. Lets download the data, process it and peform label transfer!

In [ ]:
MB_SHH <- readRDS("/QRISdata/Q5291/Tuan/SHH_Riemondy/MB_SHH/MB_SHH.rds")

In [ ]:
# Reformats the counts and data slots of this seurat object, this is necessary for peforming clustering 
MB_SHH@assays$RNA$counts <- Matrix::Matrix(as.matrix(MB_SHH@assays$RNA$counts),sparse = T)
MB_SHH@assays$RNA$data <- MB_SHH@assays$RNA$counts

In [ ]:
## Peforms sample clustering
MB_data <- ScaleData(MB_SHH, assay = "RNA")
MB_data <- FindVariableFeatures(MB_data, verbose = FALSE, selection.method = "vst")
MB_data <- RunPCA(MB_data, assay = "RNA", verbose = FALSE)
MB_data <- FindNeighbors(MB_data, reduction = "pca", dims = 1:30, verbose = FALSE)
MB_data <- RunUMAP(MB_data, reduction = "pca", dims = 1:30, verbose = FALSE)
MB_data <- FindClusters(MB_data, resolution = 0.5, verbose = FALSE)

Lets look at the clustering and relative cell state annotations provided in the reference dataset

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

DimPlot(MB_data) + DimPlot(MB_data, group.by = "cell.cycle.phase")+ DimPlot(MB_data, group.by = "Subpopulations") + DimPlot(MB_data, group.by = "orig.ident")

These cell states are for human genes only, so we need to susbet the data to only include human genes

lets first plot the percentage of genes that are human and mouse in each sample 

We can see that there is a clear pattern of where the mouse and human tissue is. As we only want spots that have uman cells in them we will subset to only include spots that have atleast 10% human genes

## Deconvolution

We can see that there seems to be a trend that where the drug is there are less genes. This makes sense as the drug is supressing the rapid poliferation and activity of the cancer

Now lets run deconvolution using the reference dataset with CARD -> CARD is a deconvolution method that uses a reference scRNA dataset along with spatial information to generating spot deconvolution results

In [ ]:
CARD_Deconvolution <- function(data, ref_data, ident.column, assay = "SPT") {

    #use only connom genes 
    #common_genes <- intersect(rownames(ref_data), rownames(data[[assay]]$counts))
    
    #get spatial coordinates
    coords <- GetTissueCoordinates(data) 
    coords$y <- coords$imagerow
    coords$x <- coords$imagecol
    coords$imagerow <- NULL
    coords$imagecol <- NULL

    #generate CARD object
    CARD_obj = createCARDObject(
	sc_count = ref_data@assays$RNA$counts,#[common_genes,],
	sc_meta = ref_data@meta.data,
	spatial_count = data[[assay]]$counts,#[common_genes,],
	spatial_location = coords,
	ct.varname = ident.column,
	ct.select = unique(ref_data@meta.data[[ident.column]]),
	sample.varname = "orig.ident") 

    #Run deconvolution
    CARD_obj = CARD_deconvolution(CARD_object = CARD_obj)

    df <- data.frame(CARD_celltype = colnames(CARD_obj@Proportion_CARD)[max.col(CARD_obj@Proportion_CARD)])
    rownames(df) <- rownames(CARD_obj@Proportion_CARD)

    data_sub <- subset(data, cells = rownames(df))
    data_sub@meta.data$CARD_celltype <- df$CARD_celltype
    
    return(list("seurat_object" = data_sub,
                "card_object" = CARD_obj))
}


In [ ]:
VisA_decon <- CARD_Deconvolution(C1_human, MB_data, ident.column = "Subpopulations", assay = "human")
VisB_decon <- CARD_Deconvolution(T1_human, MB_data, ident.column = "Subpopulations", assay = "human")
VisC_decon <- CARD_Deconvolution(C2_human, MB_data, ident.column = "Subpopulations", assay = "human")
VisD_decon <- CARD_Deconvolution(T2_human, MB_data, ident.column = "Subpopulations", assay = "human")

In [ ]:
# VisA_decon <- CARD_Deconvolution(C1_MALDIVIS_human, MB_data, ident.column = "Subpopulations")
# VisB_decon <- CARD_Deconvolution(C2_MALDIVIS_human, MB_data, ident.column = "Subpopulations")
# VisC_decon <- CARD_Deconvolution(T1_MALDIVIS_human, MB_data, ident.column = "Subpopulations")
# VisD_decon <- CARD_Deconvolution(T2_MALDIVIS_human, MB_data, ident.column = "Subpopulations")

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

cols <- as.list(RColorBrewer::brewer.pal(name = "Paired", n =7))
names(cols) <- sort(unique(MB_data@meta.data$Subpopulations))

SpatialDimPlot(object = VisA_decon$seurat_object, group.by = "CARD_celltype", cols = cols) | 
SpatialDimPlot(object = VisB_decon$seurat_object, group.by = "CARD_celltype", cols = cols) |
SpatialDimPlot(object = VisC_decon$seurat_object, group.by = "CARD_celltype", cols = cols) |
SpatialDimPlot(object = VisD_decon$seurat_object, group.by = "CARD_celltype", cols = cols) 

In [ ]:
C1_human <- subset_organism_genes(VisA_decon$seurat_object, percent.threshold = 50, assay = "human")
T1_human <- subset_organism_genes(VisC_decon$seurat_object, percent.threshold = 50, assay = "human")
C2_human <- subset_organism_genes(VisB_decon$seurat_object, percent.threshold = 50, assay = "human")
T2_human <- subset_organism_genes(VisD_decon$seurat_object, percent.threshold = 50, assay = "human")

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

cols <- as.list(RColorBrewer::brewer.pal(name = "Paired", n =7))
names(cols) <- sort(unique(MB_data@meta.data$Subpopulations))

decon <- (SpatialDimPlot(object = C1_human, group.by = "CARD_celltype", cols = cols, crop = FALSE, pt.size.factor = 1.3, stroke = 0.1) | 
SpatialDimPlot(object = T1_human, group.by = "CARD_celltype", cols = cols, crop = FALSE, pt.size.factor = 1.3, stroke = 0.1) |
SpatialDimPlot(object = C2_human, group.by = "CARD_celltype", cols = cols, crop = FALSE, pt.size.factor = 1.3, stroke = 0.1) |
SpatialDimPlot(object = T2_human, group.by = "CARD_celltype", cols = cols, crop = FALSE, pt.size.factor = 1.3, stroke = 0.1)) 

decon 
ggsave(decon, filename = paste0(OUT_DIR, "plots/decon.pdf"), height = 7, width = 30)

In [ ]:
colors <- unlist(unname(cols))
p1 <- CARD.visualize.pie(
	proportion = VisA_decon$card_object@Proportion_CARD,
	spatial_location = VisA_decon$card_object@spatial_location, 
 	colors = colors, 
  	radius = 3) ### You can choose radius = NULL or your own radius number

p2 <- CARD.visualize.pie(
	proportion = VisB_decon$card_object@Proportion_CARD,
	spatial_location = VisB_decon$card_object@spatial_location, 
 	colors = colors, 
  	radius = 3)

p3 <- CARD.visualize.pie(
	proportion = VisC_decon$card_object@Proportion_CARD,
	spatial_location = VisC_decon$card_object@spatial_location, 
 	colors = colors, 
  	radius = 3)

p4 <- CARD.visualize.pie(
	proportion = VisD_decon$card_object@Proportion_CARD,
	spatial_location = VisD_decon$card_object@spatial_location, 
 	colors = colors, 
  	radius = 3)


In [ ]:
options(repr.plot.width = 30, repr.plot.height = 10)

p1|p2|p3|p4

We can also look at the proportions of each cell type across the tissue. Lets look at one of our <span style = "color: red;">**Treated**</span> samples for example

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

p2 <- CARD.visualize.prop(
	proportion = VisC_decon$card_object@Proportion_CARD,        
	spatial_location = VisC_decon$card_object@spatial_location, 
	ct.visualize = sort(unique(MB_data@meta.data$Subpopulations)),  ### selected cell types to visualize
	colors = c("lightblue","lightyellow","red"), ### if not provide, we will use the default colors
	NumCols = 4,                                 ### number of columns in the figure panel
        pointSize = 1)                             ### point size in ggplot2 scatterplot  
print(p2)


We can also look at correlation between each of these cell types within a tissue section. And we can look at two cell types of interest. In this case we will look at SHH-C2 and SHH-A1

In [ ]:
## Calculates cell type correlations
p3 <- CARD.visualize.Cor(VisC_decon$card_object@Proportion_CARD,colors = NULL) # if not provide, we will use the default colors

## Plots two cell types of interest on the same tissue section
p4  <- CARD.visualize.prop.2CT(
proportion = VisC_decon$card_object@Proportion_CARD,                             ### Cell type proportion estimated by CARD
spatial_location = VisC_decon$card_object@spatial_location,                      ### spatial location information
ct2.visualize = c("SHH-C2","SHH-A1"),              ### two cell types you want to visualize
colors = list(c("lightblue","lightyellow","red"),c("lightblue","lightyellow","black")))       ### two color scales                             

print(p3|p4)

In addition to CARD we can also run label transfer between our visium data and a reference medulablastoma scRNA dataset. 

First we have to process the data so that it is in the same format as the scRNA reference. We also need to cluster and generate embedding for *Seurat* to run 'label transfer'

In [ ]:
process_data <- function(data) {
    data <- NormalizeData(data)
    data <- ScaleData(data, assay = "Spatial")
    data <- FindVariableFeatures(data, verbose = FALSE, selection.method = "vst")
    data <- RunPCA(data, assay = "Spatial", verbose = FALSE)
    data <- FindNeighbors(data, reduction = "pca", dims = 1:30, verbose = FALSE)
    data <- RunUMAP(data, reduction = "pca", dims = 1:30, verbose = FALSE)
    return(data)
}

In [ ]:
suppressWarnings({
    # we will use the same objects that CARD was run on 
    visium_A_human <- process_data(VisA_decon$seurat_object)
    visium_B_human <- process_data(VisB_decon$seurat_object)
    visium_C_human <- process_data(VisC_decon$seurat_object)
    visium_D_human <- process_data(VisD_decon$seurat_object)

})

Great, lets run label transfer now and look how the results compare!

In [ ]:
run_label_transfer <- function(Ref_data, Query_data, ref.col.name  = "Subpopulations", out.col.name = "predicted.id") {
 
  comm_gene1 <- intersect(rownames(Ref_data), 
                        rownames(Query_data))
  
  anchors <- FindTransferAnchors(reference = Ref_data, 
                                 query = Query_data, features = comm_gene1)
  sc_obj.predictions.assay <- TransferData(anchorset = anchors, 
                                              refdata = Ref_data@meta.data[[col]],  
                                              weight.reduction = Query_data[["pca"]], 
                                              dims = 1:40) 
  
  rownames(sc_obj.predictions.assay) <- colnames(Query_data)

  
  Query_data$predicted.id <- sc_obj.predictions.assay[predicted.id]

    
  return(Query_data)
}

In [ ]:
visium_A_human <- run_label_transfer(MB_data,visium_A_human)
visium_B_human <- run_label_transfer(MB_data,visium_B_human)
visium_C_human <- run_label_transfer(MB_data,visium_C_human)
visium_D_human <- run_label_transfer(MB_data,visium_D_human)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

SpatialDimPlot(visium_A_human, group.by = "predicted.id", cols = cols)|
SpatialDimPlot(visium_B_human, group.by = "predicted.id", cols = cols)|
SpatialDimPlot(visium_C_human, group.by = "predicted.id", cols = cols)|
SpatialDimPlot(visium_D_human, group.by = "predicted.id", cols = cols)

Lets running this with immune cells included

In [ ]:
MB_ALL <- readRDS("/QRISdata/Q5291/Tuan/SHH_Riemondy/MB_All/MB_All.rds")

In [ ]:
MB_SHH <- subset(MB_ALL, subset = subgroup == "SHH") 

In [ ]:
# Reformats the counts and data slots of this seurat object, this is necessary for peforming clustering 
MB_SHH@assays$RNA$counts <- Matrix::Matrix(as.matrix(MB_SHH@assays$RNA$counts),sparse = T)
MB_SHH@assays$RNA$data <- MB_SHH@assays$RNA$counts## Peforms sample clustering

In [ ]:
MB_data <- ScaleData(MB_SHH, assay = "RNA")
MB_data <- FindVariableFeatures(MB_data, verbose = FALSE, selection.method = "vst")
MB_data <- RunPCA(MB_data, assay = "RNA", verbose = FALSE)
MB_data <- FindNeighbors(MB_data, reduction = "pca", dims = 1:30, verbose = FALSE)
MB_data <- RunUMAP(MB_data, reduction = "pca", dims = 1:30, verbose = FALSE)
MB_data <- FindClusters(MB_data, resolution = 0.5, verbose = FALSE)

In [ ]:
matching_indices <- match(rownames(MB_data@meta.data), rownames(MB_SHH@meta.data))
# Create a new column in the larger Seurat object
MB_data$new_column <- NA

# Assign values from the smaller Seurat object to the new column in the larger Seurat object
MB_data$new_column[!is.na(matching_indices)] <- MB_SHH$Subpopulations[matching_indices[!is.na(matching_indices)]]
MB_data$new_column[is.na(matching_indices)] <- "None"

MB_data$all.cell.types <- ifelse(MB_data$cell_type == "malignant", MB_data$new_column, MB_data$cell_type)

In [ ]:
DimPlot(MB_data, group.by = "cell_type") | DimPlot(MB_data, group.by = "orig.ident") | DimPlot(MB_data, group.by = "all.cell.types")

In [ ]:
visium_A_human_all <- run_label_transfer(MB_data,visium_A_human, col = "all.cell.types")
visium_B_human_all <- run_label_transfer(MB_data,visium_B_human, col = "all.cell.types")
visium_C_human_all <- run_label_transfer(MB_data,visium_C_human, col = "all.cell.types")
visium_D_human_all <- run_label_transfer(MB_data,visium_D_human, col = "all.cell.types")

In [ ]:
head(visium_A_human_all@meta.data)

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 7)

SpatialDimPlot(visium_A_human_all, group.by = "predicted.id")|
SpatialDimPlot(visium_B_human_all, group.by = "predicted.id")|
SpatialDimPlot(visium_C_human_all, group.by = "predicted.id")|
SpatialDimPlot(visium_D_human_all, group.by = "predicted.id")

In [ ]:
cols <- as.list(RColorBrewer::brewer.pal(name = "Paired", n =length(unique(visium_A_human_all$all.cell.types))))
names(cols) <- sort(unique(visium_A_human_all$all.cell.types))

## title

## Integration of Transcriptomic and Metabolomic data

Because we have matched transcriptomic and metabolomic data for each spot, we can directly compare and associated metabolites and genes which are associated with the <span style = "color: blue;">**Control**</span> and <span style = "color: red;">**Treated**</span> samples. 

First we have to create an object that contains both datasets in different assays: metabolomic data = "SPM", transcriptomic data = "SPT"

In [ ]:
# subset the visium object so it only contains the spots that match the metabolomic data
visium_c_sub <- subset(visium_C, cells = intersect(colnames(T1_test_x), colnames(visium_C)))

In [ ]:
combined.T1

In [ ]:
# Create combined data object 
combined.T1 <- CreateSeuratObject(counts = T1_test_x@assays$Spatial$counts, assay = "SPM", meta.data = T1_test_x@meta.data)
combined.T1@assays$SPM$data <- T1_test_x@assays$Spatial$data
combined.T1@images <- visium_c_sub@images

# make assay for transcriptomic data 
st_assay <- CreateAssay5Object(counts = visium_c_sub@assays$Spatial$counts, data = NULL)

# add assay to combined object
combined.T1[["SPT"]] <- st_assay

Next we have to peform processing on the transcriptomic and metabolomic data. We will start with the ST data ...

In [ ]:
DefaultAssay(combined.T1) <- "SPT"
combined.T1 <- NormalizeData(combined.T1)
combined.T1 <- FindVariableFeatures(combined.T1)
combined.T1 <- ScaleData(combined.T1)
combined.T1 <- RunPCA(combined.T1, verbose = FALSE, reduction.name = "spt.pca")
combined.T1 <- FindNeighbors(combined.T1, dims = 1:30, reduction ="spt.pca")

In [ ]:

combined.T1 <- RunUMAP(combined.T1, dims = 1:30, reduction.name = "spt.umap", reduction = "spt.pca", verbose = FALSE)
combined.T1 <- FindClusters(combined.T1, resolution = 0.5, verbose = FALSE )

We need to change the name of the seurat clusters so that the results don't get overwritten when running this again for the SM data

In [ ]:
combined.T1@meta.data$SPT_clusters <- combined.T1@meta.data$seurat_clusters

Lets clustering the SPM data now! We will use a resolution of 1. But you can check other resolution values and see how the results change.

In [ ]:
DefaultAssay(combined.T1) <- "SPM"
combined.T1 <- FindVariableFeatures(combined.T1, verbose = FALSE)
combined.T1 <- ScaleData(combined.T1, verbose = FALSE)
combined.T1 <- RunPCA(combined.T1, verbose = FALSE, reduction.name = "spm.pca")
combined.T1 <- FindNeighbors(combined.T1, dims = 1:50, reduction = "spm.pca", verbose = FALSE)

In [ ]:
combined.T1 <- RunUMAP(combined.T1, dims = 1:50, reduction.name = "spm.umap", reduction = "spm.pca", verbose = FALSE)
combined.T1 <- FindClusters(combined.T1, resolution = 1, verbose = FALSE)

In [ ]:
combined.T1@meta.data$SPM_clusters <- combined.T1@meta.data$seurat_clusters

Great! Now lets look at the results ...

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 15)

plot1 <- DimPlot(combined.T1, group.by = "SPT_clusters", label = TRUE, reduction = "spt.umap") + SpatialDimPlot(combined.T1, group.by = "SPT_clusters")
plot2 <- DimPlot(combined.T1, label = TRUE, reduction = "spm.umap", group.by = "SPM_clusters")  + SpatialDimPlot(combined.T1, group.by = "SPM_clusters")
plot1/plot2

In [ ]:
saveRDS(combined.T1, paste0(OUT_DIR, "data_objects/T1_combined.RDS"))

In [ ]:
library(mclust)

labels_true <- combined.T1@meta.data$SPT_clusters

# Extract cluster assignments for method 2
labels_pred <- combined.T1@meta.data$SPM_clusters

# Calculate adjusted Rand index
adjusted_rand_index <- adjustedRandIndex(labels_true, labels_pred)

adjusted_rand_index

In [ ]:
library(gplots)

cluster_table <- table(labels_true, labels_pred)

# Calculate proportions of cluster type 2 within each cluster of cluster type 1
proportions <- prop.table(cluster_table, margin = 1)

# Create a heatmap to visualize the proportions
heatmap_clustered <- heatmap.2(as.matrix(proportions),
                               Rowv = TRUE, Colv = TRUE,
                               dendrogram = "both",
                               scale = "none", # you can change this to "row", "column", or "none"
                               col = colorRampPalette(c("white", "blue"))(100),
                               key = TRUE, keysize = 1.5, key.title = NA, # Add a scale bar
                               trace = "none")


plot(heatmap_clustered)

We can see from these results that the clustering is quite similar! we get clear cluserting of the tumour with some sub-clusters being displayed in both the metabolomic (bottom) and transcriptomic (top) data.

Next we can **Integrate** the two data modalities together using some functions avalible from *Seurat*

In [ ]:
mm.integration <- FindMultiModalNeighbors(
  merged_MALDIVIS, reduction.list = list("spt.pca", "spm.pca"), 
  dims.list = list(1:30, 1:50), return.intermediate = TRUE)

In [ ]:
x <- rep(0.5, length(names(mm.integration@misc$modality.weight@modality.weight.list$spm.pca)))
names(x) <- names(mm.integration@misc$modality.weight@modality.weight.list$spm.pca)
mm.integration@misc$modality.weight@modality.weight.list$spm.pca <- x

x <- rep(0.5, length(names(mm.integration@misc$modality.weight@modality.weight.list$spt.pca)))
names(x) <- names(mm.integration@misc$modality.weight@modality.weight.list$spt.pca)
mm.integration@misc$modality.weight@modality.weight.list$spt.pca <- x

In [ ]:
mm.integration <- FindMultiModalNeighbors(
  merged_MALDIVIS, reduction.list = list("spt.pca", "spm.pca"), 
  dims.list = list(1:30, 1:50), return.intermediate = TRUE, modality.weight = mm.integration@misc$modality.weight)

In [ ]:
mm.integration <- RunUMAP(mm.integration, nn.name = "weighted.nn", reduction.name = "wnn.umap", reduction.key = "wnnUMAP_")
mm.integration <- FindClusters(mm.integration, graph.name = "wsnn", algorithm = 3, resolution = 0.5, verbose = FALSE, cluster.name = "wnn.0.5")
mm.integration <- FindClusters(mm.integration, graph.name = "wsnn", algorithm = 3, resolution = 0.3, verbose = FALSE, cluster.name = "wnn.0.3")

Lets look at the results. We can see that cluster <span style = "color: #8494FF;">**8**</span> and <span style = "color: #F8766D;">**0**</span> express our drug the most. Also of interest is cluster 2,3 and 4:

From looking at the plots: \
&nbsp;  &nbsp;  &nbsp; &nbsp;  <span style = "color: #ABA300;">**2**</span> = Looks like the ***human*** cells poliferating at the mouse/human border\
&nbsp;  &nbsp;  &nbsp; &nbsp;  <span style = "color: #7CAE00;">**3**</span> = Looks like the ***mouse*** cells at the mouse/human border \
&nbsp;  &nbsp;  &nbsp; &nbsp;  <span style = "color: #0CB702;">**4**</span> = Looks like spots that contain both ***mouse*** and ***human*** cells

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

(DimPlot(mm.integration, group.by = "wnn.0.5",  reduction = 'wnn.umap', label = TRUE, repel = TRUE, label.size = 5) |
DimPlot(mm.integration, group.by = "wnn.0.3",reduction = 'wnn.umap', label = TRUE, repel = TRUE, label.size = 5) |
DimPlot(mm.integration, group.by = "SPM_Clusters", reduction = 'wnn.umap', label = TRUE, repel = TRUE, label.size = 5, cols = colour_palate_SPM))/
(DimPlot(mm.integration, group.by = "SPT_Clusters", reduction = 'wnn.umap', label = TRUE, repel = TRUE, label.size = 5, cols = colour_palate_SPT) |
DimPlot(mm.integration, group.by = "sample", reduction = 'wnn.umap', label = TRUE, repel = TRUE, label.size = 5) |
FeaturePlot(mm.integration, reduction = 'wnn.umap',features = c("mz-448.246729024761")))

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 8)
SpatialDimPlot(mm.integration,  group.by = 'wnn.0.5', label = F)
SpatialDimPlot(mm.integration,  group.by = 'wnn.0.3', label = F)

In [ ]:
saveRDS(mm.integration, paste0(OUT_DIR, "data_objects/integrated_data.RDS"))

In [ ]:
mm.integration <- readRDS(paste0(OUT_DIR, "data_objects/integrated_data.RDS"))

In [ ]:
parsex <- CardinalIO::parseImzML("/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/raw_data/MALDI/DHB/vlp94a_dhb.imzML", ibd=TRUE)

In [ ]:
pmz <- parsex[["ibd"]][["mz"]]
pintensity <- parsex[["ibd"]][["intensity"]]
pcoord <- parsex[["run"]][["spectrumList"]][["positions"]]

In [ ]:
mass.range = c(160,1500)
mzmin <- mass.range[1]
mzmax <- mass.range[2]
resolution <- 10

In [ ]:
seq.ppm <- function(from, to, ppm) {
	length.out <- (log(to) - log(from)) / log((1 + 1e-6 * ppm) / (1 - 1e-6 *ppm))
	length.out <- floor(1 + length.out)
	i <- seq_len(length.out)
	from * ((1 + 1e-6 * ppm) / (1 - 1e-6 * ppm))^(i-1)
}

In [ ]:
mzout <- seq.ppm(
    from=floor(mzmin),
    to=ceiling(mzmax),
    ppm=resolution / 2) # seq.ppm == half-bin-widths
tol <- c(relative = resolution * 1e-6)

In [ ]:
spectra <- matter::sparse_mat(index=pmz, data=pintensity,
			domain=mzout, nrow=length(mzout), ncol=length(pintensity),
			tolerance=tol, sampler="linear")

In [ ]:
coord <- data.frame(
			x=as.numeric(pcoord[["position x"]]),
			y=as.numeric(pcoord[["position y"]]))


In [ ]:
test_10 <- MSImagingExperiment(spectra,
		featureData=MassDataFrame(mz=mzout),
		pixelData=PositionDataFrame(coord=coord, run="test"),
		metadata=list(parse=parsex),
		centroided=NA)

In [ ]:
pmz_x <- matter::matter_list(all_data_annotated@assays$Spatial@meta.data$raw_mz)

In [ ]:
pintensity_x <- matter::matter_list(all_data_annotated@assays$Spatial$counts)

In [ ]:
head(all_data_annotated@assays$Spatial@meta.data$raw_mz)

In [ ]:
class(pintensity)

### Pathway analysis







In [ ]:
library(fields)
library(Matrix)
library(dplyr)
library(rlist)
library(stringr)
library(fgsea)
library(pbapply)

#' @description
#' @param seurat is a Seurat object that contains spatial metabolic infomration
#' @param path is the output path for the visualization.
#' @param ppm_error is the parts-per-million error tolerance of matching m/z value with potential metabolites
#' @param ion_mode is only needed when ppm_error is not specified, goes to function estimate_mz_resolution_error will be used to access the ppm_error
#' @param tof_resolution is the tof resolution of the instrument used for MALDI run, calculated by ion (ion mass,m/z)/(Full width at half height)
#' @param input_mz is used when mass_matrix doesn't have the column names to be the m/z value, it a list of m/z values with one-to-one correspondence with each column of the mass_matrix
#' @param num_retained_component is an integer value to indicated preferred number of PCs to retain
#' @param resampling_factor is a numerical value >0, indicate how you want to resample the size of roginal matrix
#' @param p_val_threshold is the p value threshold for pathways to be significant
#' @param byrow is a boolean to indicates whether each column of the matrix is built byrow or bycol.
#' @return pca is the pca information of original data
#' @return pathway_enrichment_pc is the pathway enrichment results for each PC

#' @example principal_component_pathway_analysis(mass_matrix = readRDS("~/mass_matrix.rds") [,1:150], 
#' width = 912,height = 853,ppm_error = NULL,ion_mode = "positive",tof_resolution = 30000,input_mz = NULL,num_retained_component = NULL,variance_explained_threshold = 0.9,resampling_factor = 2,p_val_threshold = 0.05)

# Which help to build a pathway db based on detected metabolites
get_analytes_db = function(input_id,analytehaspathway,
                           chem_props,pathway) {
  
  rampid = unique(chem_props$ramp_id[which(chem_props$chem_source_id %in% unique(input_id))])
  #
  pathway_ids = unique(analytehaspathway$pathwayRampId[which(analytehaspathway$rampId %in% rampid)])
  
  analytes_db = lapply(pathway_ids, function(x) {
    content = analytehaspathway$rampId[which(analytehaspathway$pathwayRampId == x)]
    content = content[which(grepl(content, pattern = "RAMP_C"))]
    return(content)
  })
  analytes_db_name = unlist(lapply(pathway_ids, function(x) {
    name = pathway$pathwayName[which(pathway$pathwayRampId == x)]
    return(name)
  }))
  names(analytes_db) = analytes_db_name
  return(analytes_db)
}


#Helper function list_to_pprcomp

# Which help to build a pathway db based on detected metabolites
list_to_pprcomp <- function(lst) {
  # Create an empty object with class pprcomp
  obj <- structure(list(), class = "prcomp")
  # Assign components from the list to the object
  obj$sdev <- lst$sdev
  obj$rotation <- lst$rotation
  obj$center <- lst$center
  obj$scale <- lst$scale
  obj$x <- lst$x
  # Add other components as needed
  
  # Return the constructed pprcomp object
  return(obj)
}

label_background = function(mass_matrix,
                            level){
  labelled = apply(mass_matrix, MARGIN =1, FUN = function(x){
    if(length(which(x==0))>=level*length(x)){
      return("Background")
    }else{
      return("Target")
    }
  })
}
find_index <- function(lst, value) {
  indices <- which(sapply(lst, function(x) value %in% x))
  if (length(indices) == 0) {
    return(NULL)  # If value not found, return NULL
  } else {
    return(indices)
  }
}

principal_component_pathway_analysis = function(seurat,
                                                path = getwd(),
                                                ppm_error = NULL,
                                                ion_mode = NULL,
                                                tof_resolution = 30000,
                                                input_mz = NULL,
                                                num_retained_component = NULL,
                                                variance_explained_threshold = 0.9,
                                                resampling_factor = 2,
                                                p_val_threshold = 0.05,
                                                byrow = T) {
  # PCA analysis
  print("Scaling original matrix")
  #mass_matrix = t(seurat@assays[["SPM"]]@layers[["counts"]])[,1:100]
  mass_matrix = t(seurat@assays[["SPM"]]@layers[["counts"]]) 
  mass_matrix_with_coord = cbind(x= seurat@images[["slice1"]]@coordinates[["row"]],
                      y = seurat@images[["slice1"]]@coordinates[["col"]],
                      as.matrix(mass_matrix))
  width = seurat@images[["slice1"]]@coordinates[["col"]]
  height = seurat@images[["slice1"]]@coordinates[["row"]]
  
  if (!is.null(resampling_factor)) {
    print("Running matrix resampling")
    pb = txtProgressBar(
      min = 0,
      max = ncol(mass_matrix),
      initial = 0,
      style = 3
    )
    if (!is.numeric(resampling_factor)) {
      stop("Please enter correct resampling_factor")
    }
    new.width = as.integer(width / resampling_factor)
    new.height = as.integer(height / resampling_factor)
    
    resampled_mat = matrix(nrow =  new.height * new.width)
    for (i in 1:ncol(mass_matrix)) {
      temp_mz_matrix = matrix(mass_matrix[, i],
                              ncol = width,
                              nrow = height,
                              byrow = byrow)
      resampled_temp = ResizeMat(temp_mz_matrix, c(new.width,
                                                   new.height))
      resampled_mat = cbind(resampled_mat, as.vector(resampled_temp))
      setTxtProgressBar(pb, i)
    }
    close(pb)
    resampled_mat = resampled_mat[,-1]
    colnames(resampled_mat) = colnames(mass_matrix)
    print("Resampling finished!")
    gc()
  }else{
    resampled_mat = mass_matrix
  }
  print("Running the principal component analysis (can take some time)")
  # Runing PCA
  
  resampled_mat_standardised = as.matrix(t(
    t(resampled_mat) - Matrix::colSums(resampled_mat) / nrow(resampled_mat)
  ))
  print("Computing the covariance")
  cov_mat <- cov(resampled_mat_standardised)
  print("Computing the eigenvalue/eigenvectors")
  eigen_result <- eigen(cov_mat)
  gc()
  # Extract eigenvectors and eigenvalues
  eigenvectors <- eigen_result$vectors
  eigenvalues <- eigen_result$values
  
  print("Computing PCA")
  pc = pbapply::pblapply(1:ncol(resampled_mat_standardised), function(i) {
    temp = resampled_mat_standardised[, 1] * eigenvectors[1, i]
    for (j in 2:ncol(resampled_mat_standardised)) {
      temp = temp + resampled_mat_standardised[, j] * eigenvectors[j, i]
    }
    return(temp)
  })
  pc = do.call(cbind, pc)
  colnames(pc) = paste0("PC", 1:ncol(eigenvectors))
  # make pca object
  colnames(eigenvectors) = paste0("PC", 1:ncol(eigenvectors))
  rownames(eigenvectors) = colnames(resampled_mat)
  pca = list(
    sdev = sqrt(eigenvalues),
    rotation = eigenvectors,
    center = Matrix::colSums(resampled_mat) / nrow(resampled_mat),
    scale = FALSE,
    x = pc
  )
  pca = list_to_pprcomp(pca)
  print("PCA finished")
  rm(mass_matrix)
  gc()
  eigenvalues = pca$sdev ^ 2
  # Step 5: Compute Principal Components
  # Choose number of principal components, k
  # if not input, use scree test to help find retained components
  
  if (is.null(num_retained_component)) {
    if (!is.null(variance_explained_threshold)) {
      tryCatch({
        cumulative_variance = cumsum(eigenvalues) / sum(eigenvalues)
        par(mfrow = c(1, 1))
        par(mar = c(2, 2, 1, 1))
        # Plot cumulative proportion of variance explained
        plot(
          cumulative_variance,
          type = 'b',
          main = "Cumulative Variance Explained",
          xlab = "Number of Principal Components",
          ylab = "Cumulative Proportion of Variance Explained"
        )
        
        # Add a horizontal line at the desired threshold
        threshold = variance_explained_threshold  # Example threshold
        abline(h = threshold,
               col = "red",
               lty = 2)
        
        # Find the number of principal components to retain based on the threshold
        retained =  which(cumulative_variance >= threshold)[1] - 1
      },
      error = function(cond) {
        stop(
          "Check if correct variance threshold for principle components are inputted, should be numeric value between 0 and 1"
        )
      },
      warning = function(cond) {
        stop(
          "Check if correct variance threshold for principle components are inputted, should be numeric value between 0 and 1"
        )
      })
      
    } else{
      # if threshold not inputted, use Kaiser's criterion
      print(
        "Both variance_explained_threshold and num_retained_component not inputted, use Kaiser's criterion for determination"
      )
      plot(
        eigenvalues,
        type = 'b',
        main = "Scree Plot",
        xlab = "Principal Component",
        ylab = "Eigenvalue"
      )
      
      # Add a horizontal line at 1 (Kaiser's criterion)
      abline(h = 1,
             col = "red",
             lty = 2)
      
      # Add a vertical line at the elbow point
      elbow_point <- which(diff(eigenvalues) < 0)[1]
      abline(v = elbow_point,
             col = "blue",
             lty = 2)
      retained = length(which(eigenvalues >= 1))
    }
  } else{
    retained = as.integer(num_retained_component)
    if (is.na(retained)) {
      stop("Please enter correct number of principle components to retain")
    }
  }
  
  if (!is.null(input_mz)) {
    if ((length(input_mz) != ncol(resampled_mat)) |
        (!is.numeric(input_mz))) {
      stop(
        "Please ensure input_mz has one-to-one correspondence with each row of the mass_matrix"
      )
    } else{
      input_mz = input_mz
    }
  } else{
    tryCatch({
      input_mz = data.frame(cbind(
        row_id = 1:ncol(resampled_mat),
        mz = as.numeric(str_extract(row.names(seurat@assays[["SPM"]]@features), pattern = "\\d+\\.?\\d*"))
      ))
    },
    error = function(cond) {
      stop(
        "Check whether column names of the input matrix is correctly labelled as the m/z ratio"
      )
    },
    warning = function(cond) {
      stop(
        "Check whether column names of the input matrix is correctly labelled as the m/z ratio"
      )
    })
  }
  
 
  # source(paste0(dirname(system.file(package = "SpaMTP")), "fct_db_adduct_filter.R"))
  # source(paste0(dirname(system.file(package = "SpaMTP")), "/R/fct_formula_filter.R"))
  # source(paste0(dirname(system.file(package = "SpaMTP")), "/R/fct_proc_db.R"))
  #### Load the Cleaned and summarized DB ####
  Chebi_db     = readRDS( "/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/Chebi_1_names.rds")
  HMDB_db      = readRDS("/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/HMDB_1_names.rds")
  
  # Set the db that you want to search against
  db = rbind(HMDB_db, Chebi_db)
  # set which adducts you want to search for
  adduct_file = readRDS("/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/adduct_file.rds")
  if (is.null(ion_mode)) {
    stop("Please enter correct polarity:'positive' or 'negative'")
  } else{
    if (ion_mode == "positive") {
      test_add = sub(" ", "", adduct_file$adduct_name[which(adduct_file$charge >= 0)])
    } else if (ion_mode == "negative") {
      test_add = sub(" ", "", adduct_file$adduct_name[which(adduct_file$charge <= 0)])
    }
  }
  # Using Chris' pipeline for annotation
  # 1) Filter DB by adduct.
  db_1 = db_adduct_filter(db, test_add, polarity = ifelse(ion_mode == "positive",
                                                          "pos", "neg"))
  
  # 2) only select natural elements
  db_2 = formula_filter(db_1)
  
  # 3) search db against mz df return results
  # Need to specify ppm error
  # If ppm_error not specified, use function to estimate
  # Set error tolerance
  ppm_error = 1e6 / tof_resolution / sqrt(2 * log(2))
  db_3 = proc_db(input_mz, db_2, ppm_error) %>% mutate(entry = str_split(Isomers,
                                                                         pattern = "; "))
  print("Query necessary data and establish pathway database")
  input_id = lapply(db_3$entry, function(x) {
    x = unlist(x)
    index_hmdb = which(grepl(x, pattern = "HMDB"))
    x[index_hmdb] = paste0("hmdb:", x[index_hmdb])
    index_chebi = which(grepl(x, pattern = "CHEBI"))
    x[index_chebi] = tolower(x[index_chebi])
    return(x)
  })
  chem_props = readRDS("/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/chem_props.rds")
  db_3 = db_3 %>% mutate(inputid = input_id)
  rampid = c()
  chem_source_id = unique(chem_props$chem_source_id)
  print("Query db for addtional matching")
  pb2 = txtProgressBar(
    min = 0,
    max = nrow(db_3),
    initial = 0,
    style = 3
  )
  for (i in 1:nrow(db_3)) {
    rampid[i] = (chem_props$ramp_id[which(chem_source_id %in% db_3$inputid[i][[1]])])[1]
    setTxtProgressBar(pb2, i)
  }
  close(pb2)
  db_3 = cbind(db_3, rampid)
  print("Query finished")
  ####################################################################################################
  
  # get rank pathway database
  print("Getting reference pathways")
  analytehaspathway = readRDS("/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/analytehaspathway.rds")
  pathway = readRDS( "/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/pathway.rds")
  source = readRDS( "/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/source.rds")
  
  
  pathway_db = get_analytes_db(unlist(input_id), analytehaspathway,
                               chem_props, pathway)
  
  pathway_db = pathway_db[which(!duplicated(names(pathway_db)))]
  # get names for the ranks
  name_rank = lapply(input_mz$mz, function(x) {
    return(unique(na.omit(db_3$rampid[which(db_3$observed_mz == x)])))
  })
  
  #Set progress bar
  
  print("Runing set enrichment analysis")
  pca_sea_list = list()
  pb_new = txtProgressBar(
    min = 0,
    max = retained,
    initial = 0,
    style = 3
  )
  for (i in 1:retained) {
    # get the absolute value and sign of the loading
    loading = data.frame(cbind(
      abs_loading = abs(pca[["rotation"]][, i]),
      sign_loading = sign(pca[["rotation"]][, i])
    )) %>% arrange(desc(abs_loading))
    # run set enrichment analysis
    
    ranks = unlist(lapply(1:length(pca[["rotation"]][, i]), function(x) {
      pc_new = rep(pca[["rotation"]][x, i], times = length(name_rank[[x]])) +
        sample(-5:5, length(name_rank[[x]]), replace = T) * 1e-7
      names(pc_new) = name_rank[[x]]
      return(pc_new)
    }))
    ranks = ranks[which(!duplicated(names(ranks)))]
    suppressWarnings({
      gsea_result = fgsea::fgsea(
        pathways =  pathway_db,
        stats = ranks,
        minSize = 5,
        maxSize = 500
      ) %>% filter(pval <= p_val_threshold) %>% mutate(principle_component = paste0("PC", i)) %>% mutate(leadingEdge_metabolites = lapply(leadingEdge, function(x) {
        temp = unique(unlist(x))
        metabolites_name = c()
        for (z in 1:length(temp)) {
          index <- find_index(name_rank , temp[z])
          direction = sign(sum(pca[["rotation"]][index, i]))
          metabolites_name = c(metabolites_name,
                               paste0(
                                 tolower(chem_props$common_name[which(chem_props$ramp_id == temp[z])])[1],
                                 ifelse(direction >= 0, "↑", "↓")
                               ))
        }
        
        return(metabolites_name)
      }))
    })
    setTxtProgressBar(pb_new, i)
    # Make sure sign of loading is positive to make it positively correlate with the PC
    pca_sea_list = list.append(pca_sea_list,
                               gsea_result)
  }
  close(pb_new)
  names(pca_sea_list) = paste0("PC", 1:retained)
  
  ######################### Creating html
  # image_matrix = matrix(
  #   rowSums(resampled_mat),
  #   ncol = new.width,
  #   nrow = new.height,
  #   byrow = T
  # )
  # # plot(mass_matrix_with_coord[,1],
  # #      mass_matrix_with_coord[,2])
  # image(image_matrix)
  # Assign different colours to different layers
  image_matrix =  rowSums(resampled_mat)
    
  quantiles <-
    quantile(as.numeric(as.vector(image_matrix)), probs = seq(0, 1, 0.2))
  quantiles <-
    unique(quantiles + seq(0, 1e-10, length.out = length(quantiles)))
  col_index <-
    cut(as.vector(image_matrix),
        breaks = quantiles,
        labels = FALSE)
  
  library(jsonlite)
  colors <- c("red", "blue", "green", "yellow", "orange")
  
  # (1) Tissue image
  quantiles = quantile(as.numeric(as.vector(image_matrix)),
                       probs = seq(0, 1, 0.2))
  quantiles <-
    unique(quantiles + seq(0, 1e-10, length.out = length(quantiles)))
  
  col_index <-
    cut(t(image_matrix), breaks = quantiles, labels = FALSE)
  
  matrix_data_melted = data.frame(cbind(mass_matrix_with_coord[,1:2],
                                        image_matrix)) %>%
    mutate(color = col_index)
  
  matrix_data_melted = matrix_data_melted %>% rowwise() %>% mutate(col_nam  = colors[color])
  
  index = which(matrix_data_melted$image_matrix!= 0)
  
  matrix_data_melted = matrix_data_melted[index, ]
  tissue_image <- list(
    y = matrix_data_melted$x,
    x = matrix_data_melted$y,
    value = matrix_data_melted$image_matrix,
    colour = matrix_data_melted$col_nam,
    width = new.width,
    height = new.height
  )
  
  
  #a = reshape2::melt(c(1: nrow(pca$x)))
  # Convert to JSON
  
  # Optionally, write to a file
  
  
  
  #PCA_result
  # Convert list to JSON
  # Extract relevant components
  num = 5
  prcomp_data <- list(
    rotation = t(pca$rotation[, 1:num]),
    center = pca$center,
    scale = pca$scale,
    sdev = pca$sdev,
    x = t(pca$x[index, 1:num]),
    mz = rownames(pca[["rotation"]]),
    coordinate = t(cbind(
      matrix_data_melted$x ,
      matrix_data_melted$y
    )[index]),
    name =  sprintf(
      "(%d,%d)",
      matrix_data_melted$x,
      matrix_data_melted$y
    )[index]
  )
  
  ################### GSEA results
  var_gsea_table = c()
  for (i in 1:num) {
    temp = pca_sea_list[[i]]
    if (nrow(temp) == 0) {
      empty_row = t(c("No Signifcant Pathway Found",
                      rep(NA, times = ncol(temp) - 1)))
      temp = rbind(temp, empty_row, use.names = FALSE)
    }
    metabolites = c()
    if (nrow(temp) == 0) {
      var_gsea_table = c(var_gsea_table,
                         paste0('var PC', i, '=[', paste(rep('[],', times = 4),
                                                         collapse = ""), "]"))
      next
    }
    for (j in 1:length(temp$leadingEdge_metabolites)) {
      metabolites = c(metabolites,
                      paste0(unique(temp$leadingEdge_metabolites[[j]]), collapse = ", "))
    }
    temp_table = cbind(temp[, 1:7], metabolites = metabolites)
    var_gsea_table = c(
      var_gsea_table,
      paste0(
        'var PC',
        i,
        '=[',
        toJSON(temp_table$pathway),
        ',',
        toJSON(temp_table$pval),
        ',',
        toJSON(temp_table$NES),
        ',',
        toJSON(temp_table$metabolites),
        ']',
        collapse = ""
      )
    )
  }
  
  
  json_data <- jsonlite::toJSON(prcomp_data)
  json_matrix_data <- jsonlite::toJSON(tissue_image)
  
  html_temp = paste0(
    '<!DOCTYPE html>
  <html lang="en">
  <head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
  <script src="gmm_class.js"></script>
  <script src="kmeans.js"></script>
  <title>Prcomp Visualization</title>
  <style>
    .container {
      display: flex;
      justify-content: start; /* Optional: Adjust as needed */
    }
    .box {
      border: 1px solid black; /* Optional: Add borders for visibility */
      position: relative;
    }

    .input-container {
            display: inline-block;
        }

        /* Optional styling for label */
        label {
            margin-right: 5px; /* Adjust spacing between label and input */
        }
        .red-text {
    color: red;
    position: absolute;
    top: 0;
    left: 0;
}
  </style>
  </head>
  <body>
  <h1>Prcomp Data</h1>
  <div class="input-container"></div>
  <label>x-axis selection
    <select id="xSelect">
        <option value="" disabled selected>Select your PC</option>
        <option value="PC1" selected="selected">PC1</option>
        <option value="PC2">PC2</option>
        <option value="PC3">PC3</option>
        <option value="PC4">PC4</option>
        <option value="PC5">PC5</option>
    </select>
    </label>
    <label>y-axis selection
      <select id="ySelect">
          <option value="" disabled selected>Select your PC</option>
          <option value="PC1">PC1</option>
          <option value="PC2"selected="selected">PC2</option>
          <option value="PC3">PC3</option>
          <option value="PC4">PC4</option>
          <option value="PC5">PC5</option>
      </select>
      </label>
      <label for="integerInput">Enter an Integer for Cluster Numbers:</label>
      <input type="number" id="numcluster" name="integerInput" step="1" min = "2"> 
      <label>Clustering method
        <select id="clustering">
            <option value="" disabled selected>Select clustering method</option>
            <option value="GMM">GMM</option>
            <option value="Kmean">Kmean</option>
        </select>
        <button type="button" id= "button1">Regenerate Clusters</button>
        </label>
      </div>
    <div id = "Warning" class = "red-text"> </div>
    <div class="container">
      <div>
        <div id="pcaPlot" class="box" style="width: 600px; height: 400px;">
        </div>
        <div class="box" id="plot" style="position: relative;width: 600px; height: 400px;z-index: 2;">
          <canvas id="rasterCanvas" style="position: absolute; top: 0; left: 0;width: 100%; height: 99%;z-index: 0;"></canvas>
          <canvas id="pointCanvas" style="position: absolute; top: 0; left: 0;width: 100%; height: 99%;z-index: 1;"></canvas>
        </div>
      </div>
      <div>
        <div id="table1" class="box" style="width: 600px; height: 400px;"></div>
        <div class="box" id="table2" style="width: 600px; height: 400px;"></div>
      </div>
    </div>

  <script>


  // Embed the JSON data directly \n',
paste0(var_gsea_table, collapse = "\n")
,
'
const prcompData = [',
json_data ,
'];
const matrixData = [',
json_matrix_data,
'];
// Example of using the data
// Example of using the data
//PCA plot

 var data_table1 = [{
  type: "table",
  header: {
    values: [["<b>Pathway</b>"], ["<b>p_value</b>"],
				 ["<b>NES</b>"], ["<b>metabolites</b>"]],
    align: "center",
    line: {width: 1, color: "black"},
    fill: {color: "grey"},
    font: {family: "Arial", size: 12, color: "white"}
  },
  cells: {
    values: PC1,
    align: "center",
    line: {color: "black", width: 1},
    font: {family: "Arial", size: 11, color: ["black"]}
  }
}]

var layout_table1 = {
  title: "PC1 Pathway Information"
}

Plotly.newPlot("table1", data_table1, layout_table1);

// Table 2
var data_table2  = [{
  type: "table",
  header: {
    values: [["<b>Pathway</b>"], ["<b>p_value</b>"],
				 ["<b>NES</b>"], ["<b>metabolites</b>"]],
    align: "center",
    line: {width: 1, color: "black"},
    fill: {color: "grey"},
    font: {family: "Arial", size: 12, color: "white"}
  },
  cells: {
    values: PC2,
    align: "center",
    line: {color: "black", width: 1},
    font: {family: "Arial", size: 11, color: ["black"]}
  }
}]
var layout_table2 = {
  title: "PC2 Pathway Information"
}
Plotly.newPlot("table2", data_table2 ,layout_table2);


// pca PLOT
// Function to initialize parameters of GMM


function createArray(length) {
    // Calculate the value that each slot should hold
    const slotValue = 1 / length;

    // Create an array and populate it with the calculated value
    const array = Array(length).fill(slotValue);

    // Adjust the last element to ensure the sum equals 1
    array[length - 1] += 1 - array.reduce((sum, value) => sum + value, 0);

    return array;
}
function generateColors(n) {
    const colors = [];
    const increment = 360 / n;

    for (let i = 0; i < n; i++) {
        const hue = Math.round(increment * i);
        const saturation = 70; // You can adjust this value if needed
        const lightness = 70; // You can adjust this value if needed
        const color = `hsl(${hue}, ${saturation}%, ${lightness}%)`;
        colors.push(color);
    }

    return colors;
}
function transpose(array) {
    if (array.length === 0 || !Array.isArray(array[0])) return [];

    // Create an array to hold the result with the same number of sub-arrays as there are elements in each sub-array
    let result = [];
    for (let i = 0; i < array[0].length; i++) {
        result[i] = [];
        for (let j = 0; j < array.length; j++) {
            result[i].push(array[j][i]);
        }
    }

    return result;
}


// Assuming your data structure is stored in a variable called data_structure
var xCoordinates = matrixData[0].x;
var yCoordinates = matrixData[0].y;
var colors = matrixData[0].colour;
var value = matrixData[0].value
const canvas = document.getElementById("rasterCanvas");
const ctx = canvas.getContext("2d");
const container_tissue = document.getElementById("plot");
canvas.width = container_tissue.offsetWidth;
canvas.height = container_tissue.offsetHeight;
        // Calculate scaling factors
const scaleX = canvas.width/matrixData[0].height/1.1; // Assuming original width is 600
const scaleY = canvas.height/matrixData[0].width/1.1;
// draw raster image
  // Set canvas size


for (var i = 0; i < xCoordinates.length; i++) {
  //
            // Convert coordinates to canvas coordinates
            var canvasX = xCoordinates[i] * scaleX;
            var canvasY = yCoordinates[i] * scaleY;
            // Draw point
            ctx.fillStyle = colors[i];
            ctx.fillRect(canvasX, canvasY, 1, 1);
        }

function byRowToArrayByColumn(arr, numRows, numCols) {
    const result = [];
    for (let col = 0; col < numCols; col++) {
        for (let row = 0; row < numRows; row++) {
            result.push(arr[row * numCols + col]);
        }
    }
    return result;
}
function transposeArray(array) {
    return array[0].map((_, colIndex) => array.map(row => row[colIndex]));
}
var pc = [PC1,PC2,PC3,PC4,PC5]
console.log(pc)
function updatepcaPlot() {
  var num_to_cluster = Number(document.getElementById("numcluster").value)
  console.log(num_to_cluster)
  var candidate = ["PC1","PC2","PC3","PC4","PC5"]
  var xValue = document.getElementById("xSelect").value;
  var yValue = document.getElementById("ySelect").value;
  var new_pc1_index = candidate.findIndex(num => num === xValue)
  var new_pc2_index = candidate.findIndex(num => num === yValue)
  var method = document.getElementById("clustering").value;
  var pca_input = prcompData[0].x[new_pc1_index].map((element, index) => [element, prcompData[0].x[new_pc2_index][index]])
  //var pca_input = prcompData[0].x[0].map((element, index) => [element, prcompData[0].x[1][index]])


  //console.log(kmean_result)
  //console.log(kmean_result.centroids)

  let xMax = pca_input.reduce((max, subarray) => Math.max(max, subarray[0]), -Infinity);
  let yMax = pca_input.reduce((max, subarray) => Math.max(max, subarray[1]), -Infinity);
  let xMin = pca_input.reduce((min, subarray) => Math.min(min, subarray[0]), Infinity);
  let yMin = pca_input.reduce((min, subarray) => Math.min(min, subarray[1]), Infinity);
  let dx = xMax-xMin;
	let dy = yMax-yMin;
	let covariances = Array(num_to_cluster).fill(0)
		.map(_ => [[dx*dx*.01, 0], [0, dy*dy*.01]]);
  let colors = generateColors(num_to_cluster)
  if(method === "GMM"){
    let gmm = new GMM({
	  weights: Array(num_to_cluster).fill(1/num_to_cluster),
	  means: Array(num_to_cluster).fill(0).map(_ => [xMin + Math.random()*dx, yMin + Math.random()*dy]),
	  covariances: covariances
});
  //console.log(pca_input)
  pca_input.forEach(p => gmm.addPoint(p));
  gmm.runEM(10);
  let assignments = []

  for (let i = 0; i < pca_input.length; i++) {
    let probNorm = gmm.predictNormalize(pca_input[i]);
    let index = probNorm.indexOf(Math.max(...probNorm));
    assignments.push(colors[index])
    // Do something with probNorm if needed
 }
  //console.log(assignments)
  //prcompData[0].x[0].map((element, index) => [element, prcompData[0].x[1][index]])
  var update_pca = {
    "marker.color": [assignments]
  };
  //const canvas = document.querySelector("cycles_canvas");
  //const draw = new Draw(canvas, xMin, xMax, yMin, yMax);
  Plotly.restyle("pcaPlot", update_pca, 0);
  //for(let i=0; i<gmm.clusters; i++) {
	//		draw.ellipse(gmm.means[i], gmm.covariances[i], clusterColors[i]);
	//	}

  // update colours
  //console.log(assignments)
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  //let new_colour = assignments
  for (var i = 0; i < yCoordinates.length; i++) {
  //
            // Convert coordinates to canvas coordinates
            var canvasX = xCoordinates[i] * scaleX;
            var canvasY = yCoordinates[i] * scaleY;
            // Draw point
            ctx.fillStyle = assignments[i];
            ctx.fillRect(canvasX, canvasY, 1, 1);
        }

  }else if(method === "Kmean"){
    let assignments = []
    let kmean_result = kmeans(pca_input, num_to_cluster)
    console.log(kmean_result)
    for (let i = 0; i < pca_input.length; i++) {
    assignments.push(colors[kmean_result[1][i]])
    // Do something with probNorm if needed
 }
    // Do something with probNorm if needed
    //console.log(assignments)

 var update_pca = {
    "marker.color": [assignments]
  };
  //const canvas = document.querySelector("cycles_canvas");
  //const draw = new Draw(canvas, xMin, xMax, yMin, yMax);
  Plotly.restyle("pcaPlot", update_pca, 0);

  ctx.clearRect(0, 0, canvas.width, canvas.height);

  for (var i = 0; i < yCoordinates.length; i++) {
  //
            // Convert coordinates to canvas coordinates
            var canvasX = xCoordinates[i] * scaleX;
            var canvasY = yCoordinates[i] * scaleY;
            // Draw point
            ctx.fillStyle = assignments[i];
            ctx.fillRect(canvasX, canvasY, 1, 1);
        }
  }
}
document.getElementById("clustering").addEventListener("change", updatepcaPlot);
var button = document.getElementById("button1")

button.onclick = function(){
  updatepcaPlot();
}
  // Simulated PCA data (you would use your actual PCA data here)
  var pcaData = {
    names: prcompData[0].name,
    pc1: prcompData[0].x[0],
    pc2: prcompData[0].x[1]
  };

  var trace1pc = {
    x: pcaData.pc1,
    y: pcaData.pc2,
    mode: "markers",
    type: "scatter",
    marker: { size: 0.3 },
    text: pcaData.names,
    hoverinfo: "text+x+y"
  };

  var data_pc = [trace1pc];

  var layout_pc = {
    title: "PCA Plot",
    xaxis: { title: "PC1" },
    yaxis: { title: "PC2" },
    width: 600,
    height: 400
  };

  Plotly.newPlot("pcaPlot", data_pc, layout_pc);

  function updatePlot() {
      var candidate = ["PC1","PC2","PC3","PC4","PC5"]
      var xValue = document.getElementById("xSelect").value;
      var yValue = document.getElementById("ySelect").value;

      var new_pc1_index = candidate.findIndex(num => num === xValue)
      var new_pc2_index = candidate.findIndex(num => num === yValue)
      var pcaData = {
        names: prcompData[0].name,
        pc1: prcompData[0].x[new_pc1_index],
        pc2: prcompData[0].x[new_pc2_index]
      };

      var trace1pc = {
        x: pcaData.pc1,
        y: pcaData.pc2,
        mode: "markers",
        type: "scatter",
        marker: { size: 0.3 },
        text: pcaData.names,
        hoverinfo: "text+x+y"
      };
      var layout_pc = {
      title: "PCA Plot",
      xaxis: { title: candidate[new_pc1_index] },
      yaxis: { title: candidate[new_pc2_index] },
      width: 600,
      height: 400
     };
      var data_pc = [trace1pc];

      Plotly.newPlot("pcaPlot", data_pc, layout_pc);
    // Update table with new content
    var data_table1_new = [{
  type: "table",
  header: {
    values: [["<b>Pathway</b>"], ["<b>p_value</b>"],
				 ["<b>NES</b>"], ["<b>metabolites</b>"]],
    align: "center",
    line: {width: 1, color: "black"},
    fill: {color: "grey"},
    font: {family: "Arial", size: 12, color: "white"}
  },
  cells: {
    values: pc[new_pc1_index],
    align: "center",
    line: {color: "black", width: 1},
    font: {family: "Arial", size: 11, color: ["black"]}
  }
}]

var layout_table1_new = {
  title: "PC"+(new_pc1_index+1)+" Pathway Information"
}

Plotly.react("table1", data_table1_new, layout_table1_new);
Plotly.relayout("table1", layout_table1_new);
// Table 2
var data_table2_new  = [{
  type: "table",
  header: {
    values: [["<b>Pathway</b>"], ["<b>p_value</b>"],
				 ["<b>NES</b>"], ["<b>metabolites</b>"]],
    align: "center",
    line: {width: 1, color: "black"},
    fill: {color: "grey"},
    font: {family: "Arial", size: 12, color: "white"}
  },
  cells: {
    values: pc[new_pc2_index],
    align: "center",
    line: {color: "black", width: 1},
    font: {family: "Arial", size: 11, color: ["black"]}
  }
}]
var layout_table2_new = {
  title: "PC"+(new_pc2_index+1)+" Pathway Information"
}
Plotly.react("table2", data_table2_new ,layout_table2_new);
Plotly.relayout("table2", layout_table2_new);

document.getElementById("pcaPlot").on("plotly_hover", function(data){
    // Get index of clicked point
    var pointIndex = data.points[0].pointIndex;
    // console.log(pointIndex)
    let x_new_scatter = xCoordinates[pointIndex]*scaleX
    let y_new_scatter = yCoordinates[pointIndex]*scaleY

        ctx2.clearRect(0, 0, canvas2.width, canvas2.height);

        // Draw new point
        ctx2.fillStyle = "red";  // Color of the new point
        ctx2.beginPath();
        ctx2.arc(x_new_scatter, y_new_scatter , 3, 0, Math.PI * 2, true);  // Small circle for the point
        console.log(x_new_scatter)
        ctx2.fill();
        // Update last point coordinates
        lastX = x_new_scatter;
        lastY = y_new_scatter;
});
}



    // Event listeners to update the plot when selections change
    document.getElementById("xSelect").addEventListener("change", updatePlot);
    document.getElementById("ySelect").addEventListener("change", updatePlot);
    document.getElementById("clustering").addEventListener("change", updatepcaPlot);

// Example of using the data
console.log(prcompData[0].rotation[0]);






  // Plot each pixel

  // let xCoords = Array.from({length: 100}, (_, i) => 2*i);
  // let yCoords = Array.from({length: 100}, (_, i) => i);
  // let colors2 = new Array(10000).fill().map(() => `rgb(${Math.random()*255}, ${Math.random()*255}, ${Math.random()*255})`);

  // // Set canvas size
  // canvas.width = 100;
  // canvas.height = 100;
  // for (let x = 0; x < xCoords.length; x++) {
  //     for (let y = 0; y < yCoords.length; y++) {
  //         // Index for the colors array
  //         let index = x * yCoords.length + y;
  //         ctx.fillStyle = colors2[index];
  //         ctx.fillRect(xCoords[x], yCoords[y], 1, 1);
  //     }
  // }
// Create a trace for the heatmap
var trace = {
  y: xCoordinates,
  x: yCoordinates,
  z: value,
  mode: "markers",
  type: "heatmap",
  color: colors,
  //marker: {
  //  color:colors,
  //  size: 0.2
  //},
  //colorscale: "Viridis"
};

let lastX = null;
let lastY = null;
const canvas2 = document.getElementById("pointCanvas");
const ctx2 = canvas2.getContext("2d");
canvas2.width = container_tissue.offsetWidth;
canvas2.height = container_tissue.offsetHeight;
// Create the plot

//Plotly.newPlot("plot", [trace,trace_heatmap_scatter], layout,{staticPlot: true});
document.getElementById("pcaPlot").on("plotly_hover", function(data){
    // Get index of clicked point
    var pointIndex = data.points[0].pointIndex;
    // console.log(pointIndex)
    let x_new_scatter = xCoordinates[pointIndex]*scaleX
    let y_new_scatter = yCoordinates[pointIndex]*scaleY

        ctx2.clearRect(0, 0, canvas2.width, canvas2.height);

        // Draw new point
        ctx2.fillStyle = "yellow";  // Color of the new point
        ctx2.beginPath();
        ctx2.arc(x_new_scatter, y_new_scatter , 3, 0, Math.PI * 2, true);  // Small circle for the point
        console.log(x_new_scatter)
        ctx2.fill();
        // Update last point coordinates
        lastX = x_new_scatter;
        lastY = y_new_scatter;
});

document.addEventListener("DOMContentLoaded", function() {
    var divElement = document.getElementById("Warning");
    var newText = document.createElement("span"); // Creates a new <span> element
    newText.className = "Warning"; // Assigns the class for styling
    newText.textContent = "Note that ↑ and ↓ of PC pathway information are the sign of the loading corresponds to the metabolites m/z, since the sign is abitarily assigned during PCA depends on the setting, the arrow is only comparable within each PC and do not have biological implications, they are just used to describe how the metabolite contribute the PC"; // Adds text

    divElement.style.position = "relative"; // Ensures that the div can hold absolute positioned elements
    divElement.appendChild(newText); // Adds the newly created <span> to the div
});


</script>
</body>
</html>')
  setwd(path)
  gmm = readLines( "/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/gmm_class.js")
  writeLines(gmm,"gmm_class.js")
  kmean = readLines( "/QRISdata/Q1851/Andrew_C/SpaMTP/tys_datafiles/kmeans.js")
  writeLines(kmean,"kmeans.js")
  
  writeLines(html_temp,"pca_analysis.html")
  return(list(pca = pca,
              pathway_enrichment_pc = pca_sea_list,
              new.width = as.integer(width/as.numeric(resampling_factor)),
              new.height = as.integer(height/as.numeric(resampling_factor))))
}



In [ ]:
seurat = readRDS("~/andrew_updates/brain_public_combined_FMP10.RDS")


seurat@assays[["SPM"]]@layers[["counts"]] = seurat@assays[["SPM"]]@layers[["counts"]][1:100,]
seurat@assays[["SPM"]]@features = seurat@assays[["SPM"]]@features[1:100]

seurat
path = getwd()
ppm_error = NULL
polarity = "positive"
tof_resolution = 30000
input_mz = NULL
num_retained_component = NULL
variance_explained_threshold = 0.9
resampling_factor = NULL
p_val_threshold = 0.05
byrow = T


In [ ]:
principal_component_pathway_analysis(merged_MALDIVIS, ppm_error = NULL,
polarity = "positive",
tof_resolution = 30000,
input_mz = NULL,
num_retained_component = NULL,
variance_explained_threshold = 0.9,
resampling_factor = NULL,
p_val_threshold = 0.05,
byrow = T)

## Integrating Metabolomic and Transcriptomic data through Auto-Encoder

In [ ]:
library(MOFA2)

In [ ]:
MOFAobject <- create_mofa(list("Transcriptomics" = combined.T1@assays$SPT$counts, "Metabolomics" = combined.T1@assays$SPM$counts))

In [ ]:
MOFAobject

In [ ]:
plot_data_overview(MOFAobject)

In [ ]:
data_opts <- get_default_data_options(MOFAobject)
head(data_opts)

model_opts <- get_default_model_options(MOFAobject)
head(model_opts)

train_opts <- get_default_training_options(MOFAobject)
head(train_opts)

In [ ]:
MOFAobject <- prepare_mofa(
  object = MOFAobject,
  data_options = data_opts,
  model_options = model_opts,
  training_options = train_opts
)

In [ ]:
outfile = file.path(getwd(),"model.hdf5")
MOFAobject.trained <- run_mofa(MOFAobject, outfile, use_basilisk = TRUE)

## Transcriptional Activity associated with 

We can also look at the relative expression of genes known to be associated with the pathway Palbociclib interacts with

In [ ]:
options(repr.plot.width = 30, repr.plot.height = 15)

SpatialFeaturePlot(visium_A, features = c("hg38-CDK6", "hg-38CDK4", "hg-38CCND1", "hg38-CCND2", "hg38-CCNE1", "hg38-RB", "hg38-E2F1", "hg38-CDK2", "hg38-EEF2K", "hg38-PFKM"), ncol = 7, slot = "counts")/
SpatialFeaturePlot(visium_B, features = c("hg38-CDK6", "hg-38CDK4", "hg-38CCND1", "hg38-CCND2", "hg38-CCNE1", "hg38-RB", "hg38-E2F1", "hg38-CDK2", "hg38-EEF2K", "hg38-PFKM"), ncol = 7, slot = "counts")/
SpatialFeaturePlot(visium_C, features = c("hg38-CDK6", "hg-38CDK4", "hg-38CCND1", "hg38-CCND2", "hg38-CCNE1", "hg38-RB", "hg38-E2F1", "hg38-CDK2", "hg38-EEF2K", "hg38-PFKM"), ncol = 7, slot = "counts")/
SpatialFeaturePlot(visium_D, features = c("hg38-CDK6", "hg-38CDK4", "hg-38CCND1", "hg38-CCND2", "hg38-CCNE1", "hg38-RB", "hg38-E2F1", "hg38-CDK2", "hg38-EEF2K", "hg38-PFKM"), ncol = 7, slot = "counts")

In [ ]:
ImageMZPlot(T1_seurat_annotated, mz = 449.247 , size = 2.5)

In [ ]:
SearchAnnotations(T1_seurat_annotated, "lactate")

## Correlation Analysis 

In [ ]:
# subset the visium object so it only contains the spots that match the metabolomic data
visium_c_sub <- subset(visium_C, cells = intersect(colnames(T1_test_x), colnames(visium_C)))# Create combined data object 
combined.T1 <- CreateSeuratObject(counts = T1_test_x@assays$Spatial$counts, assay = "SPM", meta.data = T1_test_x@meta.data)
combined.T1@assays$SPM$data <- T1_test_x@assays$Spatial$data
combined.T1@images <- visium_c_sub@images

# make assay for transcriptomic data 
st_assay <- CreateAssay5Object(counts = visium_c_sub@assays$Spatial$counts)

# add assay to combined object
combined.T1[["SPT"]] <- st_assay

In [ ]:
#C1_xy <- FetchData(C1_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
#C2_xy <- FetchData(C2_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
T1_xy <- FetchData(T1_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))
T2_xy <- FetchData(T2_MALDIVIS_annot, vars = c('mz-448.246729024761','SHH.A.enrich.scores','SHH.B.enrich.scores', 'SHH.C.enrich.scores'))C1_results <- rcorr(as.matrix(C1_xy), type=c("pearson"))
T1_results <- rcorr(as.matrix(T1_xy), type=c("pearson"))
C2_results <- rcorr(as.matrix(C2_xy), type=c("pearson"))
T2_results <- rcorr(as.matrix(T2_xy), type=c("pearson"))## Have to set all the NA values in the P-value matrix to 0 
C1_results$P[is.na(C1_results$P)] <- 0
T1_results$P[is.na(T1_results$P)] <- 0
T2_results$P[is.na(T2_results$P)] <- 0
C2_results$P[is.na(C2_results$P)] <- 0

In [ ]:
T1_xy <- FetchData(T1_MALDIVIS_annot, vars = c('mz-448.246729024761','mz-448.206388634493'))

In [ ]:
combined.T1 <- readRDS(paste0(OUT_DIR, "data_objects/T1_combined.RDS"))

In [ ]:
as.matrix(T1_xy)[1:10,]

In [ ]:
T1_test_cor <- rcorr(t(as.matrix(combined.T1@assays$SPM$counts)), type=c("pearson"))

In [ ]:
T1_test_cor

## Load Saved Data

#### Load SpaMTP Seurat Objects

In [ ]:
C1_seurat <- readRDS(paste0(OUT_DIR,"data_objects/C1_seurat.RDS"))
T1_seurat <- readRDS(paste0(OUT_DIR,"data_objects/T1_seurat.RDS"))
C2_seurat <- readRDS(paste0(OUT_DIR,"data_objects/C2_seurat.RDS"))
T2_seurat <- readRDS(paste0(OUT_DIR,"data_objects/T2_seurat.RDS"))

#### Load Annotated SpaMTP Objects

In [ ]:
all_data_annotated <- readRDS(paste0(OUT_DIR,"data_objects/all_data_annotated.RDS"))

#### Load Merged Visium/MALDI Data Objects

In [ ]:
C1_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/C1_MALDIVIS_annot.RDS"))
T1_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/T1_MALDIVIS_annot.RDS"))
T2_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/T2_MALDIVIS_annot.RDS"))
C2_MALDIVIS_annot <- readRDS(paste0(OUT_DIR,"data_objects/C2_MALDIVIS_annot.RDS"))

#### Load Binned and Clustered Merged Visium/MALDI Objects

In [ ]:
T1_test_x <- readRDS( paste0(OUT_DIR,"data_objects/T1_Merged_clusters.RDS"))
C1_test_x <- readRDS( paste0(OUT_DIR,"data_objects/C1_Merged_clusters.RDS"))
T2_test_x <- readRDS( paste0(OUT_DIR,"data_objects/T2_Merged_clusters.RDS"))
C2_test_x <- readRDS( paste0(OUT_DIR,"data_objects/C2_Merged_clusters.RDS"))

In [ ]:
saveSeuratDataX <- function(data, outdir, assay = "Spatial", slot = "counts", annotations = FALSE){

  message(paste0("Generating new directory to store output here: ", outdir))
  message(paste0("Writing ", slot," slot to matrix.mtx, barcode.tsv, genes.tsv"))
  DropletUtils::write10xCounts(data[[assay]][slot], path = outdir, overwrite = TRUE)

  message("Writing @metadata slot to metadata.csv")
  data.table::fwrite(data@meta.data, paste0(outdir,"barcode_metadata.csv"))
  if (annotations){
    data.table::fwrite(data[[assay]]@meta.data, paste0(outdir,"feature_metadata.csv"))
  }

}

In [ ]:
saveSeuratDataX(C1_MALDIVIS_annot,annotations = TRUE, outdir = "/QRISdata/Q1851/Andrew_C/Metabolomics/Exported_data/C1_SPM/", assay = "SPM" )

In [ ]:
saveSeuratDataX(T2_MALDIVIS_annot,annotations = TRUE, outdir = "/QRISdata/Q1851/Andrew_C/Metabolomics/Exported_data/T2_SPM/", assay = "SPM")
saveSeuratDataX(C2_MALDIVIS_annot, annotations = TRUE,outdir = "/QRISdata/Q1851/Andrew_C/Metabolomics/Exported_data/C2_SPM/", assay = "SPM")
saveSeuratDataX(T1_MALDIVIS_annot, annotations = TRUE,outdir = "/QRISdata/Q1851/Andrew_C/Metabolomics/Exported_data/T1_SPM/", assay = "SPM")

## Additional Analysis

In [ ]:
merged_MALDIVIS <- readRDS(file = paste0(OUT_DIR, "data_objects/merged_MALDIVIS_with_clusters_HMDB.RDS"))

In [ ]:
library(Seurat)
library(ggplot2)
library(dplyr)
library(stringr)
library(SingleCellExperiment)
library(scater)
library(edgeR)
library(pheatmap)
library(limma)
library(utils)
library(stats)
library(grDevices)

#### SpaMTP Differential Peaks Analysis Functions ########################################################################################################################################################################################

#' Helper function for suppressing function progress messages
#'
#' @param message_text Character string containing the message being shown
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the messsage will be suppressed (default = TRUE).
#'
verbose_message <- function(message_text, verbose) {
  if (verbose) {
    message(message_text)
  }
}

#' Runs pooling of a merged Seurat Dataset to generate pseudo-replicates for each sample
#'       - This function is used by run_edgeR_annotations()
#'
#' @param data.filt A Seurat Object containing count values for pooling.
#' @param idents A character string defining the idents column to pool the data against.
#' @param n An integer defining the amount of pseudo-replicates to generate for each sample (default = 3).
#' @param assay Character string defining the assay where the mz count data and annotations are stored (default = "Spatial").
#' @param slot Character string defining the assay storage slot to pull the relative mz intensity values from (default = "counts").
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @returns A SinglCellExpereiment object which contains pooled (n)-pseudo-replicate counts data based on the Seurat Object input
#' @export
#'
#' @examples
#' # run_pooling <- list(seuratObj, idents = "sample", n = 3, assay = "Spatial", slot = "counts")
run_pooling <- function(data.filt, idents, n, assay, slot, verbose = TRUE) {

  cell_metadata <- data.filt@meta.data
  samples <- unique(cell_metadata[[idents]])

  verbose_message(message_text = paste0("Pooling one sample into ", n ," replicates..."), verbose = verbose)

  nrg <- n
  for(i in c(1:length(samples))){
    set.seed(i)
    wo<-which(cell_metadata[[idents]]== samples[i])
    cell_metadata[wo,'orig.ident2']<-paste(samples[i],sample(c(1:n),length(wo)
                                                             ,replace=T,prob=rep(1/nrg,nrg)),sep='_')
  }
  gene_data <- row.names(data.filt)
  filtered.sce <- SingleCellExperiment::SingleCellExperiment(assays = list(counts = data.filt[[assay]][slot]),
                                       colData = cell_metadata)


  tempf=strsplit(filtered.sce@colData[["orig.ident2"]],'_')
  pid=NULL
  for(i in 1:length(tempf)){
    pidone=tempf[[i]]
    if(length(pidone)!=3){
      pidone=c(pidone[1],'yes',pidone[2])
    }
    pid=rbind(pid,pidone)
  }

  filtered.sce@colData$type=pid[,2]

  summed <- scater::aggregateAcrossCells(filtered.sce,
                                 id=SingleCellExperiment::colData(filtered.sce)[,'orig.ident2'])

  return(summed)
}




#' Runs EdgeR analysis for pooled data
#'       - This function is used by run_edgeR_annotations()
#'
#' @param pooled_data A SingleCellExperiment object which contains the pooled pseudo-replicate data.
#' @param seurat_data A Seurat object containing the merged Xenium data being analysed (this is subset).
#' @param ident A character string defining the ident column to perform differential expression analysis against.
#' @param output_dir A character string defining the ident column to perform differential expression analysis against.
#' @param run_name A character string defining the title of this DE analysis (will be used when saving DEGs to .csv file).
#' @param n An integer that defines the number of pseudo-replicates per sample (default = 3).
#' @param logFC_threshold A numeric value indicating the logFC threshold to use for defining significant genes (default = 1.2).
#' @param assay A character string defining the assay where the mz count data and annotations are stored (default = "Spatial").
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#' @param return.individual Boolean value defining whether to return a list of individual edgeR objects for each designated ident. If FALSE, one merged edgeR object will be returned (default = FALSE).
#'
#' @returns A modified edgeR object which contains the relative pseudo-bulking analysis outputs, including a DEGs data.frame with a list of differential expressed m/z peaks
#' @export
#'
#' @examples
#' # pooled_obj <- run_pooling(SeuratObj, "sample", n = 3)
#' # run_DE(pooled_obj, SeuratObj, "sample", "~/Documents/DE_output/", "run_1", n = 3, logFC_threshold = 1.2, annotation.column = "all_IsomerNames", assay = "Spatial")
run_DE <- function(pooled_data, seurat_data, ident, output_dir, run_name, n, logFC_threshold, assay, return.individual = FALSE, verbose = TRUE){

  verbose_message(message_text = paste("Running edgeR DE Analysis for ", run_name, " -> with samples [", paste(unique(unlist(seurat_data@meta.data[[ident]])), collapse = ", "), "]"), verbose = verbose)

  annotation_result <- list()
  for (condition in unique(seurat_data@meta.data[[ident]])){
    verbose_message(message_text = paste0("Starting condition: ",condition), verbose = verbose)

    groups <- SingleCellExperiment::colData(pooled_data)[[ident]]
    groups <- gsub(condition, "Comp_A", groups)
    groups <- ifelse(groups != "Comp_A", "Comp_B", groups)



    y <- edgeR::DGEList(SingleCellExperiment::counts(pooled_data), samples=SingleCellExperiment::colData(pooled_data)$orig.ident2, group = groups)


    y$samples$condition <- groups
    y$samples$ident <- sub("_(.*)", "", y$samples$samples)

    keep <- edgeR::filterByExpr(y, group = groups, min.count = 2, min.total.count = 10)
    y <- y[keep,]
    y <- edgeR::calcNormFactors(y)

    design <- stats::model.matrix(~groups)

    design[,2] <- 1-design[,2]

    y <- edgeR::estimateDisp(y, design, robust=TRUE)
    fit <- edgeR::glmQLFit(y, design, robust=TRUE)
    res <- edgeR::glmTreat(fit, coef=ncol(fit$design), lfc=log2(logFC_threshold))
    summary(limma::decideTests(res))
    res$table$regulate <- dplyr::recode(as.character(limma::decideTests(res)),"0"="Normal","1"="Up","-1"="Down")
    de_group_edgeR <- res$table[order(res$table$PValue),]
    table(limma::decideTests(res))
    res <- edgeR::topTags(res,n = nrow(y))
    res$table$regulate <- "Normal"
    res$table$regulate[res$table$logFC>0 & res$table$FDR<0.05] <- "Up"
    res$table$regulate[res$table$logFC<0 & res$table$FDR<0.05] <- "Down"
    de_group_edgeR <- res$table[order(res$table$FDR),]
    #table(de_group_edgeR$regulate)


    if (!(is.null(output_dir))){
      utils::write.csv(de_group_edgeR, paste0(output_dir,condition,"_",run_name, ".csv"))
    }

    de_group_edgeR$gene <- rownames(de_group_edgeR)

    y$DEGs <- de_group_edgeR
    annotation_result[[condition]] <- y

  }

  if (return.individual){
    annotation_result
    return(annotation_result)
  } else {



    edger <- edgeR::DGEList(
      counts = annotation_result[[1]]$counts,
      samples = annotation_result[[1]]$samples
    )
    edger$samples$group <- edger$samples$ident
    edger$samples$condition <- NULL

    degs <- lapply(names(annotation_result), function(x){
      annotation_result[[x]]$DEGs$cluster <- x
      rownames(annotation_result[[x]]$DEGs) <- NULL
      annotation_result[[x]]$DEGs
    })

    combined_degs <- do.call(rbind, degs)
    rownames(combined_degs) <- 1:length(combined_degs$cluster)

    edger$DEGs <- combined_degs
    return(edger)
  }

}


#' Finds all differentially expressed genes between comparison groups specified
#'       - This function uses run_pooling() and run_DE() to pool and run EdgeR analysis
#'
#' @param data A Seurat object
#' @param ident A character string defining the metadata column or groups to compare genes values between.
#' @param n An integer that defines the number of pseudo-replicates (pools) per sample (default = 3).
#' @param logFC_threshold A numeric value indicating the logFC threshold to use for defining significant genes (default = 1.2).
#' @param DE_output_dir A character string defining the directory path for all output files to be stored. This path must a new directory. Else, set to NULL as default.
#' @param run_name A character string defining the title of this DE analysis that will be used when saving DEDs to .csv file (default = 'FindAllDEGs').
#' @param assay A character string defining the assay where the count data and annotations are stored (default = "Spatial").
#' @param slot Character string defining the assay storage slot to pull the relative expression values from. Note: EdgeR requires raw counts, all values must be positive (default = "counts").
#' @param return.individual Boolean value defining whether to return a list of individual edgeR objects for each designated ident. If FALSE, one merged edgeR object will be returned (default = FALSE).
#' @param verbose Boolean indicating whether to show the message. If TRUE the message will be show, else the message will be suppressed (default = TRUE).
#'
#' @returns Returns an list() contains the EdgeR DE results. Pseudo-bulk counts are stored in $counts and DEGs are in $DEGs.
#' @export
#'
#' @examples
#' # FindAllDEGs(SeuratObj, "sample",DE_output_dir = "~/Documents/DE_output/")
FindAllDEGs <- function(data, ident, n = 3, logFC_threshold = 1.2, DE_output_dir = NULL, run_name = "FindAllDEGs", assay = "Spatial", slot = "counts", return.individual = FALSE, verbose = TRUE){

  if (!(is.null(DE_output_dir))){
    if (dir.exists(DE_output_dir)){
      warning("Please supply a directory path that doesn't already exist")
      stop("dir.exists(DE_output_dir) = TRUE")
    } else{
      dir.create(DE_output_dir)
    }
  }

  #Step 1: Run Pooling to split each unique ident into 'n' number of pseudo-replicate pools
  pooled_data <- run_pooling(data,ident, n = n, assay = assay, slot = slot, verbose = verbose)

  #Step 2: Run EdgeR to calculate differentially expressed m/z peaks
  DEG_results <- run_DE(pooled_data, data, ident = ident, output_dir = DE_output_dir, run_name = run_name, n=n, logFC_threshold=logFC_threshold, assay = assay, verbose = verbose, return.individual = return.individual)

  # Returns an EDGEr object which contains the pseudo-bulk counts in $counts and DEGs in $DEGs
  return(DEG_results)

}




#' Generates a Heatmap of DEGs generated from edgeR analysis run using FindAllDEGs().
#'       - this function uses pheatmap() to plot data
#'
#' @param edgeR_output A list containing outputs from edgeR analysis (from FindAllDEGs()). This includes pseudo-bulked counts and DEGs.
#' @param n A numeric integer that defines the number of UP and DOWN regulated peaks to plot (default = 25).
#' @param only.pos Boolean indicating if only positive markers should be returned (default = FALSE).
#' @param FDR.threshold Numeric value that defines the FDR threshold to use for defining most significant results (default = 0.05).
#' @param logfc.threshold Numeric value that defines the logFC threshold to use for filtering significant results (default = 0.5).
#' @param order.by Character string defining which parameter to order markers by, options are either 'FDR' or 'logFC' (default = "FDR").
#' @param scale A character string indicating if the values should be centered and scaled in either the row direction or the column direction, or none. Corresponding values are "row", "column" and "none"
#' @param color A vector of colors used in heatmap (default = grDevices::colorRampPalette(c("navy", "white", "red"))(50)).
#' @param cluster_cols Boolean value determining if columns should be clustered or hclust object (default = F).
#' @param cluster_rows Boolean value determining if rows should be clustered or hclust object (default = T).
#' @param fontsize_row A numeric value defining the fontsize of rownames (default = 15).
#' @param fontsize_col A numeric value defining the fontsize of colnames (default = 15).
#' @param cutree_cols A numeric value defining the number of clusters the columns are divided into, based on the hierarchical clustering(using cutree), if cols are not clustered, the argument is ignored (default = 9).
#' @param silent Boolean value indicating if the plot should not be draw (default = TRUE).
#' @param plot_annotations_column Character string indicating the column name that contains the metabolite annotations to plot. Annotations = TRUE must be used in FindAllDEGs() for edgeR output to include annotations. If plot_annotations_column = NULL, m/z vaues will be plotted (default = NULL).
#' @param save_to_path Character string defining the full filepath and name of the plot to be saved as.
#' @param plot.save.width Integer value representing the width of the saved pdf plot (default = 20).
#' @param plot.save.height Integer value representing the height of the saved pdf plot (default = 20).
#' @param nlabels.to.show Numeric value defining the number of annotations to show per m/z (default = NULL).
#'
#' @returns A heatmap plot of significantly differentially expressed peaks defined in the edgeR ouput object.
#' @export
#'
#' @import dplyr
#'
#' @examples
#' # DEGs <- FindAllDEGs(SeuratObj, "sample")
#'
#' # DEGsHeatmap(DEGs)
DEGsHeatmap <- function(edgeR_output,
                         n = 5,
                         only.pos = FALSE,
                         FDR.threshold = 0.05,
                         logfc.threshold = 0.5,
                         order.by = "FDR",
                         scale ="row",
                         color = grDevices::colorRampPalette(c("navy", "white", "red"))(50),
                         cluster_cols = F,
                         cluster_rows = T,
                         fontsize_row = 15,
                         fontsize_col = 15,
                         cutree_cols = 9,
                         silent = TRUE,
                         save_to_path = NULL,
                         plot.save.width = 20,
                         plot.save.height = 20){


  degs <- edgeR_output$DEGs
  degs <- subset(degs, FDR < FDR.threshold)

  if (order.by == "FDR"){

    grouped_pos<- degs %>%
      group_by(cluster) %>%
      filter( logFC > logfc.threshold) %>%
      arrange(desc(regulate)) %>%
      slice_head(n = n)


    if (only.pos) {
      grouped_neg <- NULL

    } else {
      grouped_neg <- degs %>%
        group_by(cluster) %>%
        filter(logFC < - logfc.threshold) %>%
        arrange(regulate) %>%
        slice_head(n = n)
    }
    df <- do.call(rbind, list(grouped_pos,grouped_neg))
    df <- df[order(df$cluster, dplyr::desc(df$regulate)), ]

  } else {
    if ( order.by != "logFC"){
      warning("order.by has invalid argument. Must be either 'FDR' or 'logFC'. Heatmap defaulting to order by logFC")
    }

    grouped_pos<- degs %>%
      group_by(cluster) %>%
      filter(logFC > logfc.threshold) %>%
      arrange(-logFC) %>%
      slice_head(n = n)


    if (only.pos) {
      grouped_neg <- NULL
    } else {
      grouped_neg <- degs %>%
        group_by(cluster) %>%
        filter(logFC < - logfc.threshold) %>%
        arrange(logFC) %>%
        slice_head(n = n)
    }
    df <- do.call(rbind, list(grouped_pos,grouped_neg))
    df <- df[order(df$cluster, -df$logFC), ]
  }



  col_annot <- data.frame(sample = edgeR_output$samples$ident)
  row.names(col_annot) <- colnames(as.data.frame(edgeR::cpm(edgeR_output,log=TRUE)))

  mtx <- as.matrix(as.data.frame(edgeR::cpm(edgeR_output,log=TRUE))[unique(df$gene),])

  p <- pheatmap::pheatmap(mtx,scale=scale,color=color,cluster_cols = cluster_cols, annotation_col=col_annot, cluster_rows = cluster_rows,
                          fontsize_row = fontsize_row, fontsize_col = fontsize_col, cutree_cols = cutree_cols, silent = silent)

   if (!(is.null(save_to_path))){
     save_pheatmap_as_pdf(pheatmap = p, filename = save_to_path, width = plot.save.width, height = plot.save.height)
   }

  return(p)
}


#' Saves the DEGs generated pheatmap as a PDF
#'
#' @param pheatmap A pheatmap plot object that is being saved.
#' @param filename Character string defining the full filepath and name of the plot to be saved as.
#' @param width Integer value representing the width of the saved pdf plot (default = 20).
#' @param height Integer value representing the height of the saved pdf plot (default = 20).
#'
#' @export
#'
#' @examples
#' # save_pheatmap_as_pdf(pheatmap, filename = "/Documents/plots/pheatmap1")
save_pheatmap_as_pdf <- function(pheatmap, filename, width=20, height=20){

  pdf(paste0(filename,".pdf"), width=width, height=height)
  grid::grid.newpage()
  grid::grid.draw(pheatmap$gtable)
  dev.off()
}

########################################################################################################################################################################################################################

In [ ]:
head(merged_MALDIVIS@meta.data)

In [ ]:
test <- readRDS('/QRISdata/Q5291/MS_Spatial_metabolomics/10. Data/processed_data/DHB/data_objects/all_data_annotated.RDS')

In [ ]:
ImageDimPlot(test, group.by = "ssc", size = 1.6, fov = c("fov", "fov.2", "fov.3", "fov.4"))

In [ ]:
tumour <- subset(test, subset = ssc %in% c(3))

In [ ]:
ImageDimPlot(tumour, group.by = "ssc", size = 1.6, fov = c("fov", "fov.2", "fov.3", "fov.4"))

In [ ]:
tumour

In [ ]:
SpatialMZPlot(merged_MALDIVIS, mz = 448.246, plusminus = 0.05, pt.size.factor = 2.5, ncol = 2, assay = "SPM") & scale_fill_gradientn(colors = viridis(100), limits = c(0, 200),  na.value = viridis(100)[100])

In [ ]:
Plot3DFeature(merged_MALDIVIS, features = c('mz-448.246729024761', "E2F1"), 
               assays = c("SPM", "SPT"), )

In [ ]:
FindNearestMZ(merged_MALDIVIS, 448.246)


In [ ]:
DefaultAssay(merged_MALDIVIS) <- "SPT"

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

SpatialFeaturePlot(subset(merged_MALDIVIS, subset = percent.human > 75), features = 'hg38-E2F1', ncol = 2, slot = "data")

In [ ]:
human <- subset(merged_MALDIVIS, subset = percent.human > 75)

In [ ]:
head(human)

In [ ]:
P1 <- subset(merged_MALDIVIS, subset = sample == "T1")


## Pathway

In [ ]:
metabolites <- read.csv("/scratch/user/uqacause/files/project_data_objects/MB/refined_annotations_DE.csv")
genes <- read.csv("/scratch/user/uqacause/files/project_data_objects/MB/gene_markers_DE.csv")

In [ ]:
head(genes)

In [ ]:
test <- FishersPathwayAnalysis( list("mz" = metabolites$gene, "genes" = genes$gene), analyte_type = c("mz", "genes"), polarity = "positive")

In [ ]:
?FishersPathwayAnalysis

In [ ]:
human <- subset(merged_MALDIVIS, subset = percent.human > 50)

In [ ]:
x <- RefineLipids(merged_MALDIVIS@assays$SPM@meta.data, annotation.column = "all_IsomerNames")

In [ ]:
SearchAnnotations(merged_MALDIVIS, "LacCer(40:1", assay = "SPM", column.name = "Species.Name.Simple")

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)

SpatialMZPlot(merged_MALDIVIS, mz = 946.7204, plusminus = 0.05, pt.size.factor = 2.5, ncol = 2, assay = "SPM") |
SpatialMZPlot(human, mz = 946.7204, pt.size.factor = 2, ncol = 2, assay = "SPM", crop = F)  & scale_fill_gradientn(colors = viridis(100), limits = c(0, 150),  na.value = viridis(100)[100])

In [ ]:
SpatialFeaturePlot(merged_MALDIVIS, features = "hg38-ALDH1A2")